# Track 1 · 5-fold CV, 5 architectures → one Codabench ZIP

Trains REVE, bidirectional Conformer, AMSPBE, ResCNN and TCN-LC on the same 5 folds (25 models), tunes the
architecture weights on the out-of-fold blocks, scores the local test once and writes a checked submission ZIP.

**Setup:** GPU T4 × 2, Internet on, the Hugging Face token as a Kaggle secret, the Alljoined dataset attached.
Edit the two lines in the next cell, then *Save & Run All*.

**Time:** ~9 h with an extracted cache. From the raw dataset the extraction adds ~4–5 h, so the first session will
not finish everything: attach its output to a new version and run again, finished runs are picked up.

**Outputs:** `UPLOAD_TO_CODABENCH/t1_cv*.zip` (submission), `FOR_TEAM.zip` (OOF predictions, tables, folds),
`cv_runs/` (all runs, used for resuming).

## Configuration

In [ ]:
import os, sys, json, time, shutil, subprocess
from pathlib import Path

# ======================= edit these two =======================
HF_SECRET = "HF_TOKEN"                                   # name of the Kaggle secret holding your Hugging Face token
DATA_PATH = "/kaggle/input/xu2025-alljoined-1-6m"        # the attached Alljoined dataset (raw folder or extracted cache)
# ==============================================================

SMOKE_TEST = False
T_START = time.time()

KAGGLE = Path(os.environ.get("KAGGLE_ROOT", "/kaggle"))
INPUT, WORK = KAGGLE / "input", KAGGLE / "working"
SCRATCH = (KAGGLE / "temp") if (KAGGLE / "temp").exists() else Path("/tmp/t1_scratch")
OUT, SRC = WORK / "reve_kaggle", WORK / "src"
for d in (OUT, SRC, SCRATCH):
    d.mkdir(parents=True, exist_ok=True)


def resolve_data(path):
    """Returns (cache_dir, raw_study). Accepts an extracted cache, or the raw dataset at any level of
    <root>/download/nm000134/sub-*/..."""
    p = Path(path)
    assert p.exists(), f"DATA_PATH not found: {p}"
    for d in [p, *[q for q in p.rglob("meta.json")][:1]]:
        d = d if d.is_dir() else d.parent
        if all((d / f).exists() for f in ("meta.json", "X_train.npy", "emb_table.npy")):
            return d, None
    if (p / "download" / "nm000134").is_dir():
        return None, p
    if p.name == "download" and (p / "nm000134").is_dir():
        return None, p.parent
    if p.name == "nm000134":
        return None, p.parent.parent if p.parent.name == "download" else _link_raw(p)
    hits = [q for q in p.rglob("nm000134") if q.is_dir()]
    if hits:
        return resolve_data(hits[0])
    if any(p.glob("sub-*")) and (p / "stimuli").exists():
        return None, _link_raw(p)
    raise RuntimeError(f"{p}: neither an extracted cache nor the raw study (…/download/nm000134)")


def _link_raw(nm):
    root = SCRATCH / "raw_study"
    (root / "download").mkdir(parents=True, exist_ok=True)
    if not (root / "download" / "nm000134").exists():
        (root / "download" / "nm000134").symlink_to(nm, target_is_directory=True)
    return root


CACHE_DIR, RAW_STUDY = resolve_data(DATA_PATH)
KEEP_EXTRACTION = RAW_STUDY is not None          # keep the extracted cache in the output, for the next session
print(f"data: {'cache ' + str(CACHE_DIR) if CACHE_DIR else 'raw study ' + str(RAW_STUDY) + ' (will be extracted)'}")

# split: team manifest (seed 33); its train + val part is cut into 5 folds, local test untouched
TEAM_MANIFEST = "auto"
EXPECTED_SCIENCE_SHA = "6509d84c6aa9ba2559f18b994f8a606e160cde3369843ad87a6beea776e434db"
N_FOLDS, FOLD_SEED = 5, 33
SCORE_LOCAL_TEST = True

AUG = dict(shift=3, amp=0.1, ch_drop=0.1, noise=0.1, avg_p=0.3, avg_k=3)
ARCHS = ["reve", "bidir_conformer", "amspbe", "rescnn", "tcnlc"]
RECIPES = dict(
    reve=dict(tag="research", epochs=6, samples_per_epoch=300_000, batch=256, lr=3e-4, backbone_lr_mult=0.3, llrd=0.9, freeze_layers=0,
              freeze_epochs=1, weight_decay=0.05, schedule="onecycle", ema=0.999, swa_keep=6, grad_clip=1.0, temperature_init=0.07,
              temperature_cap=100.0, init_seed=0, head_kw=dict(head="tokens+attn", attn_layers=None, n_queries=4), aug=dict(AUG)),
    bidir_conformer=dict(tag="r2recipe", bidir=True,
                         conformer=dict(k=40, t_kernel=13, pool=13, stride=6, depth=2, heads=4, drop=0.3, proj_drop=0.5),
                         epochs=20, patience=6, samples_per_epoch=300_000, batch=512, lr=3e-4, weight_decay=1e-2,
                         schedule="onecycle", ema=None, swa_keep=6, grad_clip=1.0, temperature_init=0.07, temperature_cap=100.0,
                         aug=dict(AUG)),
    amspbe=dict(tag="dtha4", n_members=4,
                amspbe=dict(K=16, t_kernel=7, t_pool=60, drop=0.3, proj_drop=0.5, bottleneck=384, use_tsam=True, use_dwconv=True),
                epochs=30, patience=8, samples_per_epoch=300_000, batch=512, lr=1e-3, weight_decay=1e-2, schedule="onecycle",
                ema=0.999, swa_keep=6, grad_clip=1.0, temperature_init=0.07, temperature_cap=100.0, amp=False, aug=dict(AUG)),
    rescnn=dict(tag="p13", rescnn=dict(width=96, stem_kernel=15, kernel=9, dilations=[1, 2, 4], dropout=0.15, pool=12, hidden=512,
                                       head_drop=0.2),
                epochs=25, patience=6, samples_per_epoch=300_000, batch=512, lr=1e-3, weight_decay=1e-2, schedule="onecycle",
                ema=0.999, swa_keep=6, grad_clip=1.0, temperature_init=0.07, temperature_cap=100.0, amp=False, aug=dict(AUG)),
    tcnlc=dict(tag="msdtcn_logcov_sym", tcnlc=dict(width=96, stem_kernel=7, kernel=5, dilations=[1, 2, 4, 8], n_blocks=3, dropout=0.15,
                                                   pool=8, cov_dim=16, shrink=0.05, hidden=512, head_drop=0.2),
               sym_w=0.5, epochs=14, patience=5, samples_per_epoch=300_000, batch=512, lr=1e-3, weight_decay=1e-2, schedule="onecycle",
               ema=0.999, swa_keep=6, grad_clip=1.0, temperature_init=0.07, temperature_cap=100.0, amp=False, aug=dict(AUG)),
)
EST_MINUTES = dict(reve=50, bidir_conformer=12, amspbe=16, rescnn=8, tcnlc=10)
MONITOR_MAX_EPOCHS = 20000

# ensemble
WEIGHT_STEP = 0.1               # simplex grid over the 5 architecture weights (1001 points), scored on the OOF blocks
MIN_GAIN_OVER_EQUAL = 0.003     # tuned weights only if they beat equal weights by this (top-5 + 0.5 * MRP)
ZIP_REVE_FOLDS = None           # e.g. 3 to keep only the 3 best REVE folds (smaller ZIP)
TIME_BUDGET_HOURS = 11.3
RESERVE_MIN = 60                # kept at the end for the ensemble and the ZIP checks

# packaging and checks
WEIGHTS_DTYPE = "float16"
CHECK_EPOCHS, REPLAY_EPOCHS = 4000, 2000
CHECK_HARNESS = True
EVAL_LIMIT_MIN = 120

# environment
BUILD_ENV = True
NEED_HF = True
N_PROCESSES = "auto"
PRECISION = "auto"
PINS = ["torch==2.8.0", "torchvision==0.23.0", "torchaudio==2.8.0", "braindecode==1.8.1", "neuralset==0.4.0",
        "neuralfetch==0.4.0", "neuraltrain==0.4.0", "neuralbench==0.4.0", "exca==0.5.29", "benchopt>=1.10", "mne",
        "huggingface_hub", "safetensors", "pandas", "pyarrow", "scipy", "scikit-learn", "tqdm", "matplotlib"]
KIT_URL = "https://github.com/neural-interfaces26/2026-competition.git"
KAGGLE_LIMIT_HOURS, FINISH_MINUTES = TIME_BUDGET_HOURS, 10

if SMOKE_TEST:
    RECIPES = {k: {**r, "epochs": 2, "samples_per_epoch": 4096, "batch": 64, "swa_keep": 2} for k, r in RECIPES.items()}
    EST_MINUTES = {k: 1 for k in EST_MINUTES}
    MONITOR_MAX_EPOCHS, WEIGHT_STEP, RESERVE_MIN = 2000, 0.25, 3
    CHECK_EPOCHS, REPLAY_EPOCHS = 640, 320
print(f"{len(ARCHS)} architectures x {N_FOLDS} folds | scratch {SCRATCH} ({shutil.disk_usage(SCRATCH).free / 2**30:.0f} GiB free)")

## Environment
Pinned Python 3.12 environment (uv), the competition kit, and a check that REVE can be downloaded with your token.

In [ ]:
def sh(cmd, env=None, cwd=None, check=True):
    """Runs a command and streams its output here (also in the commit log)."""
    print("$", " ".join(str(c) for c in cmd)[:300], flush=True)
    p = subprocess.Popen([str(c) for c in cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env, cwd=cwd, bufsize=1)
    tail = []
    for line in p.stdout:
        print(line, end="", flush=True)
        tail = (tail + [line])[-60:]
    if p.wait() != 0 and check:
        raise RuntimeError(f"command failed ({p.returncode}): {' '.join(str(c) for c in cmd)[:200]}\n{''.join(tail)[-3000:]}")
    return p.returncode


VENV = SCRATCH / "venv312"
if BUILD_ENV:
    PY = VENV / "bin" / "python"
    if not (VENV / "READY").exists():
        sh([sys.executable, "-m", "pip", "install", "-q", "uv"])
        sh([sys.executable, "-m", "uv", "venv", "--python", "3.12", str(VENV)])
        sh([sys.executable, "-m", "uv", "pip", "install", "--python", str(PY), *PINS])
        (VENV / "READY").write_text("ok")
else:
    PY = Path(sys.executable)
(OUT / "pip_freeze.txt").write_text(subprocess.run([str(PY), "-m", "pip", "freeze"], capture_output=True, text=True).stdout
                                    or subprocess.run([sys.executable, "-m", "uv", "pip", "freeze", "--python", str(PY)], capture_output=True, text=True).stdout)

KIT = SCRATCH / "2026-competition"
if (KIT / ".git").exists():
    sh(["git", "-C", KIT, "pull", "-q"], check=False)
else:
    sh(["git", "clone", "-q", "--depth", "1", KIT_URL, KIT])
KIT_DATE = subprocess.run(["git", "-C", str(KIT), "log", "-1", "--format=%ci"], capture_output=True, text=True).stdout.strip()

ENV = {**os.environ, "HF_HOME": str(SCRATCH / "hf"), "MNE_DATA": str(SCRATCH / "mne_data"), "PYTHONUNBUFFERED": "1",
       "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True", "NEURALBENCH_EXTRACT_CPUS": "2", "TOKENIZERS_PARALLELISM": "false",
       "NCCL_P2P_DISABLE": "1", "TORCH_NCCL_ASYNC_ERROR_HANDLING": "1",
       "MPLBACKEND": "Agg"}   # Kaggle's inline plot backend does not exist in the separate environment   # no GPU-to-GPU P2P: safer on cloud T4 pairs
ENV.setdefault("REVE_POSITIONS_PATH", str(SCRATCH / "reve_positions"))
try:
    from kaggle_secrets import UserSecretsClient
    ENV["HF_TOKEN"] = UserSecretsClient().get_secret(globals().get("HF_SECRET", "HF_TOKEN"))
    print("Hugging Face token: from Kaggle Secrets ok")
except Exception as e:
    print(f"WARNING: no HF_TOKEN secret ({type(e).__name__}): fine if REVE-base downloads without one, otherwise add it (Add-ons → Secrets)")

HF_CHECK = r"""
import os, shutil, sys
from pathlib import Path
from huggingface_hub import HfApi, hf_hub_download, snapshot_download
tok = os.environ.get("HF_TOKEN") or None
api = HfApi(token=tok)
try:
    print("token: valid, user", api.whoami()["name"])
except Exception as e:
    print("token: none or invalid (" + type(e).__name__ + ")")
ok = True
try:                       # positions: braindecode downloads them WITHOUT the token, so prefetch them here with it
    f = hf_hub_download("brain-bzh/reve-positions", "positions.json", token=tok)
    dst = Path(os.environ["REVE_POSITIONS_PATH"]); dst.mkdir(parents=True, exist_ok=True)
    shutil.copy(f, dst / "reve_positions.json")
    print("ACCESS_OK brain-bzh/reve-positions (positions prefetched for braindecode)")
except Exception as e:
    ok = False; print("ACCESS_FAIL brain-bzh/reve-positions:", type(e).__name__, str(e)[:300])
try:                       # the pretrained weights, into the cache that REVE.from_pretrained reads
    d = snapshot_download("brain-bzh/reve-base", token=tok)
    size = sum(p.stat().st_size for p in Path(d).rglob("*") if p.is_file()) / 2**20
    print(f"ACCESS_OK brain-bzh/reve-base ({size:.0f} MB downloaded)")
except Exception as e:
    ok = False; print("ACCESS_FAIL brain-bzh/reve-base:", type(e).__name__, str(e)[:300])
sys.exit(0 if ok else 3)
"""
HF_OK = True
if globals().get("NEED_HF", True):            # the ensemble notebook needs no download
    hf = subprocess.run([str(PY), "-c", HF_CHECK], capture_output=True, text=True, env=ENV, timeout=900)
    print("Hugging Face access:\n  " + "\n  ".join((hf.stdout + hf.stderr[-600:] * (hf.returncode != 0)).strip().splitlines()))
    HF_OK = hf.returncode == 0
    if not HF_OK:
        msg = ("REVE cannot be downloaded with this token. Open https://huggingface.co/brain-bzh/reve-base (and /reve-positions) "
               "while logged in, accept the access conditions if asked, create a *read* token (Settings → Access Tokens), "
               "put it in Kaggle Add-ons → Secrets as HF_TOKEN, tick it for this notebook, and re-run this cell.")
        if SMOKE_TEST:
            print("WARNING: " + msg)
        else:
            raise RuntimeError(msg)          # stop now, not after hours of extraction

probe = subprocess.run([str(PY), "-c", "import sys, torch; print(sys.version.split()[0], torch.__version__, torch.cuda.device_count(), "
                        "[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())], "
                        "torch.cuda.get_device_capability(0) if torch.cuda.is_available() else None)"],
                       capture_output=True, text=True, env=ENV)
print("environment:", probe.stdout.strip() or probe.stderr[-1500:])
N_GPU = int(probe.stdout.split()[2]) if probe.returncode == 0 else 0
NPROC = max(1, N_GPU) if N_PROCESSES == "auto" else int(N_PROCESSES)
GPU_NAME = probe.stdout.split("[", 1)[-1].split("]")[0].strip("'") if N_GPU else "cpu"
print(f"kit {KIT_DATE[:10]} | {N_GPU} GPU(s) | training processes: {NPROC}")
if N_GPU == 0:
    print("WARNING: the environment sees no GPU: check Settings → Accelerator (GPU T4 × 2)")

## Shared code
Model code (also shipped inside the ZIP), data access, split and metrics.

In [ ]:
%%writefile {SRC}/t1_common.py
"""Shared code of the Kaggle REVE notebook: the model code (MODEL_SRC, also copied into submission.py), data access,
the agreed split and the metrics. Imported by t1_train.py and t1_package.py."""
import json
import math
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

import sys as _sys
_sys.path.insert(0, str(Path(__file__).parent))
from t1_split import *  # noqa: E402,F401,F403  (team split protocol: canonical ids, manifest, checksums)

MODEL_SRC = r'''
import json
import math
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F


def _norm_name(name):
    s = str(name).strip()
    if s.upper().startswith("EEG"):
        s = s[3:].strip()
    return s.split("-")[0].strip().upper()


class Prep(nn.Module):
    """Maps the input electrodes to the training electrodes BY NAME (missing ones -> zeros) and resamples time if needed."""

    def __init__(self, cfg):
        super().__init__()
        self.train_ch = [str(c) for c in cfg["train_ch_names"]]
        self.n_times_train = int(cfg["n_times_train"])
        n = len(self.train_ch)
        self.register_buffer("ch_idx", torch.arange(n), persistent=False)
        self.register_buffer("ch_ok", torch.ones(n), persistent=False)

    def set_input_channels(self, ch_names=None, n_chans=None):
        n_train = len(self.train_ch)
        if ch_names is not None:
            pos = {}
            for i, c in enumerate(ch_names):
                pos.setdefault(_norm_name(c), i)
            found = [pos.get(_norm_name(c), -1) for c in self.train_ch]
        else:
            n_in = n_train if n_chans is None else int(n_chans)
            found = [i if i < n_in else -1 for i in range(n_train)]
        dev = self.ch_idx.device
        self.ch_idx = torch.tensor([max(i, 0) for i in found], device=dev)
        self.ch_ok = torch.tensor([1.0 if i >= 0 else 0.0 for i in found], device=dev)
        return int(sum(i >= 0 for i in found)), n_train

    def forward(self, X):
        X = X.float()
        if X.shape[-1] != self.n_times_train:
            X = F.interpolate(X, size=self.n_times_train, mode="linear", align_corners=False)
        return X[:, self.ch_idx] * self.ch_ok[None, :, None]


def input_norm(X, mode):
    """Per-epoch input normalisation (allowed at test time: it only uses the epoch itself)."""
    if not mode:
        return X
    dims = (1, 2) if mode == "trial" else (2,)
    return (X - X.mean(dims, keepdim=True)) / X.std(dims, keepdim=True).clamp_min(1e-6)


class Projector(nn.Module):
    def __init__(self, d_in, d_out, drop=0.5):
        super().__init__()
        self.lin = nn.Linear(d_in, d_out)
        self.res = nn.Sequential(nn.GELU(), nn.Linear(d_out, d_out), nn.Dropout(drop))
        self.ln = nn.LayerNorm(d_out)

    def forward(self, x):
        x = self.lin(x)
        return self.ln(x + self.res(x))


class Conformer(nn.Module):
    """Temporal + spatial convolution patches, then a small transformer over the time tokens."""

    def __init__(self, n_ch, n_t, d_out, k=40, t_kernel=13, pool=13, stride=6, depth=2, heads=4, drop=0.3, proj_drop=0.5):
        super().__init__()
        self.patch = nn.Sequential(
            nn.Conv2d(1, k, (1, t_kernel)), nn.Conv2d(k, k, (n_ch, 1)), nn.BatchNorm2d(k), nn.ELU(),
            nn.AvgPool2d((1, pool), (1, stride)), nn.Dropout(drop))
        layer = nn.TransformerEncoderLayer(k, heads, dim_feedforward=2 * k, dropout=drop, batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(layer, depth)
        with torch.no_grad():
            n_tok = self.patch(torch.zeros(1, 1, n_ch, n_t)).shape[-1]
        self.proj = Projector(n_tok * k, d_out, proj_drop)

    def forward(self, X):
        z = self.patch(X[:, None]).squeeze(2).transpose(1, 2)      # (B, tokens, k)
        return self.proj(self.enc(z).flatten(1))


# Standard 10-05 electrode positions (metres, MNE head frame) of the 32 Emotiv electrodes of Alljoined-1.6M.
POS_1005 = {
    "CZ": (0.0004, -0.0092, 0.1002), "FCZ": (0.0004, 0.0274, 0.0887), "AFZ": (0.0002, 0.0808, 0.0354),
    "FP1": (-0.0294, 0.0839, -0.007), "F5": (-0.0645, 0.048, 0.0169), "F1": (-0.0275, 0.0569, 0.0603),
    "CP5": (-0.0796, -0.0466, 0.0309), "CP3": (-0.0636, -0.047, 0.0656), "CP1": (-0.0355, -0.0473, 0.0913),
    "P1": (-0.0286, -0.0805, 0.0754), "P3": (-0.053, -0.0788, 0.0559), "P5": (-0.0673, -0.0763, 0.0284),
    "P7": (-0.0724, -0.0735, -0.0025), "PO7": (-0.0548, -0.0975, 0.0028), "PO3": (-0.0365, -0.1009, 0.0372),
    "O1": (-0.0294, -0.1124, 0.0088), "PZ": (0.0003, -0.0811, 0.0826), "POZ": (0.0002, -0.1022, 0.0506),
    "OZ": (0.0001, -0.1149, 0.0147), "O2": (0.0298, -0.1122, 0.0088), "PO4": (0.0368, -0.1008, 0.0364),
    "PO8": (0.0557, -0.0976, 0.0027), "P8": (0.0731, -0.0731, -0.0025), "P6": (0.0679, -0.0759, 0.0281),
    "P4": (0.0557, -0.0786, 0.0566), "P2": (0.0319, -0.0805, 0.0767), "CP2": (0.0384, -0.0471, 0.0907),
    "CP4": (0.0666, -0.0466, 0.0656), "CP6": (0.0833, -0.0461, 0.0312), "F2": (0.0295, 0.0576, 0.0595),
    "F6": (0.0679, 0.0498, 0.0164), "FP2": (0.0299, 0.0849, -0.0071),
}


class ChannelMerger(nn.Module):
    """Spatial attention (Defossez et al. 2023) over the electrodes, from their Fourier-embedded 3-D positions."""

    def __init__(self, ch_names, n_virtual=270, n_freqs=10, margin=0.2, dropout=0.2):
        super().__init__()
        pos = torch.tensor([POS_1005[_norm_name(c)] for c in ch_names], dtype=torch.float32)
        pos = (pos - pos.min(0).values) / (pos.max(0).values - pos.min(0).values).clamp_min(1e-6)
        freqs = 2 * math.pi * torch.arange(n_freqs, dtype=torch.float32) / (1 + 2 * margin)
        loc = (pos + margin)[:, None, :] * freqs[None, :, None]
        grid = (loc[:, :, None, None, 0] + loc[:, None, :, None, 1] + loc[:, None, None, :, 2]).flatten(1)
        self.register_buffer("emb", torch.cat([grid.cos(), grid.sin()], 1), persistent=False)
        self.register_buffer("pos", pos, persistent=False)
        self.register_buffer("valid", torch.ones(len(ch_names), dtype=torch.bool), persistent=False)
        self.heads = nn.Parameter(torch.randn(n_virtual, self.emb.shape[1]) / self.emb.shape[1] ** 0.5)
        self.dropout = float(dropout)

    def forward(self, x):
        scores = (self.heads @ self.emb.T).float().masked_fill(~self.valid[None], float("-inf"))
        if self.training and self.dropout > 0:
            center = torch.rand(x.shape[0], 1, 3, device=x.device)
            banned = (self.pos[None] - center).norm(dim=-1) <= self.dropout
            w = torch.softmax(scores[None].masked_fill(banned[:, None, :], float("-inf")), dim=2).nan_to_num()
            return torch.bmm(w.to(x.dtype), x)
        return torch.einsum("vc,bct->bvt", torch.softmax(scores, dim=1).nan_to_num().to(x.dtype), x)


class ConvSequence(nn.Module):
    """Dilated 1-D convolutions with residual connections, batch norm, GELU and a GLU gate every `glu` layers."""

    def __init__(self, sizes, kernel=3, dilation_growth=2, dilation_period=5, glu=2, glu_context=1, conv_dropout=0.0):
        super().__init__()
        self.sequence, self.glus = nn.ModuleList(), nn.ModuleList()
        dilation = 1
        for k, (c_in, c_out) in enumerate(zip(sizes[:-1], sizes[1:])):
            if dilation_period and k % dilation_period == 0:
                dilation = 1
            layers = [nn.Conv1d(c_in, c_out, kernel, 1, kernel // 2 * dilation, dilation=dilation)]
            dilation *= dilation_growth
            if k < len(sizes) - 2:
                layers += [nn.BatchNorm1d(c_out), nn.GELU()]
                if conv_dropout:
                    layers.append(nn.Dropout(conv_dropout))
            self.sequence.append(nn.Sequential(*layers))
            self.glus.append(nn.Sequential(nn.Conv1d(c_out, 2 * c_out, 1 + 2 * glu_context, padding=glu_context), nn.GLU(dim=1))
                             if glu and (k + 1) % glu == 0 else nn.Identity())

    def forward(self, x):
        for module, glu in zip(self.sequence, self.glus):
            y = module(x)
            x = glu(y + x if y.shape == x.shape else y)
        return x


class SimpleConv(nn.Module):
    """Port of neuraltrain's SimpleConvTimeAgg (NeuralBench `simpleconv_time_agg`) for a fixed montage."""

    def __init__(self, ch_names, n_t, d_out, hidden=320, depth=4, n_virtual=270, initial_linear=270, backbone_out=512,
                 merger_dropout=0.2, conv_dropout=0.0, readout="linear", pool=13, stride=6, proj_drop=0.5):
        super().__init__()
        self.merger = ChannelMerger(ch_names, n_virtual=n_virtual, dropout=merger_dropout)
        self.initial = nn.Conv1d(n_virtual, initial_linear, 1)
        self.encoder = ConvSequence([initial_linear] + [hidden] * (depth - 1) + [backbone_out], conv_dropout=conv_dropout)
        self.readout = readout
        if readout == "linear":
            self.time_agg = nn.Linear(n_t, 1)
            self.head = nn.Linear(backbone_out, d_out)
        else:
            self.pool = nn.AvgPool1d(pool, stride)
            self.head = Projector(((n_t - pool) // stride + 1) * backbone_out, d_out, proj_drop)

    def forward(self, X):
        x = self.encoder(self.initial(self.merger(X)))
        if self.readout == "linear":
            return self.head(self.time_agg(x).squeeze(-1))
        return self.head(self.pool(x).flatten(1))


class ResMLP(nn.Module):
    """Residual MLP on the epoch averaged in bins of `pool` samples."""

    def __init__(self, n_ch, n_t, d_out, width=2048, depth=4, pool=4, drop_in=0.5, drop=0.15):
        super().__init__()
        self.pool = pool
        self.inp = nn.Sequential(nn.Linear(n_ch * (n_t // pool), width), nn.LayerNorm(width), nn.GELU(), nn.Dropout(drop_in))
        self.blocks = nn.ModuleList([nn.Sequential(nn.Linear(width, width), nn.LayerNorm(width), nn.GELU(), nn.Dropout(drop))
                                     for _ in range(depth)])
        self.out = nn.Linear(width, d_out)

    def forward(self, X):
        x = self.inp(F.avg_pool1d(X, self.pool).flatten(1))
        for b in self.blocks:
            x = x + b(x)
        return self.out(x)


# ---------------------------------------------------------------------------------------------------------------
# REVE encoder, vendored from braindecode 1.8.1 (braindecode/models/reve.py, BSD-3-Clause, authors: Jonathan Lys;
# model: El Ouahidi et al. 2025, "REVE: a foundation model for EEG"). Only the forward path used here is kept
# (patch embedding, 4-D positional encoding, transformer): parameter names are identical, so braindecode
# checkpoints load as they are, and the submission needs neither braindecode nor any download at inference.
# einops calls are replaced by the equivalent reshape / transpose.
# ---------------------------------------------------------------------------------------------------------------
class _GEGLU(nn.Module):
    def forward(self, x):
        x, gates = x.chunk(2, dim=-1)
        return F.gelu(gates) * x


class _RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        y = x.float()
        return (y * torch.rsqrt(y.pow(2).mean(-1, keepdim=True) + self.eps)).type_as(x) * self.weight


class _FeedForward(nn.Module):
    def __init__(self, dim, hidden_dim):
        super().__init__()
        self.net = nn.Sequential(_RMSNorm(dim), nn.Linear(dim, hidden_dim * 2, bias=False), _GEGLU(),
                                 nn.Linear(hidden_dim, dim, bias=False))

    def forward(self, x):
        return self.net(x)


class _Attention(nn.Module):
    def __init__(self, dim, heads=8, head_dim=64):
        super().__init__()
        self.heads = heads
        self.norm = _RMSNorm(dim)
        self.to_qkv = nn.Linear(dim, heads * head_dim * 3, bias=False)
        self.to_out = nn.Linear(heads * head_dim, dim, bias=False)

    def forward(self, x):
        b, s, _ = x.shape
        q, k, v = (t.reshape(b, s, self.heads, -1).transpose(1, 2) for t in self.to_qkv(self.norm(x)).chunk(3, dim=-1))
        out = F.scaled_dot_product_attention(q, k, v)
        return self.to_out(out.transpose(1, 2).reshape(b, s, -1))


class _Transformer(nn.Module):
    def __init__(self, dim, depth, heads, head_dim, mlp_dim):
        super().__init__()
        self.layers = nn.ModuleList([nn.ModuleList([_Attention(dim, heads, head_dim), _FeedForward(dim, mlp_dim)])
                                     for _ in range(depth)])

    def forward(self, x, keep=None):
        outs = []
        for i, (attn, ff) in enumerate(self.layers):
            x = attn(x) + x
            x = ff(x) + x
            if keep is not None and i in keep:
                outs.append(x)
        return x if keep is None else (x, outs)


class _FourierEmb4D(nn.Module):
    def __init__(self, dimension, freqs, increment_time=0.1, margin=0.4):
        super().__init__()
        self.dimension, self.freqs, self.increment_time, self.margin = dimension, freqs, increment_time, margin

    def forward(self, positions_):
        positions = positions_.clone()
        positions[:, :, -1] *= self.increment_time
        batch_dims = list(positions.shape[:-1])
        freqs_w = torch.arange(self.freqs).to(positions)
        freqs_z = freqs_w[:, None]
        freqs_y = freqs_z[:, None]
        freqs_x = freqs_y[:, None]
        width = 1 + 2 * self.margin
        positions = positions + self.margin
        p_x, p_y, p_z, p_w = (2 * math.pi * f / width for f in (freqs_x, freqs_y, freqs_z, freqs_w))
        positions = positions[..., None, None, None, None, :]
        loc = positions[..., 0] * p_x + positions[..., 1] * p_y + positions[..., 2] * p_z + positions[..., 3] * p_w
        loc = loc.view(batch_dims + [-1])[..., :self.dimension // 2]
        return torch.cat([torch.cos(loc), torch.sin(loc)], dim=-1)

    @staticmethod
    def add_time_patch(pos, num_patches):
        batch, nchans, _ = pos.shape
        pos_repeated = pos.unsqueeze(2).repeat(1, 1, num_patches, 1)
        t = torch.arange(0, num_patches, 1, device=pos.device).float().view(1, 1, num_patches, 1).expand(batch, nchans, num_patches, 1)
        return torch.cat((pos_repeated, t), dim=-1).view(batch, nchans * num_patches, 4)


class ReveEncoder(nn.Module):
    """REVE-base encoder (braindecode 1.8.1 names and defaults): (B, C, T at 200 Hz) + positions (B, C, 3) -> (B, C, P, 512)."""

    def __init__(self, embed_dim=512, depth=22, heads=8, head_dim=64, mlp_dim_ratio=2.66, freqs=4, patch_size=200, patch_overlap=20):
        super().__init__()
        self.embed_dim, self.depth, self.patch_size, self.patch_overlap = embed_dim, depth, patch_size, patch_overlap
        self.to_patch_embedding = nn.Sequential(nn.Linear(patch_size, embed_dim))
        self.fourier4d = _FourierEmb4D(embed_dim, freqs=freqs)
        self.mlp4d = nn.Sequential(nn.Linear(4, embed_dim, bias=False), nn.GELU(), nn.LayerNorm(embed_dim))
        self.ln = nn.LayerNorm(embed_dim)
        self.transformer = _Transformer(embed_dim, depth, heads, head_dim, int(embed_dim * mlp_dim_ratio))

    def forward(self, eeg, pos, keep=None):
        """keep: indices of transformer layers whose outputs are also returned (attention-pooling head)."""
        patches = eeg.unfold(dimension=2, size=self.patch_size, step=self.patch_size - self.patch_overlap)
        b, c, p, _ = patches.shape
        pos4 = _FourierEmb4D.add_time_patch(pos, p)
        x = self.to_patch_embedding(patches).reshape(b, c * p, self.embed_dim) + self.ln(self.fourier4d(pos4) + self.mlp4d(pos4))
        if keep is None:
            return self.transformer(x).reshape(b, c, p, self.embed_dim)
        x, outs = self.transformer(x, keep)
        return x.reshape(b, c, p, self.embed_dim), outs


class AttnPool(nn.Module):
    """Learned queries cross-attending over the tokens of several REVE layers (REVE's own pretraining pools all layers
    with an attention query); a learned embedding tells the layers apart."""

    def __init__(self, dim, n_layers, n_queries=4, heads=8, d_out=1536, drop=0.5):
        super().__init__()
        self.q = nn.Parameter(torch.randn(n_queries, dim) * 0.02)
        self.layer_emb = nn.Parameter(torch.zeros(n_layers, 1, dim))
        self.norm = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.out = Projector(n_queries * dim, d_out, drop)

    def forward(self, outs):
        t = torch.cat([o + e for o, e in zip(outs, self.layer_emb)], dim=1)       # (B, layers x tokens, D)
        t = self.norm(t)
        z, _ = self.attn(self.q[None].expand(t.shape[0], -1, -1).to(t.dtype), t, t, need_weights=False)
        return self.out(z.flatten(1))


class ReveBackbone(nn.Module):
    """REVE-base encoder -> 32 electrode tokens -> 64 numbers each -> projector to the embedding.

    The 120-Hz epoch is resampled to 200 samples (REVE's rate), each electrode is z-scored and clamped at ±15 (as in
    the official REVE example); a 1-s window makes ONE 512-d token per electrode, placed by its 3-D position.
    `pos` (the electrodes' positions from REVE's position bank) is part of the weights: nothing is looked up or
    downloaded at inference. `pretrained` and `freeze` are training-time flags, ignored here.
    """
    SFREQ, N_TIMES = 200, 200

    def __init__(self, ch_names, n_t, d_out, tok_dim=64, proj_drop=0.5, reve_kw=None, pretrained=True, freeze=False,
                 head="tokens", attn_layers=None, n_queries=4, n_times_model=200):
        super().__init__()
        self.reve = ReveEncoder(**(reve_kw or {}))
        self.register_buffer("pos", torch.zeros(len(ch_names), 3))
        self.n_times_model, self.head_kind = int(n_times_model), head
        n_patches = (self.n_times_model - self.reve.patch_size) // (self.reve.patch_size - self.reve.patch_overlap) + 1
        dim = self.reve.embed_dim
        if "tokens" in head:
            self.down = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, tok_dim), nn.GELU())
            self.head = Projector(len(ch_names) * n_patches * tok_dim, d_out, proj_drop)
        self.keep = None
        if "attn" in head:
            d = self.reve.depth
            self.keep = sorted({(int(i) + d) % d for i in (attn_layers if attn_layers is not None else range(d))})
            self.pool = AttnPool(dim, len(self.keep), n_queries, d_out=d_out, drop=proj_drop)

    def forward(self, X):
        X = F.interpolate(X.float(), size=self.n_times_model, mode="linear", align_corners=False)
        X = ((X - X.mean(-1, keepdim=True)) / X.std(-1, keepdim=True, unbiased=False).clamp_min(1e-6)).clamp(-15.0, 15.0)
        pos = self.pos.expand(X.shape[0], -1, -1)
        if self.keep is None:
            f = self.reve(X, pos)                                                   # (B, C, P, D)
            return self.head(self.down(f).flatten(1))
        f, outs = self.reve(X, pos, self.keep)
        z = self.pool(outs)
        return z + self.head(self.down(f).flatten(1)) if "tokens" in self.head_kind else z


def region_of(name):
    """Coarse tri-region split (AMSPBE): 0 frontal (Fp/AF/F/FC/FT), 1 central (C/CP/T/TP/other), 2 posterior (P/PO/O/I)."""
    import re
    m = re.match(r"[A-Z]+", _norm_name(name))
    letters = m.group(0) if m else ""
    if len(letters) > 1 and letters.endswith("Z"):
        letters = letters[:-1]
    if letters in ("FP", "AF", "F", "FC", "FT"):
        return 0
    if letters in ("P", "PO", "O", "I"):
        return 2
    return 1


class AbsConv2d(nn.Conv2d):
    """Convolution whose weights are used as |W|: every spatial filter is a non-negative combination of electrodes."""

    def forward(self, x):
        return F.conv2d(x, self.weight.abs(), self.bias, self.stride, self.padding)


class BottleneckHead(nn.Module):
    def __init__(self, d_in, d_out, hidden=384, drop=0.5):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in, hidden), nn.LayerNorm(hidden), nn.GELU(), nn.Dropout(drop), nn.Linear(hidden, d_out))

    def forward(self, x):
        return self.net(x)


class AMSPBEEncoder(nn.Module):
    """Attentive Modulated Scalp-Pattern Bank Encoder (team port of the DTHA paper's brain encoder): tri-region
    temporal attention x channel attention, depthwise temporal conv, K non-negative spatial filters, bottleneck readout."""

    def __init__(self, regions, n_t, d_out, K=16, t_kernel=7, t_pool=60, drop=0.3, proj_drop=0.5, bottleneck=384, use_tsam=True,
                 use_dwconv=True):
        super().__init__()
        n_ch = len(regions)
        self.use_tsam, self.use_dwconv = bool(use_tsam), bool(use_dwconv)
        self.register_buffer("region_idx", torch.as_tensor(regions, dtype=torch.long), persistent=False)
        self.time_att = nn.Parameter(torch.ones(3, n_t))
        self.chan_att = nn.Parameter(torch.ones(n_ch, 1))
        self.dw = nn.Conv1d(n_ch, n_ch, t_kernel, padding=t_kernel // 2, groups=n_ch)
        self.act = nn.ELU()
        self.pool = nn.AdaptiveAvgPool1d(t_pool)
        self.bank = AbsConv2d(1, K, (n_ch, 1))
        self.bn = nn.BatchNorm2d(K)
        self.drop = nn.Dropout(drop)
        self.proj = (BottleneckHead(K * t_pool, d_out, int(bottleneck), proj_drop) if int(bottleneck) > 0
                     else Projector(K * t_pool, d_out, proj_drop))

    def forward(self, x):
        if self.use_tsam:
            x = x * (self.time_att[self.region_idx] * self.chan_att)[None]
        if self.use_dwconv:
            x = self.act(self.dw(x))
        x = self.pool(x)
        x = self.bn(self.bank(x[:, None]))
        return self.proj(self.drop(x.flatten(1)))


class AMSPBE(nn.Module):
    """n_members independent AMSPBE encoders. Inference: mean of the L2-normalised member outputs. Training:
    members_forward, one augmented view per member (each member gets its own loss)."""

    def __init__(self, ch_names, n_t, d_out, n_members=4, **kw):
        super().__init__()
        regions = [region_of(c) for c in ch_names]
        self.members = nn.ModuleList([AMSPBEEncoder(regions, n_t, d_out, **kw) for _ in range(int(n_members))])

    def forward(self, x):
        return torch.stack([F.normalize(m(x).float(), dim=-1) for m in self.members]).mean(0)

    def members_forward(self, xs):
        return [m(x).float() for m, x in zip(self.members, xs)]


class _ResBlock1d(nn.Module):
    def __init__(self, width, kernel=9, dilation=1, dropout=0.15):
        super().__init__()
        pad = dilation * (kernel // 2)
        self.net = nn.Sequential(nn.Conv1d(width, width, kernel, padding=pad, dilation=dilation, groups=width, bias=False),
                                 nn.BatchNorm1d(width), nn.GELU(), nn.Conv1d(width, width, 1, bias=False), nn.BatchNorm1d(width),
                                 nn.Dropout(dropout))

    def forward(self, x):
        return F.gelu(x + self.net(x))


class ResCNN(nn.Module):
    """Residual temporal CNN (the 'p13' public model): conv stem over all electrodes, 3 dilated depthwise-separable
    residual blocks, average pooling to `pool` steps, MLP readout."""

    def __init__(self, ch_names, n_t, d_out, width=96, stem_kernel=15, kernel=9, dilations=(1, 2, 4), dropout=0.15, pool=12,
                 hidden=512, head_drop=0.2):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv1d(len(ch_names), width, stem_kernel, padding=stem_kernel // 2, bias=False),
                                  nn.BatchNorm1d(width), nn.GELU())
        self.blocks = nn.Sequential(*[_ResBlock1d(width, kernel, int(d), dropout) for d in dilations])
        self.head = nn.Sequential(nn.AdaptiveAvgPool1d(pool), nn.Flatten(), nn.Linear(width * pool, hidden), nn.GELU(),
                                  nn.Dropout(head_drop), nn.Linear(hidden, d_out))

    def forward(self, x):
        return self.head(self.blocks(self.stem(x.float())))


class _MSDilatedBlock(nn.Module):
    """Multi-scale dilated temporal block: parallel depthwise convolutions with several dilations, mixed by a 1x1
    convolution, residual."""

    def __init__(self, width, kernel=5, dilations=(1, 2, 4, 8), dropout=0.15):
        super().__init__()
        self.branches = nn.ModuleList([nn.Conv1d(width, width, kernel, padding=int(d) * (kernel // 2), dilation=int(d), groups=width,
                                                 bias=False) for d in dilations])
        self.mix = nn.Sequential(nn.BatchNorm1d(width * len(dilations)), nn.GELU(), nn.Conv1d(width * len(dilations), width, 1, bias=False),
                                 nn.BatchNorm1d(width), nn.Dropout(dropout))

    def forward(self, x):
        return F.gelu(x + self.mix(torch.cat([b(x) for b in self.branches], 1)))


class TCNLC(nn.Module):
    """Multi-scale dilated TCN with two readouts: (a) the temporal profile (average pooling to `pool` steps) and
    (b) log-covariance pooling: the covariance over time of a `cov_dim`-channel projection of the features, shrunk,
    mapped by the matrix logarithm (eigendecomposition, float32) and vectorised (upper triangle). Both are
    concatenated and read out by an MLP."""

    def __init__(self, ch_names, n_t, d_out, width=96, stem_kernel=7, kernel=5, dilations=(1, 2, 4, 8), n_blocks=3, dropout=0.15,
                 pool=8, cov_dim=16, shrink=0.05, hidden=512, head_drop=0.2):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv1d(len(ch_names), width, stem_kernel, padding=stem_kernel // 2, bias=False),
                                  nn.BatchNorm1d(width), nn.GELU())
        self.blocks = nn.Sequential(*[_MSDilatedBlock(width, kernel, dilations, dropout) for _ in range(int(n_blocks))])
        self.tpool = nn.Sequential(nn.AdaptiveAvgPool1d(pool), nn.Flatten())
        self.cov_proj = nn.Conv1d(width, cov_dim, 1, bias=False)
        self.cov_dim, self.shrink = int(cov_dim), float(shrink)
        iu = torch.triu_indices(self.cov_dim, self.cov_dim)
        self.register_buffer("iu0", iu[0].clone(), persistent=False)
        self.register_buffer("iu1", iu[1].clone(), persistent=False)
        n_cov = self.cov_dim * (self.cov_dim + 1) // 2
        self.cov_norm = nn.LayerNorm(n_cov)
        self.head = nn.Sequential(nn.Linear(width * pool + n_cov, hidden), nn.GELU(), nn.Dropout(head_drop), nn.Linear(hidden, d_out))

    def log_cov(self, h):
        with torch.autocast(device_type=h.device.type, enabled=False):
            z = self.cov_proj(h.float()).float()
            z = z - z.mean(-1, keepdim=True)
            C = z @ z.transpose(1, 2) / max(1, z.shape[-1] - 1)
            eye = torch.eye(self.cov_dim, device=C.device, dtype=C.dtype)
            tr = C.diagonal(dim1=1, dim2=2).mean(-1).clamp_min(1e-6)[:, None, None]
            C = (1 - self.shrink) * C / tr + self.shrink * eye + 1e-4 * eye
            ev, U = torch.linalg.eigh(C)
            L = (U * torch.log(ev.clamp_min(1e-5))[:, None, :]) @ U.transpose(1, 2)
            off = torch.where(self.iu0 == self.iu1, 1.0, math.sqrt(2.0)).to(L.dtype)
            return self.cov_norm(L[:, self.iu0, self.iu1] * off)

    def forward(self, x):
        h = self.blocks(self.stem(x.float()))
        return self.head(torch.cat([self.tpool(h).float(), self.log_cov(h)], 1))


def make_backbone(cfg):
    ch, n_t, d_out = [str(c) for c in cfg["train_ch_names"]], int(cfg["n_times_train"]), int(cfg["d_out"])
    kw = dict(cfg.get("arch_kw") or cfg.get("conformer") or {}) if cfg.get("arch", "conformer") != "conformer" else dict(cfg.get("conformer") or {})
    arch = cfg.get("arch", "conformer")
    if arch == "conformer":
        return Conformer(len(ch), n_t, d_out, **kw)
    if arch == "simpleconv":
        return SimpleConv(ch, n_t, d_out, **kw)
    if arch == "resmlp":
        return ResMLP(len(ch), n_t, d_out, **kw)
    if arch == "reve":
        return ReveBackbone(ch, n_t, d_out, **kw)
    if arch == "amspbe":
        return AMSPBE(ch, n_t, d_out, **kw)
    if arch == "rescnn":
        return ResCNN(ch, n_t, d_out, **kw)
    if arch == "tcnlc":
        return TCNLC(ch, n_t, d_out, **kw)
    raise ValueError(f"unknown arch {arch!r}")


def time_shift(X, s):
    """Epoch shifted by s samples (edges repeated), as the training augmentation does."""
    if not s:
        return X
    idx = (torch.arange(X.shape[-1], device=X.device) - int(s)).clamp(0, X.shape[-1] - 1)
    return X[..., idx]


class Net(nn.Module):
    """backbone(epoch) [+ backbone_rev(time-reversed epoch)], followed by a fixed post-calibration."""

    def __init__(self, cfg, device="cpu"):
        super().__init__()
        self.cfg = dict(cfg)
        self.prep = Prep(cfg)
        d_out = int(cfg["d_out"])
        self.backbone = make_backbone(cfg)
        self.backbone_rev = make_backbone(cfg) if cfg.get("bidir") else None
        aux = cfg.get("aux")                         # a second, jointly trained branch (e.g. the bidirectional Conformer)
        if aux:
            acfg = {**cfg, **aux}
            self.aux = make_backbone(acfg)
            self.aux_rev = make_backbone(acfg) if aux.get("bidir") else None
        self.register_buffer("out_offset", torch.zeros(d_out))
        self.register_buffer("pred_mean", torch.zeros(d_out))
        self.register_buffer("gal_dir", torch.zeros(d_out))
        if (cfg.get("post") or {}).get("whiten"):
            self.register_buffer("post_W", torch.eye(d_out))

    def set_input_channels(self, ch_names=None, n_chans=None):
        res = self.prep.set_input_channels(ch_names, n_chans)
        for m in self.modules():
            if isinstance(m, ChannelMerger):
                m.valid = self.prep.ch_ok.bool()
        return res

    def forward(self, X):
        Xp = input_norm(self.prep(X), self.cfg.get("input_norm"))
        z = self.backbone(Xp).float()
        if self.backbone_rev is not None:
            z = z + self.backbone_rev(torch.flip(Xp, [-1])).float()
        if getattr(self, "aux", None) is not None:
            z = z + self.aux(Xp).float()
            if self.aux_rev is not None:
                z = z + self.aux_rev(torch.flip(Xp, [-1])).float()
        return z + self.out_offset

    def members_forward(self, Xs):
        """AMSPBE training: one (augmented) raw input per member -> list of member outputs."""
        return self.backbone.members_forward([input_norm(self.prep(x), self.cfg.get("input_norm")) for x in Xs])

    def post_apply(self, Z):
        p = self.cfg.get("post")
        Z = Z.float()
        if not p:
            return Z
        if p.get("center"):
            Z = Z - self.pred_mean
        if p.get("whiten"):                          # fixed matrix from TRAINING images (anti-anisotropy / hubness)
            Z = F.normalize(Z, dim=-1) @ self.post_W
        Z = F.normalize(Z, dim=-1) if p.get("norm") == "trial" else Z / float(p.get("gscale", 1.0))
        return Z + float(p.get("lam", 0.0)) * self.gal_dir

    @torch.inference_mode()
    def predict(self, X):
        """(B, C, T) -> (B, 1536) on the model's device; mixed precision on a GPU; optional time-shift TTA."""
        self.eval()
        dev = self.pred_mean.device
        X = torch.as_tensor(X).to(dev).float()
        amp = None                                   # bfloat16 on Ampere and newer (the sealed RTX PRO 4500), float16 on a T4
        if dev.type == "cuda" and self.cfg.get("amp", True):
            amp = torch.bfloat16 if torch.cuda.get_device_capability(dev)[0] >= 8 else torch.float16
        shifts = [int(s) for s in (self.cfg.get("tta_shifts") or [0])]
        out = 0.0
        for s in shifts:
            Z = []
            for i in range(0, X.shape[0], 256):
                with torch.autocast("cuda", dtype=amp or torch.float16, enabled=amp is not None):
                    Z.append(self.forward(time_shift(X[i:i + 256], s)).float())
            out = out + self.post_apply(torch.cat(Z))
        return out / len(shifts)


class Ensemble(nn.Module):
    """Mean of the members' L2-normalised predictions (each member keeps its own input mapping and post-calibration)."""

    def __init__(self, cfg, device="cpu"):
        super().__init__()
        self.cfg = dict(cfg)
        self.members = nn.ModuleList([Net(c) for c in cfg["members"]])
        w = [float(v) for v in (cfg.get("member_weights") or [1.0] * len(self.members))]
        self.register_buffer("w", torch.tensor(w) / sum(w), persistent=False)

    def set_input_channels(self, ch_names=None, n_chans=None):
        res = [m.set_input_channels(ch_names, n_chans) for m in self.members]
        return min(r[0] for r in res), res[0][1]

    def forward(self, X):
        raise NotImplementedError("an Ensemble is used through predict(X)")

    @torch.inference_mode()
    def predict(self, X):
        out = 0.0
        for wi, m in zip(self.w.tolist(), self.members):
            out = out + wi * F.normalize(m.predict(X), dim=-1)
        return out


def build_model(cfg, device="cpu"):
    return (Ensemble(cfg) if cfg.get("members") else Net(cfg)).to(device)
'''

exec(MODEL_SRC, globals())

CH_NAMES = "Cz FCz AFz Fp1 F5 F1 CP5 CP3 CP1 P1 P3 P5 P7 PO7 PO3 O1 Pz POz Oz O2 PO4 PO8 P8 P6 P4 P2 CP2 CP4 CP6 F2 F6 Fp2".split()
SPLITS = ("train", "val", "test")


def load_cfg(path):
    return json.loads(Path(path).read_text())


def log(*a):
    print(time.strftime("%H:%M:%S"), *a, flush=True)


class TrialPool:
    """All epochs of the three official splits behind one global index (memory-mapped float16)."""

    def __init__(self, root):
        root = Path(root)
        self.X = [np.load(root / f"X_{s}.npy", mmap_mode="r") for s in SPLITS]
        self.off = np.concatenate([[0], np.cumsum([len(x) for x in self.X])]).astype(np.int64)
        self.img = np.concatenate([np.load(root / f"img_{s}.npy") for s in SPLITS]).astype(np.int64)
        self.subj = np.concatenate([np.load(root / f"subj_{s}.npy") for s in SPLITS]).astype(np.int64)
        self.split = np.concatenate([np.full(len(x), i, np.int8) for i, x in enumerate(self.X)])
        self.shape = self.X[0].shape[1:]

    def __len__(self):
        return int(self.off[-1])

    def get(self, g):
        g = np.asarray(g, dtype=np.int64)
        out = np.empty((len(g),) + self.shape, np.float16)
        for s in range(3):
            m = np.flatnonzero((g >= self.off[s]) & (g < self.off[s + 1]))
            if len(m):
                loc = g[m] - self.off[s]
                o = np.argsort(loc, kind="stable")
                out[m[o]] = self.X[s][loc[o]]
        return out


def load_trials(cache):
    """Per-epoch metadata in the global order: dataset subject label, session, run, onset, stimulus path."""
    want = ("subject", "session", "run", "start", "filepath")
    parts = []
    for s in SPLITS:
        f = Path(cache) / f"trials_{s}.parquet"
        cols = [c for c in want if c in pq_columns(f)]
        parts.append(pd.read_parquet(f, columns=cols).assign(official_split=s))
    t = pd.concat(parts, ignore_index=True)
    if "filepath" not in t.columns:
        raise RuntimeError("the trial tables have no `filepath` column: the concept split needs the stimulus file names")
    return t


def pq_columns(f):
    import pyarrow.parquet as pq
    return set(pq.read_schema(f).names)


def concept_of(path):
    """THINGS stimulus path '.../aardvark/aardvark_01b.jpg' -> 'aardvark'."""
    stem = str(path).replace("\\", "/").rsplit("/", 1)[-1].rsplit(".", 1)[0]
    return re.sub(r"_\d+[a-z]?$", "", stem).lower()


def subject_label(v):
    """Dataset subject label, whatever the source writes: 'Xu2025Alljoined/9', 'sub-09', '09', 9 -> '9'."""
    s = re.sub(r"^sub-?", "", str(v).strip().replace("\\", "/").rsplit("/", 1)[-1], flags=re.I)
    return s.lstrip("0") or "0"


def _natural(x):
    return (0, int(x), "") if str(x).isdigit() else (1, 0, str(x))


def _stem(path):
    return str(path).replace("\\", "/").rsplit("/", 1)[-1].rsplit(".", 1)[0]


def canonical_ids(P, trials, emb_n, n_concepts=N_TRAIN_CONCEPTS):
    """Per cache image: canonical image_id and concept_id (see t1_split). Alljoined stimuli are numbered files; the
    training pool must be n_concepts x 10 consecutive numbers, and the blocks of 10 (= THINGS-EEG2 concepts) are CHECKED
    on the DINOv2 embeddings: images of one block are more alike than images of blocks shifted by 1..9 positions."""
    first = pd.DataFrame({"img": P.img, "path": trials["filepath"].to_numpy(), "test": P.split == 2}).drop_duplicates("img")
    stems = first["path"].map(_stem)
    numbered = stems.str.fullmatch(r"\d+")
    if not numbered.all():
        raise RuntimeError(f"stimulus files are not numbered (e.g. {stems[~numbered].iloc[0]!r}): "
                           "the canonical ids expect Alljoined's numbered stimuli")
    is_t = first["test"].to_numpy()
    tr_img, tr_num = first["img"].to_numpy()[~is_t], stems.to_numpy()[~is_t].astype(int)
    n0 = int(tr_num.min())
    iid = tr_num - n0
    n_pool = n_concepts * IMAGES_PER_CONCEPT
    assert np.array_equal(np.sort(iid), np.arange(n_pool)), (
        f"the training pool is not {n_pool} consecutive numbers ({len(iid)} images, {iid.min()}..{iid.max()})")
    E = emb_n[tr_img]

    def within(offset):
        b = (iid + offset) // IMAGES_PER_CONCEPT
        sums = np.zeros((b.max() + 1, E.shape[1])); np.add.at(sums, b, E)
        k = np.bincount(b).astype(float); ok = k > 1
        return float((((sums ** 2).sum(1) - k)[ok] / (k * (k - 1))[ok]).mean())

    sc = {o: within(o) for o in range(IMAGES_PER_CONCEPT)}
    best = max(sc, key=sc.get)
    others = float(np.median([v for o, v in sc.items() if o != best]))
    assert best == 0 and sc[0] - others > 0.03, f"numbered stimuli do not form blocks of 10 (within-block cosine by offset: {sc})"
    te_img, te_num = first["img"].to_numpy()[is_t], stems.to_numpy()[is_t].astype(int) - n0
    if len(te_num) and te_num.min() >= n_pool:
        te_id, rule = te_num, "official test pool: stimulus number as is"
    else:
        te_id, rule = n_pool + np.argsort(np.argsort(te_num)), "official test pool: numbered after the training pool, in file order"
    size = int(P.img.max()) + 1
    img_lut, con_lut = np.full(size, -1, np.int64), np.full(size, -1, np.int64)
    img_lut[tr_img], con_lut[tr_img] = iid, iid // IMAGES_PER_CONCEPT + 1
    img_lut[te_img], con_lut[te_img] = te_id, 10000 + (te_id - n_pool) + 1        # official test concept k -> 10000 + k
    info = dict(rule="image_id = stimulus number - first number; concept_id = image_id // 10 + 1 (THINGS-EEG2 order)",
                first_number=n0, n_train_pool=int(len(iid)), n_test_pool=int(len(te_id)), test_rule=rule,
                dinov2_within_block_cosine=round(sc[0], 4), dinov2_shifted_block_cosine=round(others, 4))
    return img_lut, con_lut, info


def build_split(P, trials, cfg, emb=None):
    """TRAIN (= FIT) / VAL / TEST from the frozen manifest (cfg['split_manifest']): an EEG epoch belongs to a split when its
    person AND its image belong to it. NEW = local-test people x the official test pool (Codabench-like gallery, report only)."""
    man = json.loads(Path(cfg["split_manifest"]).read_text())
    m, n_concepts = man["splits"], int(man.get("n_train_concepts", N_TRAIN_CONCEPTS))
    subj_id = np.array([int(subject_label(v)) for v in (trials["subject"] if "subject" in trials else pd.Series(P.subj))])
    if emb is None:
        emb = np.load(Path(cfg["cache"]) / "emb_table.npy").astype(np.float32)
    img_lut, con_lut, cinfo = canonical_ids(P, trials, emb / np.linalg.norm(emb, axis=1, keepdims=True), n_concepts)
    img_id, con_id = img_lut[P.img], con_lut[P.img]
    pool_test = P.split == 2
    allg = np.arange(len(P))
    S = {}
    for name, key in (("FIT", "train"), ("VAL", "val"), ("TEST", "test")):
        S[name] = allg[np.isin(subj_id, m[key]["subjects"]) & np.isin(img_id, m[key]["images"]) & ~pool_test]
    S["NEW"] = allg[np.isin(subj_id, m["test"]["subjects"]) & pool_test]
    S["ALL"] = allg
    for name, key in (("FIT", "train"), ("VAL", "val"), ("TEST", "test")):   # every person x image cell has EEG
        got_s = set(np.unique(subj_id[S[name]]).tolist())
        assert got_s == set(m[key]["subjects"]), f"{name}: subjects in the data {sorted(got_s)} != manifest {m[key]['subjects']}"
        cells = len(np.unique(subj_id[S[name]].astype(np.int64) * 100000 + img_id[S[name]]))
        need = len(m[key]["subjects"]) * len(m[key]["images"])
        assert cells == need, f"{name}: {need - cells} person x image cells of the manifest have no EEG in the cache"
    for a, b in (("FIT", "VAL"), ("FIT", "TEST"), ("VAL", "TEST")):
        assert not set(subj_id[S[a]]) & set(subj_id[S[b]]), f"a person is in {a} and {b}"
        assert not set(con_id[S[a]]) & set(con_id[S[b]]), f"a concept is in {a} and {b}"
    used = np.zeros(len(P), bool)
    for k in ("FIT", "VAL", "TEST", "NEW"):
        used[S[k]] = True
    exp = man.get("expected_epochs") or {}
    ours = dict(FIT=len(S["FIT"]), VAL=len(S["VAL"]), official_train=int((P.split == 0).sum()), official_val=int((P.split == 1).sum()))
    vs_team = {k: dict(ours=int(ours[k]), team=int(v), same=int(ours[k]) == int(v)) for k, v in exp.items() if k in ours}
    info = dict(source=man.get("source"), checksums=man["checksums"], team_science_sha256=man.get("team_science_sha256"),
                epochs_vs_team_loader=vs_team, concepts=cinfo,
                subjects={k: m[k]["subjects"] for k in SPLITS3}, unused_epochs=int((~used).sum()),
                sets={k: dict(epochs=int(len(v)), people=int(len(np.unique(subj_id[v]))), images=int(len(np.unique(img_id[v]))),
                              concepts=int(len(np.unique(con_id[v])))) for k, v in S.items()})
    return S, info, dict(subject_id=subj_id, image_id=img_id, concept_id=con_id)


def agg(pred, gidx, P, k=None, seed=0, max_chunks=None):
    """Mean prediction per person x image: over all repeats (k=None), or one query per disjoint chunk of k repeats."""
    key = P.subj[gidx] * (int(P.img.max()) + 1) + P.img[gidx]
    o = np.lexsort((np.random.default_rng(seed).random(len(gidx)), key)); sk = key[o]
    starts = np.flatnonzero(np.r_[True, sk[1:] != sk[:-1]]); sizes = np.diff(np.r_[starts, len(sk)])
    pos = np.arange(len(sk)) - np.repeat(starts, sizes)
    gid = np.repeat(np.arange(len(starts)), sizes)
    if k is None:
        keep, qkey = np.ones(len(sk), bool), gid
    else:
        c = pos // k
        n_chunks = np.maximum(sizes // k, 1)
        if max_chunks:
            n_chunks = np.minimum(n_chunks, max_chunks)
        keep = c < np.repeat(n_chunks, sizes)
        qkey = gid * (int(c.max()) + 1) + c
    rows = o[keep]
    uq, qid = np.unique(qkey[keep], return_inverse=True); qid = np.asarray(qid).reshape(-1)
    x = torch.as_tensor(np.asarray(pred[rows], dtype=np.float32))
    sums = torch.zeros(len(uq), x.shape[1]).index_add_(0, torch.as_tensor(qid), x)
    cnt = torch.zeros(len(uq)).index_add_(0, torch.as_tensor(qid), torch.ones(len(rows)))
    first = np.zeros(len(uq), np.int64); first[qid[::-1]] = rows[::-1]
    return dict(mean=sums / cnt[:, None], subj=P.subj[gidx][first], img=P.img[gidx][first])


def rank_in(A, gal, EMB_N_T):
    """Rank (0 = first) of each query's true image among the gallery images `gal` (sorted ids), by cosine."""
    dev = EMB_N_T.device
    G = EMB_N_T[torch.as_tensor(gal, device=dev)]
    tpos = torch.as_tensor(np.searchsorted(gal, A["img"]), device=dev)
    ranks = []
    for s in range(0, len(tpos), 2048):
        sc = F.normalize(A["mean"][s:s + 2048].to(dev), dim=-1) @ G.T
        true = sc.gather(1, tpos[s:s + 2048, None])
        ranks.append(((sc > true).sum(1).float() + ((sc == true).sum(1).float() - 1) / 2).cpu())
    r = torch.cat(ranks).numpy()
    return r, r / max(1, len(gal) - 1)


def scores(pred, gidx, P, EMB_N_T, ks=(None,)):
    """MRP (mean rank percentile, 0.5 = chance) and subject-aggregated top-5 among the set's own images."""
    gal = np.unique(P.img[gidx])
    out = {}
    for k in ks:
        r, frac = rank_in(agg(pred, gidx, P, k, max_chunks=None if k is None else 10), gal, EMB_N_T)
        tag = "all" if k is None else str(k)
        out[f"mrp@{tag}"], out[f"top5@{tag}"], out[f"top1@{tag}"] = float(1 - frac.mean()), float((r < 5).mean()), float((r < 1).mean())
    out["n_gallery"], out["n_queries"] = int(len(gal)), int(len(r))
    return out


def concept_top5(pred, gidx, P, EMB_N_T, concept_of_img, draws=20, seed=0):
    """Codabench-like top-5: galleries of ONE image per concept (150-way for a 150-concept split, as the official
    200-way gallery has one image per concept), averaged over random draws of that image."""
    A = agg(pred, gidx, P)
    imgs = np.unique(P.img[gidx])
    c = concept_of_img[imgs]
    rng = np.random.default_rng(seed)
    vals = []
    for _ in range(draws):
        o = rng.permutation(len(imgs))
        gal = np.sort(imgs[o][np.unique(c[o], return_index=True)[1]])
        keep = np.isin(A["img"], gal)
        r, _ = rank_in(dict(mean=A["mean"][torch.as_tensor(keep)], img=A["img"][keep]), gal, EMB_N_T)
        vals.append(float((r < 5).mean()))
    return float(np.mean(vals)), int(len(gal))


def p_better(a, b, n=1000, seed=0):
    """Paired bootstrap over queries: P(mean of a > mean of b)."""
    a, b = np.asarray(a, float), np.asarray(b, float)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(a), (n, len(a)))
    return float(((a[idx] - b[idx]).mean(1) > 0).mean())


def amp_dtype(device):
    """bfloat16 on GPUs that have it natively (Ampere and newer), float16 on a T4, None on the CPU."""
    if torch.device(device).type != "cuda":
        return None
    return torch.bfloat16 if torch.cuda.get_device_capability(device)[0] >= 8 else torch.float16


# ---------------------------------------------------------------- shared by the packaging scripts (single models and ensembles)
SOLVER_SRC = r'''

from benchmark_utils.base_solver import CompetSolver


class Solver(CompetSolver):
    name = "__SOLVER_NAME__"

    def load_model(self, meta):
        d = Path(meta["submission_dir"])
        cfg = json.loads((d / "config.json").read_text())
        model = build_model(cfg, device="cpu")
        state = torch.load(d / "weights.pt", map_location="cpu", weights_only=True)
        model.load_state_dict({k: (v.float() if v.is_floating_point() else v) for k, v in state.items()}, strict=True)
        n_ok, n_train = model.set_input_channels(meta.get("ch_names"), meta.get("n_chans"))
        print(f"[{self.name}] {n_ok}/{n_train} training electrodes found in the input")
        return model.to(meta.get("device", "cpu")).eval()
'''

REPLAY = r"""
import builtins, json, socket, sys
from pathlib import Path
import numpy as np, torch

def _offline(*a, **k):
    raise RuntimeError("network access attempted during the replay")
socket.socket.connect = _offline
socket.create_connection = _offline
sys.path.insert(0, sys.argv[1])
from benchmark_utils.base_solver import CompetSolver  # noqa: F401
from benchmark_utils.metrics import group_means, topk_accuracy
_imp = builtins.__import__
def _guard(name, *a, **k):
    if name.split(".")[0] in ("braindecode", "huggingface_hub", "transformers", "mne"):
        raise ImportError(f"{name} imported: the submission must not need it")
    return _imp(name, *a, **k)
builtins.__import__ = _guard
import runpy
sub = Path(sys.argv[2])
ns = runpy.run_path(str(sub / "submission.py"))
d = np.load(sys.argv[3])
meta = dict(submission_dir=sub, device=sys.argv[4], ch_names=json.loads(sys.argv[5]), n_chans=int(d["X"].shape[1]),
            n_times=int(d["X"].shape[2]), sfreq=float(sys.argv[6]), n_outputs=int(d["cand"].shape[1]))
solver = ns["Solver"].__new__(ns["Solver"])
model = solver.load_model(meta)
X = torch.from_numpy(d["X"].astype(np.float32))
pred = np.concatenate([model.predict(X[s:s + 64].to(meta["device"])).detach().float().cpu().numpy() for s in range(0, len(X), 64)])
assert pred.shape == (len(X), d["cand"].shape[1]) and np.isfinite(pred).all(), "bad output"
nrm = lambda v: v / np.maximum(np.linalg.norm(v, axis=-1, keepdims=True), 1e-12)
keys, pred_sa = group_means(pred, np.stack([d["subj"], d["tidx"]], axis=1))
sc = nrm(pred_sa) @ nrm(d["cand"]).T
np.save(sys.argv[7], pred)
print("REPLAY_OK", topk_accuracy(sc, keys[:, 1], 5), topk_accuracy(sc, keys[:, 1], 1), len(pred),
      int(any(m.split(".")[0] == "braindecode" for m in sys.modules)))
"""

In [ ]:
%%writefile {SRC}/t1_split.py
"""Team split protocol for Track 1 (3-way, subject- and concept-disjoint): canonical IDs, frozen manifest, checksums,
reading and comparing the team manifest. numpy + pandas only (also imported by the notebook kernel).

Canonical IDs (shared with the team):
  subject_id  dataset participant number, 1..20 (sub-01 -> 1)
  image_id    Alljoined numeric stimulus id = position in THINGS-EEG2: 0..16539 training pool, 16540..16739 official test pool
  concept_id  THINGS-EEG2 training concept number, 1..1654 = image_id // 10 + 1 (the "00001" of "00001_aardvark")
Checksum of one list:  sha256(",".join(str(i) for i in sorted(set(ids))).encode()).hexdigest()
Manifest checksum:     sha256("\\n".join(f"{split}.{kind}=" + <that joined list> for split in (train, val, test)
                                         for kind in (subjects, concepts, images)).encode()).hexdigest()
"""
import hashlib
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

N_SUBJECTS, N_TRAIN_CONCEPTS, IMAGES_PER_CONCEPT, N_TEST_POOL = 20, 1654, 10, 200
N_TRAIN_POOL = N_TRAIN_CONCEPTS * IMAGES_PER_CONCEPT                  # 16,540 images: ids 0..16539
SPLITS3, KINDS = ("train", "val", "test"), ("subjects", "concepts", "images")


def images_of(concepts):
    return sorted(int(i) for c in concepts for i in range((int(c) - 1) * IMAGES_PER_CONCEPT, int(c) * IMAGES_PER_CONCEPT))


def concepts_of(images, n_concepts=N_TRAIN_CONCEPTS):
    return sorted({int(i) // IMAGES_PER_CONCEPT + 1 for i in images if 0 <= int(i) < n_concepts * IMAGES_PER_CONCEPT})


def provisional_manifest(sp):
    """The split from its spec: given subjects; val / test concepts = the first n_val / next n_test concepts of a
    seeded permutation of 1..1654; train = every other concept; images = the 10 images of each concept."""
    n_c = int(sp.get("n_train_concepts", N_TRAIN_CONCEPTS))
    perm = np.random.default_rng(sp["seed"]).permutation(np.arange(1, n_c + 1))
    nv, nt = int(sp["n_val_concepts"]), int(sp["n_test_concepts"])
    con = dict(val=sorted(int(c) for c in perm[:nv]), test=sorted(int(c) for c in perm[nv:nv + nt]))
    con["train"] = sorted(set(range(1, n_c + 1)) - set(con["val"]) - set(con["test"]))
    sub = {k: sorted(int(s) for s in sp[f"{k}_subjects"]) if sp.get(f"{k}_subjects") is not None else None for k in SPLITS3}
    if sub["train"] is None:
        sub["train"] = sorted(set(range(1, N_SUBJECTS + 1)) - set(sub["val"]) - set(sub["test"]))
    return {k: dict(subjects=sub[k], concepts=con[k], images=images_of(con[k])) for k in SPLITS3}


def _joined(ids):
    return ",".join(str(int(i)) for i in sorted(set(int(x) for x in ids)))


def checksums(m):
    out = {f"{s}.{k}": hashlib.sha256(_joined(m[s][k]).encode()).hexdigest() for s in SPLITS3 for k in KINDS}
    out["manifest"] = hashlib.sha256("\n".join(f"{s}.{k}=" + _joined(m[s][k]) for s in SPLITS3 for k in KINDS).encode()).hexdigest()
    return out


def check_manifest(m, n_concepts=N_TRAIN_CONCEPTS, n_subjects=N_SUBJECTS):
    """Problems of a manifest (empty list = consistent with the protocol)."""
    bad = []
    for k in ("subjects", "concepts", "images"):
        for a in SPLITS3:
            for b in SPLITS3:
                if a < b and set(m[a][k]) & set(m[b][k]):
                    bad.append(f"{k} shared by {a} and {b}: {sorted(set(m[a][k]) & set(m[b][k]))[:10]}")
    for s in SPLITS3:
        if set(m[s]["images"]) != set(images_of(m[s]["concepts"])):
            bad.append(f"{s}: images are not exactly the 10 images of its concepts")
        if any(not 1 <= c <= n_concepts for c in m[s]["concepts"]):
            bad.append(f"{s}: concept ids outside 1..{n_concepts}")
        if any(not 1 <= c <= n_subjects for c in m[s]["subjects"]):
            bad.append(f"{s}: subject ids outside 1..{n_subjects}")
    if set().union(*(m[s]["concepts"] for s in SPLITS3)) != set(range(1, n_concepts + 1)):
        bad.append(f"the three splits do not cover the {n_concepts} training concepts exactly")
    unused = sorted(set(range(1, n_subjects + 1)) - set().union(*(m[s]["subjects"] for s in SPLITS3)))
    if unused:
        bad.append(f"subjects in no split: {unused}")
    return bad


def manifest_table(m, names=None):
    """Long table, one row per (split, kind, id): the frozen manifest as CSV."""
    rows = [dict(split=s, kind=k, id=int(i)) for s in SPLITS3 for k in KINDS for i in m[s][k]]
    t = pd.DataFrame(rows)
    if names is not None:
        t["name"] = [names.get(("concepts" if k == "concepts" else "images"), {}).get(i, "") if k != "subjects" else ""
                     for k, i in zip(t["kind"], t["id"])]
    return t


def manifest_json(m, sp, source, names=None, n_concepts=N_TRAIN_CONCEPTS):
    cs = checksums(m)
    pool = n_concepts * IMAGES_PER_CONCEPT
    d = dict(protocol="Track 1 team split: 3-way, subject- and concept-disjoint", source=source, spec=sp,
             definitions=__doc__.split("Canonical IDs (shared with the team):", 1)[1].strip(),
             counts={s: {k: len(m[s][k]) for k in KINDS} for s in SPLITS3}, checksums=cs,
             n_train_concepts=n_concepts,
             official_test_pool=dict(image_ids=f"{pool}..{pool + N_TEST_POOL - 1}", used="never in train / val / test"),
             splits={s: {k: m[s][k] for k in KINDS} for s in SPLITS3})
    if names:
        d["concept_names"] = {s: [names["concepts"].get(c, "") for c in m[s]["concepts"]] for s in ("val", "test")}
    return d


# ---------------------------------------------------------------- reading the team manifest (any reasonable layout)
def _split_of(v):
    t = re.sub(r"[^a-z]", "", str(v).lower())
    if not t or "official" in t or t in ("excluded", "unused", "none", "ignore", "sealed", "codabench", "pool"):
        return None
    if t in ("train", "training", "fit", "trn") or t.startswith("train"):
        return "train"
    if t in ("val", "valid", "validation", "dev", "selection") or t.startswith("val"):
        return "val"
    if "test" in t or t in ("holdout", "heldout", "lt"):
        return "test"
    return None


def _kind_of(v):
    t = str(v).lower()
    if "subj" in t or "partic" in t or t in ("sub", "subject_id", "sid"):
        return "subjects"
    if "concept" in t or "categor" in t or "class" in t:
        return "concepts"
    if "image" in t or "img" in t or "stim" in t or "file" in t or t in ("item",):
        return "images"
    return None


def _ids(values, kind, names=None):
    out = []
    lut = (names or {}).get(kind, {})
    rev = {str(v).lower(): k for k, v in lut.items()} if lut else {}
    for v in values:
        if v is None or (isinstance(v, float) and np.isnan(v)):
            continue
        if isinstance(v, (int, np.integer)) or (isinstance(v, float) and float(v).is_integer()):
            out.append(int(v)); continue
        s = str(v).strip().replace("\\", "/").rsplit("/", 1)[-1]
        stem = s.rsplit(".", 1)[0] if re.fullmatch(r".+\.(jpg|jpeg|png|bmp|webp)", s, flags=re.I) else s
        if re.fullmatch(r"\d+", stem):
            out.append(int(stem))
        elif kind == "subjects" and re.fullmatch(r"sub-?\d+", stem, flags=re.I):
            out.append(int(re.sub(r"\D", "", stem)))
        elif kind == "concepts" and re.match(r"^\d+_", stem):
            out.append(int(stem.split("_", 1)[0]))
        elif stem.lower() in rev:
            out.append(int(rev[stem.lower()]))
        else:
            raise ValueError(f"cannot read the {kind} id {v!r}: numeric ids are expected (see the definitions); "
                             "for THINGS names, attach THINGS-EEG2 image_metadata.npy (THINGS_METADATA)")
    return out


def _from_frame(df, acc, names):
    cols = list(df.columns)
    low = {str(c).lower(): c for c in cols}
    split_col = next((low[c] for c in ("split", "role", "set", "partition", "fold", "subset", "group") if c in low),
                     next((c for c in cols if "split" in str(c).lower()), None))
    if "kind" in low and "id" in low and split_col is not None:          # our own long format
        for (s, k), g in df.groupby([split_col, low["kind"]]):
            if _split_of(s) and _kind_of(k):
                acc[_split_of(s)][_kind_of(k)].update(_ids(g[low["id"]].tolist(), _kind_of(k), names))
        return
    kind_cols = {}
    for c in cols:
        k = _kind_of(c)
        if k and c != split_col and (k not in kind_cols or "id" in str(c).lower()):
            kind_cols[k] = c
    if split_col is None:
        raise ValueError(f"no split column among {cols}")
    for s, g in df.groupby(split_col):
        if _split_of(s):
            for k, c in kind_cols.items():
                acc[_split_of(s)][k].update(_ids(g[c].tolist(), k, names))


def _from_json(obj, acc, names, path=()):
    if isinstance(obj, dict):
        for k, v in obj.items():
            _from_json(v, acc, names, path + (str(k),))
    elif isinstance(obj, list) and obj and all(isinstance(x, dict) for x in obj):
        df = pd.DataFrame(obj)
        s = next((_split_of(p) for p in path if _split_of(p)), None)
        if s and not any(str(c).lower() in ("split", "role", "set", "partition", "fold", "subset") for c in df.columns):
            df["split"] = s
        _from_frame(df, acc, names)
    elif isinstance(obj, list):
        s = next((_split_of(p) for p in reversed(path) if _split_of(p)), None)
        k = next((_kind_of(p) for p in reversed(path) if _kind_of(p)), None)
        if s and k:
            acc[s][k].update(_ids(obj, k, names))


def read_manifest(paths, names=None, n_concepts=N_TRAIN_CONCEPTS, n_subjects=N_SUBJECTS):
    """A team manifest -> {split: {subjects, concepts, images}} in canonical ids. Accepts one path or several
    (CSV / TSV / parquet / JSON / NPY-dict), long or wide; missing concepts or images are derived from the other."""
    acc = {s: {k: set() for k in KINDS} for s in SPLITS3}
    for p in ([paths] if isinstance(paths, (str, Path)) else paths):
        p = Path(p)
        suf = p.suffix.lower()
        if suf == ".json":
            _from_json(json.loads(p.read_text()), acc, names)
        elif suf == ".npy":
            _from_json(np.load(p, allow_pickle=True).item(), acc, names)
        elif suf in (".parquet", ".pq"):
            _from_frame(pd.read_parquet(p), acc, names)
        else:
            _from_frame(pd.read_csv(p, sep=None, engine="python"), acc, names)
    m = {}
    for s in SPLITS3:
        con, img = set(acc[s]["concepts"]), set(acc[s]["images"])
        if img and not con:
            con = set(concepts_of(img, n_concepts))
        if con and not img:
            img = set(images_of(con))
        m[s] = dict(subjects=sorted(acc[s]["subjects"]), concepts=sorted(con), images=sorted(img))
    if not m["train"]["concepts"]:
        m["train"]["concepts"] = sorted(set(range(1, n_concepts + 1)) - set(m["val"]["concepts"]) - set(m["test"]["concepts"]))
        m["train"]["images"] = images_of(m["train"]["concepts"])
    if not m["train"]["subjects"]:
        m["train"]["subjects"] = sorted(set(range(1, n_subjects + 1)) - set(m["val"]["subjects"]) - set(m["test"]["subjects"]))
    return m


def compare_manifests(a, b, names=("ours", "theirs")):
    ca, cb = checksums(a), checksums(b)
    rows = []
    for s in SPLITS3:
        for k in KINDS:
            A, B = set(a[s][k]), set(b[s][k])
            rows.append({"split.kind": f"{s}.{k}", f"n {names[0]}": len(A), f"n {names[1]}": len(B), "identical": A == B,
                         f"only {names[0]}": sorted(A - B)[:12], f"only {names[1]}": sorted(B - A)[:12],
                         f"sha256 {names[0]}": ca[f"{s}.{k}"][:16], f"sha256 {names[1]}": cb[f"{s}.{k}"][:16]})
    rows.append({"split.kind": "MANIFEST", "identical": ca["manifest"] == cb["manifest"],
                 f"sha256 {names[0]}": ca["manifest"][:16], f"sha256 {names[1]}": cb["manifest"][:16]})
    return pd.DataFrame(rows)


def things_names(path):
    """Optional THINGS-EEG2 image_metadata.npy -> {"concepts": {id: name}, "images": {id: file}} (training pool), checked
    against the canonical rule concept_id = image_id // 10 + 1."""
    md = np.load(path, allow_pickle=True).item()
    files, concepts = list(md["train_img_files"]), list(md["train_img_concepts"])
    assert len(files) == N_TRAIN_POOL, f"{len(files)} training images in the metadata, {N_TRAIN_POOL} expected"
    out = dict(concepts={}, images={i: str(f) for i, f in enumerate(files)})
    for i, c in enumerate(concepts):
        num = int(str(c).split("_", 1)[0])
        assert num == i // IMAGES_PER_CONCEPT + 1, f"image {i}: concept {c} breaks the rule concept = image // 10 + 1"
        out["concepts"][num] = str(c).split("_", 1)[1] if "_" in str(c) else str(c)
    return out


# ---------------------------------------------------------------- the team manifest format (protocol ..._v1, seed 33)
TEAM_PARTS = {"train": "train", "val": "validation", "test": "local_test"}


def team_digest(xs):
    """The team list digest: sha256 of the sorted strings joined by newlines."""
    return hashlib.sha256("\n".join(sorted(xs)).encode("utf-8")).hexdigest()


def team_science_sha(d):
    """The team manifest checksum (as the team loader recomputes it)."""
    obj = {k: v for k, v in d.items() if k not in ("created_utc", "science_sha256")}
    return hashlib.sha256(json.dumps(obj, sort_keys=True, ensure_ascii=False, separators=(",", ":")).encode()).hexdigest()


def _things_rank(names):
    """THINGS-EEG2 training concepts are the 1,654 training-pool THINGS concepts in THINGS order: the k-th smallest THINGS
    global id ('00002_abacus' -> 2) is training concept k, whose images are (k-1)*10 .. k*10-1."""
    return {n: k + 1 for k, n in enumerate(sorted(set(names), key=lambda s: int(str(s).split("_", 1)[0])))}


def compact_team_manifest(d):
    """The manifest without its image lists (they follow from the concepts): small enough to embed in the notebook."""
    c = json.loads(json.dumps(d))
    for p in c["parts"].values():
        p.pop("image_ids", None); p.pop("candidate_gallery_image_ids", None)
    c["compact_note"] = "image_ids / candidate_gallery_image_ids removed; rebuilt by expand_team_manifest"
    return c


def expand_team_manifest(c):
    """Rebuilds the full manifest from its compact form (byte-identical content: its science_sha256 must verify)."""
    d = json.loads(json.dumps(c)); d.pop("compact_note", None)
    rank = _things_rank(n for p in d["parts"].values() for n in p["concepts"])
    for k, p in d["parts"].items():
        ids = sorted(f"{i:05d}.jpg" for n in p["concepts"] for i in range((rank[n] - 1) * IMAGES_PER_CONCEPT, rank[n] * IMAGES_PER_CONCEPT))
        p["image_ids"] = ids
        p["candidate_gallery_image_ids"] = ids if k != "train" else None
    return d


def from_team_manifest(d):
    """Teammate manifest -> (canonical manifest, concept names, n training concepts), with every check his loader makes
    plus the concept-order rule: THINGS concept names and image ids must give the same canonical concepts."""
    if team_science_sha(d) != d.get("science_sha256"):
        raise RuntimeError("team manifest: science_sha256 does not verify (file edited or truncated)")
    parts = d["parts"]
    if set(parts) != set(TEAM_PARTS.values()):
        raise RuntimeError(f"team manifest has parts {sorted(parts)}: the 3-way manifest (train / validation / local_test) is needed")
    rank = _things_rank(n for p in parts.values() for n in p["concepts"])
    m = {}
    for s, k in TEAM_PARTS.items():
        p = parts[k]
        if team_digest(p["image_ids"]) != p["image_id_sha256"] or team_digest(p["concepts"]) != p["concept_sha256"]:
            raise RuntimeError(f"team manifest, part {k}: a list does not match its sha256")
        imgs = sorted(int(str(x).rsplit(".", 1)[0]) for x in p["image_ids"])
        con = sorted(rank[n] for n in p["concepts"])
        if con != concepts_of(imgs, len(rank)) or images_of(con) != imgs:
            raise RuntimeError(f"team manifest, part {k}: THINGS concept names and image ids disagree with the rule "
                               "concept = image // 10 + 1")
        m[s] = dict(subjects=sorted(int(re.sub(r"\D", "", str(x))) for x in p["subjects"]), concepts=con, images=imgs)
    names = {"concepts": {v: k for k, v in rank.items()}}
    return m, names, len(rank)

In [ ]:
%%writefile {SRC}/threeway_manifest_seed33.compact.json
{"protocol":"xu2025alljoined_threeway_subject_concept_disjoint_v1","seed":33,"created_utc":"2026-11-18T14:58:47.736895+00:00","source":"Xu2025Alljoined public NEMAR nm000134 stim_train event metadata; THINGS-EEG2 metadata","source_event_files":1525,"parts":{"train":{"subjects":["sub-01","sub-02","sub-03","sub-04","sub-05","sub-07","sub-10","sub-12","sub-13","sub-14","sub-15","sub-17","sub-18","sub-20"],"stimulus_label":"stim_train","concepts":["00001_aardvark","00003_accordion","00004_acorn","00005_air_conditioner","00006_air_mattress","00007_air_pump","00008_airbag","00011_airplane","00012_album","00013_alligator","00014_almond","00015_aloe","00016_alpaca","00017_altar","00018_aluminum_foil","00019_amber","00020_ambulance","00021_amplifier","00022_anchor","00023_ankle","00024_anklet","00025_ant","00028_antenna","00029_anvil","00030_appetizer","00031_apple","00032_apple_tree","00033_applesauce","00034_apron","00035_aquarium","00036_arch","00037_arm","00038_armor","00039_arrow","00040_artichoke","00041_arugula","00042_ashtray","00043_asparagus","00044_avocado","00045_awning","00046_axe","00047_baby","00048_backdrop","00049_backgammon","00050_backpack","00054_badger","00055_bag","00056_bagel","00057_bagpipe","00058_baklava","00060_ball","00061_balloon","00062_ballot_box","00063_bamboo","00065_banana_peel","00067_bandage","00068_bandanna","00069_banjo","00071_banner","00072_barbed_wire","00073_barbell","00074_barcode","00075_bark","00076_barnacle","00077_barrel","00078_barrette","00079_baseball","00081_baseball_glove","00083_basket","00086_bassinet","00088_bat1","00089_bat2","00090_bathmat","00093_baton1","00094_baton2","00095_baton3","00098_battery","00100_beachball","00101_bead","00103_bean","00104_beanbag","00105_beanie","00106_bear","00107_beard","00109_bed","00110_bedpan","00111_bedpost","00113_beehive","00114_beer","00115_beet","00116_beetle","00117_bell","00119_belt","00120_belt_buckle","00122_berry","00123_bib","00125_bikini","00126_bin","00127_binder","00128_binoculars","00130_birdbath","00131_birdcage","00132_birdhouse","00134_biscuit","00135_bison","00136_blackberry","00137_blanket","00139_blender","00140_blimp","00141_blind","00142_blinder","00143_blindfold","00144_block","00145_blouse","00146_blower","00147_blowfish","00148_blowgun","00150_blueberry","00151_boa","00152_boar","00153_board","00154_board_game","00156_bobsled","00159_bologna","00160_bolt","00162_bone","00163_bongo","00165_bonsai","00167_bookmark","00168_bookshelf","00169_boomerang","00170_boot","00171_bottle","00174_bouquet","00175_bow1","00178_bowl","00179_bowler_hat","00181_bowtie","00182_box","00183_boxer_shorts","00184_boxing_gloves","00185_boy","00186_bra","00188_bracelet1","00189_bracelet2","00190_bracket","00191_braid","00192_brake","00194_brass_knuckles","00197_breadstick","00198_breakfast","00199_breathalyzer","00201_briefcase","00202_broccoli","00204_broom","00205_brownie","00206_brush","00207_brussels_sprouts","00208_bubble","00209_bubble_wrap","00210_bucket","00211_buckle","00212_buffet","00215_bull","00216_bulldozer","00219_bulletproof_vest","00220_bumper","00222_bungee","00223_bunkbed","00225_burner","00226_burrito","00229_butter","00230_butterfly","00233_cabbage","00234_cabinet","00235_cable","00236_cactus","00237_cage","00238_cake","00239_cake_mix","00241_calculator","00242_calf1","00243_calf2","00244_calzone","00245_camcorder","00247_camera1","00248_camera2","00249_camera_lens","00250_camper","00251_can","00252_can_opener","00253_candelabra","00254_candle","00256_candy","00257_candy_bar","00258_candy_cane","00259_cane","00260_canister","00261_cannon","00263_canoe","00264_cantaloupe","00266_cap","00267_cape","00268_car","00273_cardboard","00274_cardigan","00275_cardinal","00276_carousel","00277_carriage","00278_carrot","00280_cash_machine","00281_cash_register","00283_casserole","00284_cassette","00293_cello","00294_cellphone","00295_cement_mixer","00296_centerpiece","00297_centrifuge","00298_cereal","00300_chainsaw","00301_chair","00302_chalice","00303_chalk","00304_chalkboard","00305_champagne","00308_charcoal","00309_charger","00310_chariot","00311_checkbook","00312_checkers","00314_cheeseburger","00315_cheesecake","00317_cherry","00318_chess_piece","00319_chessboard","00320_chest1","00323_chicken1","00324_chicken2","00325_chicken_wire","00326_chickpea","00327_chihuahua","00328_chili","00330_chimney","00332_chinaware","00333_chinchilla","00334_chip","00335_chipmunk","00336_chips","00337_chisel","00338_chive","00339_chocolate","00341_christmas_card","00342_christmas_tree","00343_chute","00344_cigar","00345_cigarette","00346_cigarette_butt","00349_cinnamon","00351_clarinet","00352_clasp","00353_clay","00356_clipboard","00358_clipper2","00359_cloak","00360_clock","00362_clothes","00363_clothesline","00364_clothespin","00366_clove","00367_clover","00368_coal","00369_coaster","00373_cockatoo","00375_cocktail","00380_coffee_filter","00381_coffee_pot","00382_coffee_table","00385_coil","00386_coin","00387_coleslaw","00388_collar","00390_comb","00391_combination_lock","00392_comic_book","00395_computer","00396_computer_screen","00397_confetti","00398_contact_lens","00399_container","00400_cooker","00402_cookie_cutter","00403_cookie_sheet","00404_cooler","00406_copier","00407_coral","00408_cord","00410_cork","00411_corkboard","00412_corkscrew","00413_corn","00414_cornbread","00415_cornhusk","00416_cornmeal","00417_cornucopia","00418_corsage","00420_costume","00421_cot","00422_cotton_candy","00423_couch","00424_cougar","00425_counter","00427_cow","00428_coyote","00430_cracker","00431_cranberry","00432_crane","00435_crayfish","00436_crayon","00437_cream","00438_cream_cheese","00444_cross","00445_crossbow","00446_crouton","00448_crowbar","00449_crown","00450_crucifix","00454_crystal1","00455_crystal2","00456_crystal_ball","00457_cuckoo_clock","00459_cufflink","00460_cummerbund","00461_cup","00464_curling_iron","00465_curry","00467_cushion","00468_cutting_board","00469_cymbal","00471_daisy","00473_dandelion","00474_dart","00475_dartboard","00476_dashboard","00477_deer","00479_denture","00481_desk","00483_detonator","00484_dial","00485_diamond","00486_diaper","00488_dip","00490_dish","00491_dishrag","00493_dishwashing_liquid","00494_diskette","00495_diving_board","00496_dog","00497_dogfood","00498_doghouse","00499_doily","00500_doll","00501_dollhouse","00502_dolly","00503_dolphin","00504_domino","00506_donut","00507_door","00510_doorknob","00512_doormat","00513_doorstop","00516_drain","00517_drawer","00519_dress","00520_dresser","00522_drink","00524_drumstick","00525_dryer","00526_duck","00527_duckling","00528_duct","00529_duct_tape","00531_dumbbell","00532_dumbwaiter","00533_dumpling","00534_dumpster","00535_duster","00536_dustpan","00539_ear","00540_earplug","00541_earring","00542_earwig","00543_easel","00544_easter_egg","00545_eclair","00548_egg_roll","00549_eggbeater","00551_eggplant","00552_eggshell","00553_elbow","00554_electric_chair","00556_emerald","00557_enchilada","00558_engine","00559_envelope","00562_exerciser","00563_exhaust_pipe","00564_extinguisher","00565_eye","00567_eyedropper","00568_eyeliner","00569_eyepiece","00570_face","00572_fan","00575_feather","00578_fern","00579_ferret","00580_ferris_wheel","00582_fig","00583_figurine","00584_file1","00586_filing_cabinet","00587_film","00588_filter","00589_finger","00590_fingerprint","00591_fire","00592_fire_alarm","00593_fire_hydrant","00594_fire_pit","00595_firecracker","00597_firetruck","00598_firewood","00599_fireworks","00600_first-aid_kit","00601_fish","00602_fishbowl","00603_fishhook","00604_fishing_pole","00605_fishnet_stockings","00606_flag","00610_flan","00612_flashlight","00613_flask","00614_flatiron","00615_flip_flop","00616_flipper","00617_float","00618_floss","00619_flour","00620_flower","00621_flute","00622_fly","00624_flyswatter","00627_fondue","00629_foot","00630_football","00631_football_helmet","00632_footbath","00633_footprint","00634_footrest","00636_forklift","00637_fossil","00638_fountain_pen","00639_fox","00640_frame","00642_french_fries","00644_frisbee","00645_frog","00647_fruitcake","00648_fudge","00649_fungus","00651_fur_coat","00652_furnace","00653_fuse","00654_gallows","00655_game","00656_garbage","00661_gas_mask","00662_gasket","00663_gate","00664_gauge","00666_gavel","00667_gazelle","00668_gear","00669_gearshift","00670_gel","00671_gem","00672_generator","00673_gift","00674_ginger","00675_gingerbread_man","00676_giraffe","00677_girl","00678_glass","00679_glasses","00682_glue","00683_go-kart","00684_goalpost","00685_goat","00686_goblet","00687_goggles","00688_gold","00689_goldfish","00691_golf_club","00698_graffiti","00699_grain","00700_gramophone","00701_granite","00702_granola","00703_grape","00704_grapefruit","00705_grapevine","00708_grate","00709_grater","00710_gravel","00711_gravestone","00712_gravy","00713_green_beans","00715_grill","00716_grille","00718_grits","00719_groundhog","00721_guardrail","00722_guillotine","00724_guitar","00725_gum","00726_gumball","00727_gumdrop","00728_gun","00729_gurney","00730_gutter","00731_gyro","00733_hail","00734_hair","00735_hairbrush","00736_hairdryer","00737_hairnet","00738_hairpin","00740_ham","00743_hammock","00744_hamster","00745_hand","00747_handcuff","00750_handlebar","00752_hanger","00754_harmonica","00755_harness","00757_hash","00759_hatbox","00760_hatchet","00761_hawk","00762_hay","00763_headband","00764_headdress","00765_headlamp","00766_headlight","00767_headphones","00768_headrest","00770_headset","00771_hearing_aid","00772_hearse","00773_heater","00774_hedge","00775_hedgehog","00776_helicopter","00777_helmet","00780_hinge","00781_hip","00782_hippopotamus","00783_hobbyhorse","00784_hockey_stick","00786_hole","00787_holster","00788_home_plate","00789_honey","00791_honeypot","00792_hood","00794_hook1","00795_hook2","00796_hookah","00797_hopscotch","00799_horse","00800_horseshoe","00801_hose","00802_hot-air_balloon","00803_hot-water_bottle","00804_hot_chocolate","00807_hotplate","00809_hovercraft","00810_hubcap","00811_hula_hoop","00813_hummus","00814_humvee","00815_hydrant","00816_hyena","00817_ice","00818_ice-cream_cone","00819_ice_cream","00822_icemaker","00823_icepick","00824_iceskate","00825_icicle","00827_incense","00828_incubator","00829_inhaler","00830_ink","00832_insole","00833_iron","00835_ivy","00836_jack","00838_jackhammer","00839_jalapeno","00840_jam","00841_jar","00842_javelin","00843_jeans","00846_jellyfish","00847_jersey","00848_jet","00849_jetski","00850_jewel","00851_jewelry","00852_jigsaw_puzzle","00853_joystick","00854_jug","00855_juice","00856_juicer1","00857_juicer2","00859_jump_rope","00860_jumpsuit","00861_kale","00862_kaleidoscope","00863_kangaroo","00864_kayak","00865_kazoo","00866_kebab","00867_keg","00868_ketchup","00870_key","00871_keyboard","00873_kilt","00874_kimono","00875_kite","00878_knee","00881_knitting","00882_knitting_needle","00883_knob","00884_knot","00885_koala","00886_lab_coat","00887_ladder","00889_ladybug","00891_lamb_chop","00892_lamp","00893_lamppost","00895_landmine","00896_lantern","00897_lanyard","00899_lasagna","00900_laser_pointer","00901_latch","00902_latte","00904_lava","00905_lavender","00907_leaf","00908_leash","00911_leek","00912_leg","00913_leggings","00914_lego","00915_lemon","00916_lemonade","00918_leopard","00919_leotard","00920_letter_opener","00922_license_plate","00923_licorice","00924_lid","00926_lifesaver","00927_light_switch","00928_lightbulb","00929_lighter","00931_lime","00934_lion","00935_lip_balm","00936_lip_gloss","00937_lipstick","00938_lizard","00939_llama","00940_lobster","00941_lock","00942_locker","00943_locket","00944_log","00947_loom","00948_loveseat","00949_luggage","00950_lumber","00951_lunchbox","00952_macadamia","00953_macaroni","00954_machete","00955_machine_gun","00956_maggot","00957_magnet","00958_magnifier","00959_magnifying_glass","00960_mail","00962_makeup","00963_mallet","00964_man","00966_mandolin","00967_mango","00969_mannequin","00970_mantle","00971_map","00972_maple_syrup","00973_marble","00974_margarita","00976_marker","00977_marmalade","00979_mascara","00980_mashed_potato","00981_mask","00982_mast","00983_mat","00984_match","00985_matchbox","00986_mattress","00987_measuring_cup","00988_meat","00989_meat_grinder","00990_meatball","00993_meerkat","00994_megaphone","00995_melon","00996_memory_stick","00998_metronome","00999_microphone","01000_microscope","01002_milk","01005_mint","01006_mirror","01007_missile","01008_mistletoe","01009_mitten","01010_mixer","01013_mold1","01014_mold2","01015_mole","01016_money","01017_mongoose","01018_monkey","01019_moose","01022_mosquito_net","01023_moss","01024_moth","01025_motherboard","01026_motorcycle","01027_mouse1","01029_mousepad","01030_mousetrap","01031_mousse","01032_mouth","01033_mouthpiece","01034_mud","01036_muffin","01038_mulberry","01039_mulch","01040_mullet","01041_mushroom","01045_mustard","01046_nacho","01047_nail","01048_nail_clippers","01049_nail_file","01050_nail_polish","01053_navel","01055_necklace","01056_needle","01057_nest","01058_net","01061_noisemaker","01062_noodle","01064_nose","01065_notebook","01066_notepad","01067_nut","01068_nutcracker","01069_oar","01070_oatmeal","01071_octopus","01072_odometer","01073_oil","01074_oilcan","01076_olive","01080_orange_rind","01081_orangutan","01083_organ","01084_origami","01087_ottoman","01088_outfit","01089_outlet","01090_oven","01091_overalls","01093_oyster","01094_pacifier","01095_paddle","01096_padlock","01097_paint","01098_paintbrush","01099_painting","01101_palette","01102_pallet","01103_palm_tree","01104_pan","01105_pancake","01106_panda","01108_panties","01109_pants","01110_pantsuit","01111_pantyhose","01112_papaya","01113_paper","01114_paper_bag","01115_paper_plate","01116_paper_towel","01119_parachute","01120_parfait","01121_parking_meter","01122_parrot","01124_pasta","01125_pastry","01126_patch","01127_patty","01128_payphone","01129_pea","01130_peach","01131_peacock","01132_peanut","01135_pearl","01136_pecan","01138_pedometer","01139_peeler","01140_peg","01143_pencil","01145_pendulum","01146_penguin","01147_penholder","01148_penlight","01149_pennant","01152_pepper_mill","01153_peppermint","01154_pepperoni","01156_periscope","01157_pesto","01158_pet_food","01159_petal","01160_petri_dish","01162_phone","01164_photo_booth","01165_photograph","01166_piano","01168_pickle","01170_piecrust","01171_pig","01173_piggy_bank","01175_pill","01176_pillbox","01177_pillow","01178_pin","01179_pinball","01180_pincushion","01182_pine_tree","01183_pineapple","01184_pinecone","01185_ping-pong_table","01186_pinwheel","01187_pipe1","01188_pipe2","01189_pistachio","01190_pita","01191_pitcher","01192_pitchfork","01193_pizza","01195_plant","01196_plaster_cast","01197_plastic_film","01198_plate","01199_platypus","01201_playpen","01202_pliers","01203_plug","01204_plum","01205_plunger","01207_pocket_watch","01209_pogo_stick","01210_poinsettia","01211_poker","01212_polar_bear","01218_polygraph","01219_pom-pom","01220_pomegranate","01221_pony","01222_poodle","01223_pool_table","01225_poppy","01227_porcupine","01228_porthole","01230_poster","01231_pot","01232_potato","01233_potholder","01234_pothole","01235_potpie","01237_powder","01238_power_line","01239_praying_mantis","01241_printer","01242_prism","01243_projector","01244_propeller","01245_prune","01248_puddle","01249_puffin","01251_pulley","01253_pump","01255_punch1","01257_punching_bag","01259_puppy","01261_quad","01263_quiche","01264_quill","01266_rabbit","01267_raccoon","01268_racehorse","01269_rack1","01270_rack2","01271_racket","01272_radar","01273_radiator","01274_radio","01276_raft","01278_railing","01279_rain_gauge","01280_raincoat","01282_rake","01283_ram","01284_ramp","01286_rat","01287_ratchet","01288_rattle","01290_ravioli","01291_razor","01292_razor_blade","01293_ready_meal","01294_rearview_mirror","01295_recliner","01296_record","01297_record_player","01299_red_carpet","01302_reindeer","01303_remote_control","01304_retainer","01306_revolving_door","01308_rhubarb","01309_ribbon","01310_rice","01311_rickshaw","01312_rifle","01313_rim","01314_ring","01315_riser","01316_road_sign","01317_roadsweeper","01320_rock","01321_rocket","01322_rocking_chair","01323_rocking_horse","01324_roll","01325_roller","01326_roller_coaster","01327_rollerblade","01328_rollerskate","01329_rolling_pin","01332_root","01333_rope","01334_rosary","01335_rose","01336_roulette_wheel","01337_router","01338_rubber_band","01339_ruby","01340_rudder","01343_rust","01344_saddle","01345_safety_pin","01346_saffron","01347_sail","01350_salami","01351_saltshaker","01352_sand","01359_sarcophagus","01361_satellite","01362_satellite_dish","01363_sauce","01364_saucer","01365_sauerkraut","01368_sawhorse","01369_saxophone","01370_scaffold","01371_scaffolding","01372_scale","01375_scalpel","01377_scarecrow","01378_scarf","01380_scissors","01381_scone","01382_scoop","01384_scoreboard","01385_scorpion","01386_scrabble","01387_scrambled_egg","01390_screen2","01391_screw","01392_screwdriver","01393_scuba","01394_sea_urchin","01395_seafood","01397_seahorse","01399_seaplane","01400_seatbelt","01403_seesaw","01404_seismograph","01405_sequin","01406_sewage","01407_sewing_kit","01408_sewing_machine","01409_shaker","01410_shark","01411_shaving_cream","01412_shawl","01413_shears","01414_sheath","01415_sheep","01416_sheet","01417_shelf","01418_shell1","01419_shell2","01420_shell3","01421_shield","01422_ship","01423_shirt","01426_shoehorn","01427_shoelace","01428_shopping_basket","01429_shopping_cart","01430_shortbread","01431_shorts","01432_shoulder","01434_shower","01436_shower_curtain","01437_showerhead","01438_shredder","01439_shrimp","01440_shuffleboard","01441_shutter","01443_sidecar","01444_sifter","01445_silicone","01446_silverware","01447_sim_card","01449_siren","01453_ski","01454_ski_boots","01456_ski_pole","01457_skin","01458_skirt","01462_sledgehammer","01464_slicer","01466_slime","01467_sling","01469_slipper","01472_sloth","01473_slug","01474_smoke_alarm","01476_snack","01477_snail","01478_snake","01479_snorkel","01480_snow","01481_snowball","01482_snowboard","01483_snowman","01484_snowmobile","01487_snowsuit","01488_soap","01490_sock","01493_sofa_bed","01494_solar_panel","01496_sombrero","01497_sonogram","01498_sorbet","01499_souffle","01500_soup","01501_soy_sauce","01502_space_shuttle","01503_spacesuit","01505_spam","01506_spareribs","01507_spark_plug","01510_speaker","01512_speedboat","01513_speedometer","01514_spider","01515_spider_web","01516_spinach","01518_sponge","01519_spool","01521_spout","01523_springboard","01524_sprinkler","01525_sprouts","01526_spur","01528_squeegee","01529_squid","01531_squirt_gun","01533_stair","01534_stake","01535_stalagmite","01536_stamp1","01537_stamp2","01538_staple","01539_staple_gun","01540_stapler","01546_steamroller","01547_steering_wheel","01548_stem","01549_step_stool","01550_stereo","01552_stew","01554_sticker","01555_stiletto","01556_stilt","01558_stir_fry","01560_stockings","01561_stomach","01562_stool","01563_stopwatch","01564_stove1","01565_stove2","01566_straightjacket","01567_strainer","01569_straw1","01570_straw2","01572_streetlight","01573_stretcher","01574_string_cheese","01575_stroller","01576_stuffing","01577_stump","01579_subway","01580_sugar_cube","01582_suitcase","01583_sundae","01584_sundial","01585_sunflower","01586_sunglasses","01587_sunroof","01588_surfboard","01589_sushi","01590_suspenders","01591_swab","01593_sweater","01594_sweatsuit","01598_swimsuit","01599_swing","01600_swing_set","01601_switch","01603_sword","01605_syringe","01606_syrup","01610_tablecloth","01611_tablet","01612_tack","01615_tadpole","01616_taffy","01617_tag","01619_tamale","01622_tank2","01623_tape","01624_tape_measure","01626_tapestry","01627_tarantula","01628_target","01629_tarp","01630_tattoo","01632_tea","01635_teapot","01643_tennis_ball","01644_tent","01645_terrarium","01647_thermometer","01649_thermostat","01650_thimble","01651_thorn","01652_thread","01653_throne","01654_thumb","01655_thumbtack","01659_tie","01661_tile","01663_tinsel","01664_tiramisu","01666_toast","01667_toaster","01668_toaster_oven","01669_toe","01670_toga","01671_toilet","01672_toilet_paper","01673_tomato","01676_tongue","01679_tooth","01680_toothbrush","01681_toothpaste","01682_toothpick","01686_torso","01687_tortellini","01688_tortilla","01690_totem_pole","01691_toucan","01692_touchpad","01693_towel","01694_towel_rack","01695_toy","01696_tractor","01697_traffic_light","01698_trailer","01699_train","01700_train_car","01701_train_set","01702_trampoline","01703_trap","01706_tray","01708_treasure","01709_tree","01710_tree_trunk","01712_tricycle","01714_trigger","01715_tripod","01716_trolley","01717_trombone","01719_trough","01720_trowel","01721_truck","01722_trumpet","01723_trunk","01724_tuba","01727_tulip","01729_tuning_fork","01730_tupperware","01731_turban","01732_turbine","01733_turf","01735_turnstile","01736_turntable","01737_turtle","01738_turtleneck","01739_tuxedo","01740_tweezers","01741_twig","01744_umbrella","01745_undershirt","01746_underwear","01748_uniform","01749_urinal","01750_urn","01751_vacuum","01752_valve","01753_van","01755_vegetable","01756_veil","01757_velcro","01758_vending_machine","01759_vent","01760_vest","01761_vial","01762_videocassette","01763_videogame","01764_viewfinder","01765_violin","01767_visor","01770_wafer","01771_waffle","01773_wagon","01774_walker1","01775_walker2","01776_wall","01777_wallet","01780_walrus","01781_wand","01782_warthog","01783_washboard","01784_washcloth","01785_washing_machine","01787_watch","01788_water_bottle","01789_water_cooler","01790_water_filter","01791_water_fountain","01794_watermelon","01797_wax_paper","01798_weasel","01799_weather_vane","01801_wedding_cake","01802_wedge","01803_weed","01804_wetsuit","01805_whale","01807_wheel","01808_wheelbarrow","01810_whip","01811_whipped_cream","01812_whisk","01813_whistle","01814_whiteboard","01816_wick","01817_wig","01818_wind_chimes","01820_windowsill","01822_windshield_wiper","01823_windsock","01829_wire","01832_wolf","01833_woman","01834_wood","01835_wooden_leg","01838_wrap","01839_wrapping_paper","01840_wreath","01842_wrench","01843_wrist","01844_xylophone","01845_yacht","01846_yak","01847_yarn","01848_yo-yo","01849_yogurt","01850_yoke","01851_yolk","01852_zebra","01853_zipper"],"n_subjects":14,"n_concepts":1354,"n_images":13540,"concept_sha256":"a6fdd172f0d4a89959545b17e554c03f8bfb10e663385c6b46a4f5a167d09f74","image_id_sha256":"15fd8dd1df0a5527fc8f599fac69f1b843d9213fe642730334167d9417b53979","chance_top5":null},"validation":{"subjects":["sub-06","sub-08","sub-19"],"stimulus_label":"stim_train","concepts":["00009_airboat","00026_anteater","00053_badge","00066_banana_split","00070_bank","00085_basketball_hoop","00092_bathtub","00102_beaker","00112_bee","00129_bird","00138_blazer","00158_bolo_tie","00161_bomb","00177_bow3","00193_branch","00200_brick","00203_brooch","00218_bulletin_board","00224_buoy","00227_bus","00232_button2","00246_camel","00286_catapult","00290_caviar","00306_chandelier","00348_cilantro","00357_clipper1","00361_closet","00371_coat_rack","00374_cockroach","00377_cocoon","00378_coffee","00384_coffin","00394_compost","00405_coop","00439_credit_card","00458_cucumber","00463_curb","00478_defibrillator","00489_dirt_bike","00508_doorbell","00509_doorhandle","00511_doorknocker","00514_dough","00573_fast_food","00574_faucet","00576_fence","00585_file2","00596_fireplace","00608_flamethrower","00623_flypaper","00625_foam","00628_food_processor","00650_funnel","00660_garter","00665_gauze","00680_globe","00717_grinder","00720_guacamole","00732_gyroscope","00748_handkerchief","00753_hard_disk","00756_harp","00798_horn","00806_hotdog","00808_hourglass","00826_iguana","00831_inkwell","00837_jacket","00876_kitten","00877_kiwi","00880_knife","00909_lectern","00917_lens","00932_limousine","00933_lingerie","00946_lollipop","00961_mailbox","01001_microwave","01020_mop","01037_mug","01044_mustache","01051_napkin","01054_neck","01092_owl","01117_paperclip","01123_parsley","01133_peanut_butter","01137_pedal","01142_pen","01144_pencil_sharpener","01155_perfume","01163_phone_booth","01194_place_mat","01213_polaroid","01215_police_car","01217_polo_shirt","01236_potpourri","01246_puck","01247_pudding","01252_pulpit","01265_quilt","01277_rag","01281_raisin","01300_reel","01301_refrigerator","01318_robe","01349_salad","01360_sardine","01367_saw","01379_school_bus","01398_seal","01424_shoe","01433_shovel","01442_sickle","01448_sink","01459_skull","01460_skunk","01470_slot","01471_slot_machine","01504_spaghetti","01508_sparkler","01527_squash","01530_squirrel","01541_star_fruit","01542_starfish","01559_stirrup","01592_swan","01595_sweeper","01604_swordfish","01631_taxi","01633_teabag","01634_teacup","01636_teddy_bear","01638_teepee","01641_telescope","01646_test_tube","01665_toad","01678_toolbox","01684_torch","01689_tostada","01705_trashcan","01711_triangle","01754_vase","01795_waterwheel","01796_wax","01800_webcam","01815_whoopee_cushion","01827_wineglass","01828_wing"],"n_subjects":3,"n_concepts":150,"n_images":1500,"concept_sha256":"8439712026145aad9b57f7947cb9ad188bdf3129e96bf7afc09fb0673cab1e15","image_id_sha256":"53a06b8d554d4275555ead80bbbe009fdb4f265e1e564b3eb51b8ec8c02d4765","chance_top5":0.0033333333333333335},"local_test":{"subjects":["sub-09","sub-11","sub-16"],"stimulus_label":"stim_train","concepts":["00002_abacus","00052_bacon","00091_bathrobe","00099_bazooka","00118_bell_pepper","00166_book","00173_boulder","00176_bow2","00180_bowling_ball","00231_button1","00262_cannonball","00265_canvas","00269_car_door","00270_car_seat","00271_caramel","00272_card","00288_catfish","00289_cauliflower","00292_celery","00322_chick","00331_chin","00347_cigarette_holder","00350_clam","00365_cloud","00389_column","00393_compass","00419_corset","00433_crank","00434_crate","00453_crutch","00466_curtain","00480_deodorant","00487_dice","00492_dishwasher","00505_donkey","00521_drill","00537_dynamite","00550_eggnog","00560_eraser","00566_eye_patch","00577_fencepost","00607_flagpole","00611_flashbulb","00657_garbage_truck","00658_gargoyle","00693_gong","00697_gourd","00706_grass","00723_guinea_pig","00739_hairspray","00749_handle","00751_handprint","00758_hat","00779_highlighter","00785_hoe","00790_honeycomb","00805_hot_tub","00834_ironing_board","00872_keyhole","00898_laptop","00906_lawnmower","00910_leech","00925_life_jacket","00945_loincloth","00968_manhole","00978_marshmallow","00992_medal","01003_milkshake","01011_moccasin","01028_mouse2","01052_napkin_ring","01059_nightshirt","01063_noose","01086_otter","01141_pelican","01151_pepper2","01181_pine_needle","01200_playing_card","01214_pole","01216_polisher","01254_pumpkin","01258_puppet","01262_quesadilla","01289_rattlesnake","01305_revolver","01330_roof_rack","01342_ruler","01354_sandbag","01355_sandbox","01356_sandcastle","01358_sandwich","01376_scanner","01388_scraper","01389_screen1","01425_shoe_polish","01435_shower_cap","01451_skeleton","01452_skewer","01455_ski_lift","01475_smoothie","01485_snowplow","01489_soccer_ball","01491_soda","01492_soda_fountain","01495_soldering_iron","01511_spear","01517_splinter","01522_spring_roll","01532_stained_glass","01544_statue","01545_steak","01553_stick","01557_stingray","01568_strap","01596_sweet_potato","01597_swimming_pool","01602_swizzle_stick","01608_tab","01613_tackle","01614_taco","01620_tambourine","01621_tank1","01637_tee","01639_telegraph","01640_telephone_pole","01648_thermos","01658_ticktacktoe","01660_tiger","01662_timer","01685_torpedo","01704_trapdoor","01713_trident","01718_trophy","01726_tugboat","01728_tumbleweed","01742_typewriter","01743_ukulele","01769_vulture","01772_waffle_iron","01786_wasp","01792_water_heater","01793_watering_can","01819_window","01825_wine_bottle","01826_wine_cooler","01830_wire_cutters","01836_workbench","01837_worm","01841_wreck","01854_zucchini"],"n_subjects":3,"n_concepts":150,"n_images":1500,"concept_sha256":"555f7a779fa908e558ea5b157e4977d3142dbeab27f6ce38bea8be576fb413ea","image_id_sha256":"9e75798fb38ce60310b91a781b0e7c2e3c43944ba6a612007af6275fce49e616","chance_top5":0.0033333333333333335}},"official_test_stimulus_pool":{"image_ids_sha256":"7ad24b2be73a486ca55f9d1a9a56e7430fe96410f58d7295013e083e8de00b45","n_images":200,"n_concepts":200,"usage":"EXCLUDED from ALL local training/validation/local-test partitions; no test EEG read"},"constraints":{"train_val_test_participant_disjoint":true,"train_val_test_concept_disjoint":true,"image_identity_disjoint":true,"all_subject_stim_train_metadata_coverage_checked":true,"no_official_test_eeg_read":true,"metadata_only_not_enforced_loader":true},"evaluation_policy":{"validation":"Use ONLY for selection, tuning and ensemble weights","local_test":"Open ONCE after locking architecture/checkpoint/ensemble; no more tuning","official_test":"Only external final submission, not a local validation set"},"limitations":["Manifest only: train/val original NeuralBench segments must be filtered with a separate validated runner","BIDS stim_train event presence does not prove a prepared EEG epoch is successfully extracted","Old frozen loader has a hardcoded science SHA and cannot accept this manifest directly","Subject+concept holdout does not guarantee unseen near-duplicate visual or semantic information","The original 14/3/3 reserved participants may have existing stim_train EEG; verify epoch counts in prepared cache before using local_test","A 1500-image gallery is not comparable numerically to 200-image Top-5 benchmarks","Do not compare raw validation accuracy across the three-way and two-way protocols"],"science_sha256":"6509d84c6aa9ba2559f18b994f8a606e160cde3369843ad87a6beea776e434db","compact_note":"image_ids / candidate_gallery_image_ids removed; rebuilt by expand_team_manifest","prepared_loader_audit":{"science_sha256":"6509d84c6aa9ba2559f18b994f8a606e160cde3369843ad87a6beea776e434db","protocol":"xu2025alljoined_threeway_subject_concept_disjoint_v1","partitions_exposed":["train","validation"],"counts":{"train_from_train":618868,"excluded":551900,"validation_from_train":13717,"original_train":1065150,"train_from_val":146688,"validation_from_val":4443,"original_val":270466,"train_total":765556,"validation_total":18160},"train_subjects":["sub-01","sub-02","sub-03","sub-04","sub-05","sub-07","sub-10","sub-12","sub-13","sub-14","sub-15","sub-17","sub-18","sub-20"],"validation_subjects":["sub-06","sub-08","sub-19"],"train_images":13540,"validation_images":1500,"validation_gallery_size":1500,"no_local_test_eeg_read":true,"no_public_test_eeg_read":true,"meta":{"ch_names":["Cz","FCz","AFz","Fp1","F5","F1","CP5","CP3","CP1","P1","P3","P5","P7","PO7","PO3","O1","Pz","POz","Oz","O2","PO4","PO8","P8","P6","P4","P2","CP2","CP4","CP6","F2","F6","Fp2"],"sfreq":120.0,"n_chans":32,"n_times":120,"n_outputs":1536}}}

In [ ]:
%%writefile {SRC}/t1_train.py
"""Training of one architecture (REVE fine-tuning or the bidirectional Conformer from scratch) on the team split,
data-parallel over every GPU (launched with torchrun). usage: t1_train.py <config.json> <seed> <arch>

Split: the team's frozen manifest (TRAIN = FIT / VAL / local TEST, subject- and concept-disjoint). Writes, in
<out_dir>/<run>/: best.pt and swa.pt (weights), model_cfg.json, history.csv, metrics.json, and per variant v in {best, swa}:
  oof_val_<v>.npy, pred_test_<v>.npy, pred_new_<v>.npy   one row per EEG epoch (row order = split/keys_<SET>.parquet)
  oof_val_<v>_queries.npz, pred_test_<v>_queries.npz, pred_new_<v>_queries.npz   one row per subject x image (mean of
  the repeats, as the official metric): subject_id, image_id, concept_id, n_repeats, pred
All predictions are float16, L2-normalised.
"""
import contextlib
import copy
import datetime
import json
import math
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.distributed as dist
import torch.nn as nn
import torch.nn.functional as F

sys.path.insert(0, str(Path(__file__).parent))
from t1_common import *  # noqa: E402,F401,F403

cfg = load_cfg(sys.argv[1])
SEED = int(sys.argv[2])
ARCH = sys.argv[3] if len(sys.argv) > 3 else "reve"
R = cfg["recipes"][ARCH]
IS_REVE = ARCH == "reve"
IS_AMSPBE = ARCH == "amspbe"
N_MEMBERS = int(R.get("n_members", 1)) if IS_AMSPBE else 1
if R.get("amp") is False:                        # e.g. AMSPBE (BatchNorm, tiny model): float32 training and inference
    AMP = None
RUN_NAME = f"{ARCH}_{R['tag']}_s{SEED}"
OUT = Path(cfg["out_dir"]) / RUN_NAME
WORLD, RANK, LOCAL = int(os.environ.get("WORLD_SIZE", 1)), int(os.environ.get("RANK", 0)), int(os.environ.get("LOCAL_RANK", 0))
if WORLD > 1:
    dist.init_process_group("nccl" if torch.cuda.is_available() else "gloo", timeout=datetime.timedelta(minutes=30))
DEVICE = torch.device(f"cuda:{LOCAL}") if torch.cuda.is_available() else torch.device("cpu")
if DEVICE.type == "cuda":
    torch.cuda.set_device(DEVICE)
MAIN = RANK == 0
AMP = {"auto": amp_dtype(DEVICE), "fp16": torch.float16, "bf16": torch.bfloat16, "fp32": None}[cfg.get("precision", "auto")]
if DEVICE.type != "cuda":
    AMP = None
torch.backends.cudnn.benchmark = True


def barrier():
    if WORLD > 1:
        dist.barrier()


def say(*a):
    if MAIN:
        log(*a)


def agree(flag):
    """The main process's decision, shared with every process."""
    if WORLD == 1:
        return bool(flag)
    t = torch.tensor([int(flag)], device=DEVICE if dist.get_backend() == "nccl" else "cpu")
    dist.broadcast(t, 0)
    return bool(t.item())


# ---------------------------------------------------------------- data and split
CACHE = Path(cfg["cache"])
META = json.loads((CACHE / "meta.json").read_text())
CH, NT, D_OUT, SF = META["ch_names"], int(META["n_times"]), int(META["d_out"]), float(META["sfreq"])
P = TrialPool(CACHE)
trials = load_trials(CACHE)
EMB = np.load(CACHE / "emb_table.npy").astype(np.float32)
EMB_N_T = torch.from_numpy(EMB / np.linalg.norm(EMB, axis=1, keepdims=True)).float().to(DEVICE)
S, SPLIT_INFO, IDS = build_split(P, trials, cfg, EMB)
FIT, VAL, TEST, NEW = S["FIT"], S["VAL"], S["TEST"], S["NEW"]
CONCEPT_OF_IMG = np.full(int(P.img.max()) + 1, -1, np.int64); CONCEPT_OF_IMG[P.img] = IDS["concept_id"]
MAN = json.loads(Path(cfg["split_manifest"]).read_text())
NAME_OF = {int(k): v for k, v in (MAN.get("concept_names") or {}).items()}
STIM = trials["filepath"].map(lambda p: str(p).replace("\\", "/").rsplit("/", 1)[-1]).to_numpy()
if MAIN:
    OUT.mkdir(parents=True, exist_ok=True)
    sdir = Path(cfg["out_dir"]) / "split"
    sdir.mkdir(parents=True, exist_ok=True)
    (sdir / "split_info.json").write_text(json.dumps(SPLIT_INFO, indent=1))
    for name, g in (("VAL", VAL), ("TEST", TEST), ("NEW", NEW)):   # row order of every epoch-level prediction file of that set
        keys = pd.DataFrame(dict(row=np.arange(len(g)), subject_id=IDS["subject_id"][g], image_id=IDS["image_id"][g],
                                 concept_id=IDS["concept_id"][g], concept=[NAME_OF.get(int(c), "") for c in IDS["concept_id"][g]],
                                 stimulus=STIM[g],
                                 official_split=trials["official_split"].iloc[g].to_numpy()))
        for c in ("session", "run", "start"):
            if c in trials.columns:
                keys[c] = trials[c].iloc[g].to_numpy()
        keys.to_parquet(sdir / f"keys_{name}.parquet", index=False)
    pairs = np.unique(np.stack([IDS["image_id"], P.img], 1), axis=0)        # DINOv2 targets in the canonical numbering
    T = np.zeros((int(pairs[:, 0].max()) + 1, EMB.shape[1]), np.float16); T[pairs[:, 0]] = EMB[pairs[:, 1]]
    np.save(sdir / "dinov2_targets_by_image_id.npy", T)
    say(f"canonical ids: {json.dumps(SPLIT_INFO['concepts'])}")
    say(f"split ({SPLIT_INFO['source']}, manifest sha256 {SPLIT_INFO['checksums']['manifest'][:16]}…): subjects {SPLIT_INFO['subjects']}")
    for k, v in SPLIT_INFO["epochs_vs_team_loader"].items():
        say(f"epochs {k}: ours {v['ours']:,} vs team loader {v['team']:,} {'ok' if v['same'] else 'WARNING: DIFFERENT (not the same prepared data?)'}")
    say(f"sets: {json.dumps(SPLIT_INFO['sets'])} | unused epochs {SPLIT_INFO['unused_epochs']:,} | {WORLD} process(es) | AMP {AMP}")


# ---------------------------------------------------------------- REVE weights and electrode positions (braindecode, training only)
def reve_reference():
    from braindecode.models import REVE
    kw = dict(n_outputs=1, n_chans=32, n_times=200, sfreq=200, chs_info=None)
    if R.get("reve_tiny"):                      # offline code tests only: tiny random REVE
        with torch.random.fork_rng(devices=[]):
            torch.manual_seed(0)
            ref = REVE(**kw, **R["reve_tiny"])
    else:
        ref = REVE.from_pretrained("brain-bzh/reve-base", **kw)
    ref.final_layer = nn.Identity()
    return ref.eval()


def reve_positions(ch_names):
    from braindecode.models.reve import RevePositionBank
    bank = RevePositionBank()
    lower = {k.lower(): k for k in bank.mapping}
    names = [str(c).strip() for c in ch_names]
    mapped = [n if n in bank.mapping else lower.get(n.lower()) for n in names]
    assert None not in mapped, f"no REVE position for {[n for n, m in zip(names, mapped) if m is None]}"
    return bank.forward(mapped).float().clone()


if IS_REVE:
    for turn in (0, 1):                             # the main process downloads first, the others then read the cache
        if (turn == 0) == MAIN:
            REF = reve_reference()
            REVE_SD = {k: v.detach().cpu().clone() for k, v in REF.state_dict().items()}
            POS = reve_positions(CH)
        barrier()
    if MAIN:                                        # the vendored encoder (shipped in the ZIP) must equal braindecode's REVE
        enc = ReveEncoder(**(R.get("reve_tiny") or {})).eval()
        enc.load_state_dict(REVE_SD, strict=True)
        with torch.no_grad():
            x, p = torch.randn(2, 32, 200), POS[None].expand(2, -1, -1)
            diff = float((REF(x, pos=p, return_features=True)["features"] - enc(x, p)).abs().max())
        assert diff < 1e-4, f"vendored REVE differs from braindecode's ({diff})"
        say(f"vendored REVE = braindecode REVE (max |difference| {diff:.1e}) ok")
        del enc
    del REF


def model_cfg():
    if IS_AMSPBE:                               # the team AMSPBE ensemble (DTHA brain encoder), same hyper-parameters
        return dict(arch="amspbe", arch_kw=dict(R["amspbe"], n_members=N_MEMBERS), train_ch_names=list(CH), n_times_train=int(NT),
                    d_out=int(D_OUT), sfreq=float(SF), post=dict(center=False, norm="trial", lam=0.0), input_norm=None,
                    amp=R.get("amp", True))
    if ARCH in ("rescnn", "tcnlc"):             # residual CNN (p13) / multi-scale dilated TCN + log-covariance pooling
        return dict(arch=ARCH, arch_kw=dict(R.get(ARCH) or {}), train_ch_names=list(CH), n_times_train=int(NT), d_out=int(D_OUT),
                    sfreq=float(SF), post=dict(center=False, norm="trial", lam=0.0), input_norm=None, amp=R.get("amp", True))
    if not IS_REVE:                             # the round-2 / Codabench bidirectional Conformer, exactly
        return dict(arch="conformer", bidir=bool(R.get("bidir", True)), conformer=dict(R["conformer"]), train_ch_names=list(CH),
                    n_times_train=int(NT), d_out=int(D_OUT), sfreq=float(SF), post=None)
    kw = dict(tok_dim=64, proj_drop=0.5, pretrained=True, **(R.get("head_kw") or {}))
    if R.get("reve_tiny"):
        kw["reve_kw"] = dict(R["reve_tiny"])
    mc = dict(arch="reve", bidir=False, arch_kw=kw, train_ch_names=list(CH), n_times_train=NT, d_out=D_OUT, sfreq=SF,
              post=dict(center=False, norm="trial", lam=0.0), input_norm=None)
    if R.get("aux"):                            # jointly trained second branch (e.g. the bidirectional Conformer)
        mc["aux"] = json.loads(json.dumps(R["aux"]))
    return mc


class _GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lam):
        ctx.lam = lam
        return x.view_as(x)

    @staticmethod
    def backward(ctx, g):
        return -ctx.lam * g, None


class TrainWrap(nn.Module):
    """The network + the learned inverse temperature of the loss (inside the module so that DDP averages its gradient)."""

    def __init__(self, net, t0, n_subjects=0):
        super().__init__()
        self.net = net
        self.logit_scale = nn.Parameter(torch.full((N_MEMBERS,), math.log(1.0 / t0)) if N_MEMBERS > 1 else torch.tensor(math.log(1.0 / t0)))
        self.adv = (nn.Sequential(nn.Linear(D_OUT, 256), nn.GELU(), nn.Linear(256, n_subjects)) if n_subjects else None)
        self.grl = 0.0                               # gradient-reversal strength (ramped during training)

    def forward(self, X):
        if isinstance(X, (list, tuple)):             # AMSPBE: one view per member
            return self.net.members_forward(X), self.logit_scale, None
        z = self.net(X)
        a = self.adv(_GradReverse.apply(F.normalize(z.float(), dim=-1), self.grl)) if self.adv is not None else None
        return z, self.logit_scale, a


def cpu_state(m):
    return {k: v.detach().cpu().clone() for k, v in m.state_dict().items()}


def average_states(states):
    return {k: (torch.stack([s[k].float() for s in states]).mean(0).to(v.dtype) if v.is_floating_point() else v.clone())
            for k, v in states[-1].items()}


def reve_depth(name):
    if ".reve." not in f".{name}.":
        return None
    import re as _re
    m = _re.search(r"\.reve\.transformer\.layers\.(\d+)\.", f".{name}.")
    return int(m.group(1)) if m else -1


def param_groups(wrap):
    """Differential learning rates: head at lr; REVE at lr x backbone_lr_mult x llrd^(layers above it);
    the freeze_layers lowest layers (+ embeddings) never train. Conformer: one learning rate for everything."""
    if not IS_REVE:
        nd = lambda n, p: p.ndim <= 1 or n.endswith(("time_att", "chan_att"))      # AMSPBE: no decay on attention curves, biases, norms
        named = [(n, p) for n, p in wrap.named_parameters() if n != "logit_scale"]
        if IS_AMSPBE or ARCH in ("rescnn", "tcnlc"):
            return [dict(params=[p for n, p in named if not nd(n, p)], lr=R["lr"], weight_decay=R["weight_decay"], kind="head"),
                    dict(params=[p for n, p in named if nd(n, p)], lr=R["lr"], weight_decay=0.0, kind="head"),
                    dict(params=[wrap.logit_scale], lr=R["lr"], weight_decay=0.0, kind="scale")]
        return [dict(params=[p for n, p in named], lr=R["lr"], weight_decay=R["weight_decay"], kind="head"),
                dict(params=[wrap.logit_scale], lr=R["lr"], weight_decay=0.0, kind="scale")]
    depth = wrap.net.backbone.reve.depth
    head, reve = [], {}
    for n, p in wrap.named_parameters():
        if n == "logit_scale":
            continue
        d = reve_depth(n)
        if d is None:
            head.append(p)
        elif R.get("freeze_layers") and d < R["freeze_layers"]:
            p.requires_grad_(False)
        else:
            reve.setdefault(d, []).append(p)
    groups = [dict(params=head, lr=R["lr"], weight_decay=R["weight_decay"], kind="head"),
              dict(params=[wrap.logit_scale], lr=R["lr"], weight_decay=0.0, kind="scale")]
    for d, ps in sorted(reve.items()):
        mult = R["backbone_lr_mult"] * (R["llrd"] ** (depth - 1 - d) if R.get("llrd") else 1.0)
        groups.append(dict(params=ps, lr=R["lr"] * mult, weight_decay=R["weight_decay"], kind="reve"))
    return groups


def augment(X, aug, gen):
    B_, C, T = X.shape
    s = torch.randint(-aug["shift"], aug["shift"] + 1, (B_, 1, 1), device=X.device, generator=gen)
    X = X.gather(2, (torch.arange(T, device=X.device)[None, None, :] - s).clamp(0, T - 1).expand(B_, C, T))
    X = X * (1 + aug["amp"] * torch.randn(B_, 1, 1, device=X.device, generator=gen))
    X = X * (torch.rand(B_, C, 1, device=X.device, generator=gen) > aug["ch_drop"]).float()
    return X + aug["noise"] * torch.randn(X.shape, device=X.device, generator=gen)


def group_index(gidx):
    key = P.subj[gidx] * (int(P.img.max()) + 1) + P.img[gidx]
    order_ = np.argsort(key, kind="stable"); sk_ = key[order_]
    st = np.flatnonzero(np.r_[True, sk_[1:] != sk_[:-1]]); sz = np.diff(np.r_[st, len(sk_)])
    grp = np.empty(len(gidx), np.int64); grp[order_] = np.repeat(np.arange(len(st)), sz)
    return order_, st[grp], sz[grp]


def load_batch(pos, aug, rng, grp):
    """Epochs FIT[pos]; a share avg_p of them is replaced by the mean of k repeats of the same person x image
    (k = avg_k, or drawn per epoch in [avg_k, avg_kmax])."""
    X = P.get(FIT[pos]).astype(np.float32)
    m = np.flatnonzero(rng.random(len(pos)) < aug["avg_p"])
    if len(m):
        order_, g_start, g_size = grp
        kmax = int(aug.get("avg_kmax") or aug["avg_k"])
        k = rng.integers(int(aug["avg_k"]), kmax + 1, len(m)) if kmax > aug["avg_k"] else np.full(len(m), int(aug["avg_k"]))
        acc = X[m].copy()
        for r in range(kmax - 1):
            use = np.flatnonzero(r < k - 1)
            if not len(use):
                break
            o = g_start[pos[m[use]]] + (rng.random(len(use)) * g_size[pos[m[use]]]).astype(np.int64)
            acc[use] += P.get(FIT[order_[o]]).astype(np.float32)
        X[m] = acc / k[:, None, None]
    return X


@torch.no_grad()
def predict(net, gidx, chunk=512):
    """L2-normalised predictions (the submission's post-calibration), in the order of gidx."""
    net.eval()
    out = []
    for s in range(0, len(gidx), chunk):
        X = torch.from_numpy(P.get(gidx[s:s + chunk])).to(DEVICE).float()
        with torch.autocast("cuda", dtype=AMP or torch.float16, enabled=AMP is not None):
            z = net(X).float()
        out.append(F.normalize(z, dim=-1).cpu().numpy())
    Z = np.concatenate(out)
    if not np.isfinite(Z).all():
        say(f"WARNING: {int((~np.isfinite(Z)).any(1).sum())} non-finite predictions (mixed-precision overflow?)")
    return Z


VAL_MON = VAL                                       # the per-epoch monitor: VAL, or a fixed subset of its subject x image groups
if cfg.get("monitor_max_epochs") and len(VAL) > int(cfg["monitor_max_epochs"]):
    _key = IDS["subject_id"][VAL].astype(np.int64) * 100000 + IDS["image_id"][VAL]
    _uk, _inv, _cnt = np.unique(_key, return_inverse=True, return_counts=True)
    _order = np.random.default_rng(0).permutation(len(_uk))
    _keep = _order[:int(np.searchsorted(np.cumsum(_cnt[_order]), int(cfg["monitor_max_epochs"])))]
    VAL_MON = VAL[np.isin(_inv, _keep)]


def val_mrp(net):
    return scores(predict(net, VAL_MON), VAL_MON, P, EMB_N_T)["mrp@all"]


def probe_micro(net, want, n_gal):
    if DEVICE.type != "cuda":
        return want
    total, b = torch.cuda.get_device_properties(DEVICE).total_memory, want
    net.train()
    while b >= 8:
        try:
            torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(DEVICE)
            with torch.autocast("cuda", dtype=AMP or torch.float16, enabled=AMP is not None):
                out = net(torch.randn(b, len(CH), NT, device=DEVICE))
            (F.normalize(out.float(), dim=-1) @ torch.randn(n_gal, out.shape[1], device=DEVICE).T).logsumexp(1).mean().backward()
            peak = torch.cuda.max_memory_allocated(DEVICE)
            net.zero_grad(set_to_none=True); del out
            if peak < cfg["gpu_mem_fraction"] * total:
                return b
        except torch.cuda.OutOfMemoryError:
            net.zero_grad(set_to_none=True)
        torch.cuda.empty_cache(); b //= 2
    return 8


class EMA:
    def __init__(self, model, decay):
        self.model = copy.deepcopy(model).eval()
        for p in self.model.parameters():
            p.requires_grad_(False)
        self.decay, self.n = decay, 0

    @torch.no_grad()
    def update(self, model):
        self.n += 1
        d = min(self.decay, (1 + self.n) / (10 + self.n))
        for a, b in zip(self.model.parameters(), model.parameters()):
            a.lerp_(b.detach(), 1 - d)
        for a, b in zip(self.model.buffers(), model.buffers()):
            a.copy_(b)


# ---------------------------------------------------------------- training
torch.manual_seed(int(R.get("init_seed", SEED)))     # init_seed fixed across seeds -> runs can be souped (weights averaged)
net = build_model(model_cfg())
torch.manual_seed(SEED); np.random.seed(SEED)
net.set_input_channels(list(CH), len(CH))
if IS_REVE:
    with torch.no_grad():
        net.backbone.pos.copy_(POS)
    net.backbone.reve.load_state_dict(REVE_SD, strict=True)
FIT_SUBJ = np.unique(IDS["subject_id"][FIT])
SUBJ_IDX = torch.as_tensor(np.searchsorted(FIT_SUBJ, IDS["subject_id"]), device="cpu")      # 0..13 for FIT people
wrap = TrainWrap(net, R["temperature_init"], len(FIT_SUBJ) if R.get("adv_w") else 0).to(DEVICE)
groups = param_groups(wrap)
fit_imgs = np.unique(P.img[FIT]); fit_imgs_t = torch.as_tensor(fit_imgs, device=DEVICE); G_fit = EMB_N_T[fit_imgs_t]
Gc_fit = F.normalize(G_fit - G_fit.mean(0, keepdim=True), dim=-1)    # centred targets: semantic neighbours for soft targets
SOFT_EPS, SOFT_TAU, COS_W = float(R.get("soft_eps", 0)), float(R.get("soft_tau", 0.1)), float(R.get("cos_w", 0))
SYM_W = float(R.get("sym_w", 0))                        # weight of the image→EEG (in-batch) direction of the InfoNCE loss
MIX_P, MIX_A, ADV_W = float(R.get("mixup_p", 0)), float(R.get("mixup_alpha", 0.4)), float(R.get("adv_w", 0))
GB = R["batch"]; LB = GB // WORLD                       # global batch, per-process batch
micro = probe_micro(net, LB, len(fit_imgs))
ddp = torch.nn.parallel.DistributedDataParallel(wrap, device_ids=[LOCAL] if DEVICE.type == "cuda" else None) if WORLD > 1 else wrap
opt = torch.optim.AdamW([{k: v for k, v in g.items() if k != "kind"} for g in groups])
steps = max(1, min(len(FIT), R["samples_per_epoch"]) // GB)
E = R["epochs"]; total = max(8, E * steps)
if R["schedule"] == "onecycle":
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[g["lr"] for g in groups], total_steps=total, pct_start=max(0.1, 2.0 / total))
else:                                                   # "const": linear warm-up over the first epoch, then constant
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / max(2, steps)))
scaler = torch.amp.GradScaler("cuda", enabled=AMP == torch.float16)
ema = EMA(net, R["ema"]) if (MAIN and R.get("ema")) else None
reve_params = [p for g in groups if g["kind"] == "reve" for p in g["params"]]
train_params = [p for g in groups for p in g["params"] if p.requires_grad]
rng_perm = np.random.default_rng(SEED)                  # same epoch order in every process
rng_aug = np.random.default_rng(SEED * 100 + RANK)      # different augmentations per process
gen = torch.Generator(device=DEVICE); gen.manual_seed(SEED * 100 + RANK)
grp = group_index(FIT)
say(f"{RUN_NAME}: {sum(p.numel() for p in train_params) / 1e6:.1f} M trainable of {sum(p.numel() for p in wrap.parameters()) / 1e6:.1f} M | "
    f"{E} epochs x {steps} steps x {GB} (= {WORLD} x {LB}, pieces of {micro}) | {len(fit_imgs):,} training images in the loss gallery")
hist, states, best, t0 = [], [], None, time.time()
deadline = cfg.get("deadline_unix") or 1e18
for ep in range(E):
    te = time.time()
    frozen = ep < R.get("freeze_epochs", 0)
    wrap.train()
    perm = rng_perm.permutation(len(FIT))[:steps * GB]
    tot = torch.zeros((), device=DEVICE)
    skipped, t_load = 0, 0.0
    for b in range(steps):
        mine = perm[b * GB:(b + 1) * GB][RANK * LB:(RANK + 1) * LB]
        opt.zero_grad(set_to_none=True)
        pieces = [mine[i:i + micro] for i in range(0, len(mine), micro)]
        for j, pos in enumerate(pieces):
            sync = contextlib.nullcontext() if (WORLD == 1 or j == len(pieces) - 1) else ddp.no_sync()
            with sync:
                tl = time.time(); Xn = load_batch(pos, R["aug"], rng_aug, grp); t_load += time.time() - tl
                X = torch.from_numpy(Xn).to(DEVICE)
                y = torch.searchsorted(fit_imgs_t, torch.as_tensor(P.img[FIT[pos]], device=DEVICE))   # index in the FIT gallery
                lam = None
                if MIX_P > 0:                           # mixup between different images (soft targets mixed alike)
                    lam = torch.ones(len(pos), device=DEVICE)
                    sel = torch.rand(len(pos), device=DEVICE, generator=gen) < MIX_P
                    beta = torch.distributions.Beta(MIX_A, MIX_A).sample((len(pos),)).to(DEVICE)
                    lam = torch.where(sel, torch.maximum(beta, 1 - beta), lam)
                    pm = torch.randperm(len(pos), device=DEVICE, generator=gen)
                    X = lam[:, None, None] * X + (1 - lam[:, None, None]) * X[pm]
                if IS_AMSPBE:                           # every member: its own augmented view, its own temperature, its own loss
                    Xs = [augment(X, R["aug"], gen) for _ in range(N_MEMBERS)]
                    with torch.autocast("cuda", dtype=AMP or torch.float16, enabled=AMP is not None):
                        outs, ls, _ = ddp(Xs)
                    loss = sum(F.cross_entropy(ls[m].float().exp() * F.normalize(o.float(), dim=-1) @ G_fit.T, y)
                               for m, o in enumerate(outs)) / N_MEMBERS * (len(pos) / len(mine))
                    scaler.scale(loss).backward()
                    tot += loss.detach()
                    continue
                X = augment(X, R["aug"], gen)
                with torch.autocast("cuda", dtype=AMP or torch.float16, enabled=AMP is not None):
                    out, ls, adv = ddp(X)
                zn = F.normalize(out.float(), dim=-1)
                logp = F.log_softmax(ls.float().exp() * zn @ G_fit.T, dim=-1)

                def target(yy):
                    t = F.one_hot(yy, len(fit_imgs)).float()
                    if SOFT_EPS > 0:                    # semantic label smoothing: mass on visually similar training images
                        t = (1 - SOFT_EPS) * t + SOFT_EPS * F.softmax(Gc_fit[yy] @ Gc_fit.T / SOFT_TAU, dim=-1)
                    return t
                T_ = target(y) if lam is None else lam[:, None] * target(y) + (1 - lam[:, None]) * target(y[pm])
                loss = -(T_ * logp).sum(1).mean()
                if COS_W > 0:                           # + cosine regression to the (mixed) target embedding
                    tv = G_fit[y] if lam is None else F.normalize(lam[:, None] * G_fit[y] + (1 - lam[:, None]) * G_fit[y[pm]], dim=-1)
                    loss = loss + COS_W * (1 - (zn * tv).sum(1)).mean()
                if SYM_W > 0 and lam is None:           # symmetric InfoNCE: each batch image must also find its own EEG epochs
                    same = (y[:, None] == y[None, :]).float()
                    lg = F.log_softmax(ls.float().exp() * G_fit[y] @ zn.T, dim=-1)
                    loss = loss + SYM_W * (-(same / same.sum(1, keepdim=True) * lg).sum(1).mean())
                if adv is not None:                     # subject classifier behind a gradient reversal
                    loss = loss + ADV_W * F.cross_entropy(adv.float(), SUBJ_IDX[FIT[pos]].to(DEVICE))
                loss = loss * (len(pos) / len(mine))
                scaler.scale(loss).backward()
                tot += loss.detach()
        if frozen:                                      # head-only warm-up: REVE's gradients are dropped
            for p in reve_params:
                p.grad = None
        scaler.unscale_(opt)
        gnorm = torch.nn.utils.clip_grad_norm_([p for p in train_params if p.grad is not None], R["grad_clip"])
        if scaler.is_enabled():                         # float16: the loss scaler skips steps with inf/NaN gradients
            s0 = scaler.get_scale(); scaler.step(opt); scaler.update(); skipped += int(scaler.get_scale() < s0)
        elif torch.isfinite(gnorm):                     # gradients are averaged first, so every process decides alike
            opt.step()
        else:
            skipped += 1
        sched.step()
        if ADV_W > 0:                                   # DANN ramp 0 -> 1 over training
            prog = (ep * steps + b + 1) / (E * steps)
            wrap.grl = float(2.0 / (1.0 + math.exp(-10 * prog)) - 1.0)
        with torch.no_grad():
            wrap.logit_scale.clamp_(0, math.log(R["temperature_cap"]))
        if ema is not None:
            ema.update(net)
    if WORLD > 1:
        dist.all_reduce(tot)
    row = dict(epoch=ep + 1, loss=float(tot) / steps / WORLD, minutes=round((time.time() - t0) / 60, 1),
               samples_per_s=round(steps * GB / (time.time() - te)), scale=float(wrap.logit_scale.detach().exp().mean()),
               lr_head=f'{opt.param_groups[0]["lr"]:.2e}', skipped_steps=skipped,
               data_share=round(t_load / max(1e-9, time.time() - te), 2))
    if MAIN and ep > 0 and skipped > 0.05 * steps:
        log(f"WARNING: {skipped} of {steps} steps skipped (non-finite gradients): if this persists, set PRECISION = 'fp32'")
    stop = False
    if MAIN:
        states = (states + [cpu_state(net)])[-R["swa_keep"]:]
        for src, m in [("raw", net)] + ([("ema", ema.model)] if ema is not None else []):
            row[f"val_mrp_{src}"] = val_mrp(m)
            if best is None or row[f"val_mrp_{src}"] > best[0]:
                best = (row[f"val_mrp_{src}"], ep + 1, src, cpu_state(m))
        hist.append(row)
        log(" | ".join(f"{k} {v:.4f}" if isinstance(v, float) and k.startswith(("loss", "val", "scale")) else f"{k} {v}" for k, v in row.items()))
        per_epoch = (time.time() - t0) / (ep + 1)
        stop = ep + 1 < E and time.time() + per_epoch + cfg["finish_minutes"] * 60 > deadline
        if stop:
            log(f"WARNING: time: stopping after epoch {ep + 1} of {E} to finish before the session limit")
        elif R.get("patience") and ep + 1 < E and ep + 1 - best[1] >= R["patience"]:
            stop = True
            log(f"early stop: no VAL MRP gain for {R['patience']} epochs (best: epoch {best[1]})")
    if agree(stop):
        break
if WORLD > 1:                                           # safety: every process must hold the same weights
    cs = torch.tensor([sum(float(p.detach().double().sum()) for p in wrap.parameters())], dtype=torch.float64,
                      device=DEVICE if dist.get_backend() == "nccl" else "cpu")
    allcs = [torch.zeros_like(cs) for _ in range(WORLD)]
    dist.all_gather(allcs, cs)
    spread = max(abs(float(c) - float(cs)) for c in allcs)
    assert spread <= 1e-6 * max(1.0, abs(float(cs))), f"processes diverged (weight checksum spread {spread})"
    say(f"data-parallel check: the {WORLD} processes hold identical weights ok")
barrier()
if WORLD > 1:
    dist.destroy_process_group()
if not MAIN:
    sys.exit(0)

# ---------------------------------------------------------------- best and SWA variants, predictions, metrics (main process)
swa_rows, swa_best = [], None
for k in range(2, len(states) + 1):                    # SWA: uniform average of the last k epoch-end weights
    avg = average_states(states[-k:])
    net.load_state_dict(avg)
    v = val_mrp(net)
    swa_rows.append(dict(k=k, epochs=f"{len(hist) - k + 1}-{len(hist)}", val_mrp=v))
    if swa_best is None or v > swa_best[0]:
        swa_best = (v, k, avg)
if swa_best is None:                                    # a single epoch: the SWA variant is the last weights
    swa_best = (hist[-1]["val_mrp_raw"], 1, states[-1])
variants = {"best": (best[3], dict(kind="best", epoch=best[1], weights=best[2], val_mrp_monitor=best[0])),
            "swa": (swa_best[2], dict(kind="swa", k=swa_best[1], epochs=f"{len(hist) - swa_best[1] + 1}-{len(hist)}", val_mrp_monitor=swa_best[0]))}
mcfg = model_cfg()
(OUT / "model_cfg.json").write_text(json.dumps(mcfg, indent=1))
metrics = dict(run=RUN_NAME, arch=ARCH, seed=SEED, recipe=R, world_size=WORLD, amp=str(AMP), split=SPLIT_INFO["sets"],
               subjects=SPLIT_INFO["subjects"], manifest_sha256=SPLIT_INFO["checksums"]["manifest"], history=hist,
               swa_table=swa_rows, variants={})


def query_level(Z, g):
    """Mean prediction per subject x image (the official metric's aggregation), L2-normalised."""
    key = IDS["subject_id"][g].astype(np.int64) * 100000 + IDS["image_id"][g]
    uk, inv, cnt = np.unique(key, return_inverse=True, return_counts=True)
    M = np.zeros((len(uk), Z.shape[1]), np.float64); np.add.at(M, inv, Z.astype(np.float64))
    M /= np.maximum(np.linalg.norm(M, axis=1, keepdims=True), 1e-12)
    first = np.zeros(len(uk), np.int64); first[inv[::-1]] = g[::-1]
    return dict(subject_id=IDS["subject_id"][first], image_id=IDS["image_id"][first], concept_id=IDS["concept_id"][first],
                concept=np.array([NAME_OF.get(int(c), "") for c in IDS["concept_id"][first]]), n_repeats=cnt, pred=M.astype(np.float16))


def evaluate(Z, g, concept_level=True):
    sc = scores(Z, g, P, EMB_N_T)
    if concept_level:
        sc["top5_1img_per_concept"], sc["n_concepts"] = concept_top5(Z, g, P, EMB_N_T, CONCEPT_OF_IMG)
    return sc


for name, (state, desc) in variants.items():
    net.load_state_dict(state)
    if cfg.get("save_weights", True):
        torch.save(state, OUT / f"{name}.pt")
    res = {}
    for set_name, g, stem in [x for x in (("VAL", VAL, "oof_val"), ("TEST", TEST, "pred_test"), ("NEW", NEW, "pred_new"))
                              if x[0] in cfg.get("eval_sets", ["VAL", "TEST", "NEW"])]:
        Z = predict(net, g)
        if cfg.get("save_preds", True):
            np.save(OUT / f"{stem}_{name}.npy", Z.astype(np.float16))
            np.savez(OUT / f"{stem}_{name}_queries.npz", **query_level(Z, g))
        if set_name in ("TEST", "NEW") and not cfg.get("score_local_test", True):
            res[set_name] = None                         # team policy option: saved, not scored
        else:
            res[set_name] = evaluate(Z, g, concept_level=set_name != "NEW")
        if set_name == "VAL":                    # per-query rank percentiles: paired comparisons between runs
            A = agg(Z, g, P)
            _, frac = rank_in(A, np.unique(P.img[g]), EMB_N_T)
            np.savez(OUT / f"val_query_scores_{name}.npz", subj=A["subj"], img=A["img"], rank_pct=(1 - frac).astype(np.float32))
    metrics["variants"][name] = dict(**desc, **res)
    v, t, n = res["VAL"], res.get("TEST"), res.get("NEW")
    log(f"{name:4s} ({desc}):\n"
        f"      VAL  MRP {v['mrp@all']:.4f} | top-5 {v['top5@all']:.4f} among {v['n_gallery']} images | top-5 {v['top5_1img_per_concept']:.4f} among {v['n_concepts']} concepts\n"
        + (f"      TEST MRP {t['mrp@all']:.4f} | top-5 {t['top5@all']:.4f} among {t['n_gallery']} images | top-5 {t['top5_1img_per_concept']:.4f} among {t['n_concepts']} concepts\n"
           f"      NEW  MRP {n['mrp@all']:.4f} | top-5 {n['top5@all']:.4f} among the {n['n_gallery']} official test images (local-test people)"
           if t is not None else "      TEST / NEW: not scored here"))
rec = max(metrics["variants"], key=lambda n: metrics["variants"][n]["VAL"]["mrp@all"])
metrics["recommended"] = rec
metrics["train_minutes"] = round((time.time() - t0) / 60, 1)
pd.DataFrame(hist).to_csv(OUT / "history.csv", index=False)
(OUT / "metrics.json").write_text(json.dumps(metrics, indent=1, default=str))
log(f"done in {metrics['train_minutes']} min: recommended variant on VAL = {rec} → {OUT}")

In [ ]:
%%writefile {SRC}/t1_cv_ensemble.py
"""5-fold cross-validated ensemble (3 architectures x 5 folds = 15 models) on the team seed-33 manifest.

Folds: the 17 non-local-test people and the 1,504 non-local-test concepts are each cut into 5 groups (seed 33). Fold k
trains on the other people x the other concepts and is validated on (people_k x concepts_k): its OUT-OF-FOLD block.
The local test (people 9, 11, 16 x their 150 concepts) is never trained on and is scored once, here, at the end.

1. Out-of-fold scores of every model on its own block (MRP, top-5 among the block's images, top-5 among 150 concepts with
   one image each: the Codabench-like measure), all computed the same way.
2. Architecture weights chosen on the pooled out-of-fold blocks (5 x more queries than the single VAL split); kept only if
   they beat equal weights by a margin, otherwise equal weights. Inside an architecture, the 5 fold models count equally.
3. Local test and NEW (local-test people x the 200 official test images) scored once: each architecture's 5-fold average,
   equal weights, the chosen ensemble.
4. One ZIP with the 15 models (weights float16), checked: layout, parity with the in-notebook ensemble, offline replay
   with the official metric, the kit's harness, timing.

usage: python t1_cv_ensemble.py <config.json>
"""
import datetime
import itertools
import json
import os
import runpy
import shutil
import subprocess
import sys
import tempfile
import time
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

sys.path.insert(0, str(Path(__file__).parent))
from t1_common import *  # noqa: E402,F401,F403
from t1_common import MODEL_SRC  # noqa: E402

cfg = load_cfg(sys.argv[1])
CV = cfg["cv"]
OUT = Path(cfg["out_dir"]) / "cv_ensemble"
OUT.mkdir(parents=True, exist_ok=True)
WORK = Path(cfg["scratch"]) / "cv_ensemble_work"
shutil.rmtree(WORK, ignore_errors=True); WORK.mkdir(parents=True)
TRACK = Path(cfg["kit"]) / "tracks" / "image_decoding"
sys.path.insert(0, str(TRACK))
DEVICE = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
CACHE = Path(cfg["cache"])
META = json.loads((CACHE / "meta.json").read_text())
CH, SF, D_OUT = META["ch_names"], float(META["sfreq"]), int(META["d_out"])
ARCHS = CV["archs"]
FOLDS = CV["folds"]                                   # [{k, split_dir, runs: {arch: run_dir}}]
N_WAY, DRAWS = int(CV.get("n_way", 150)), int(CV.get("draws", 20))


# ---------------------------------------------------------------- query-level predictions (exactly the scorer's aggregation)
def load_set(run_dir, set_name, split_dir):
    m = json.loads((Path(run_dir) / "metrics.json").read_text())
    v = m["recommended"]
    stem = {"VAL": "oof_val", "TEST": "pred_test", "NEW": "pred_new"}[set_name]
    Z = np.load(Path(run_dir) / f"{stem}_{v}.npy").astype(np.float32)
    keys = pd.read_parquet(Path(split_dir) / f"keys_{set_name}.parquet")
    assert len(keys) == len(Z), f"{run_dir}: {set_name} predictions and keys differ in length"
    return Z, keys, v


def to_queries(Z, keys):
    """Mean over a person's repeats of an image of the L2-normalised epoch predictions (NOT re-normalised: a weighted sum
    of these is exactly what the ensemble ZIP produces after the scorer's averaging)."""
    key = keys["subject_id"].to_numpy(np.int64) * 100000 + keys["image_id"].to_numpy(np.int64)
    uk, inv, cnt = np.unique(key, return_inverse=True, return_counts=True)
    M = np.zeros((len(uk), Z.shape[1]), np.float64); np.add.at(M, inv, Z / np.maximum(np.linalg.norm(Z, axis=1, keepdims=True), 1e-12))
    M /= cnt[:, None]
    return uk, M.astype(np.float32)


T_ALL = None
for f in FOLDS:
    p = Path(f["split_dir"]) / "dinov2_targets_by_image_id.npy"
    if p.exists():
        T_ALL = np.load(p).astype(np.float32)
        break
T_N = torch.from_numpy(T_ALL / np.maximum(np.linalg.norm(T_ALL, axis=1, keepdims=True), 1e-12)).to(DEVICE)


def score(M, uk, n_way=N_WAY, draws=DRAWS, seed=0):
    """MRP and top-5 among the set's images, and top-5 among `n_way` concepts with one image each (averaged over draws)."""
    img = (uk % 100000).astype(np.int64)
    gal = np.unique(img)
    con = gal // 10 + 1 if gal.max() < 16540 else np.arange(len(gal))          # official test pool: one concept per image
    Q = F.normalize(torch.from_numpy(M).to(DEVICE), dim=-1)
    G = T_N[torch.as_tensor(gal, device=DEVICE)]
    S = Q @ G.T
    t = torch.as_tensor(np.searchsorted(gal, img), device=DEVICE)
    true = S.gather(1, t[:, None])
    r = ((S > true).sum(1).float() + ((S == true).sum(1).float() - 1) / 2).cpu().numpy()
    out = {"mrp": float(1 - (r / max(1, len(gal) - 1)).mean()), "top5_img": float((r < 5).mean()), "n_gallery": int(len(gal)),
           "n_queries": int(len(img))}
    uc = np.unique(con)
    if len(uc) > 1 and gal.max() < 16540:
        rng = np.random.default_rng(seed)
        vals = []
        for _ in range(draws):
            cs = rng.choice(uc, min(n_way, len(uc)), replace=False)
            pick = np.sort([rng.choice(np.flatnonzero(con == c)) for c in cs])          # one image per concept (gallery positions)
            q = np.flatnonzero(np.isin(img, gal[pick]))
            if not len(q):
                continue
            Sg = S[torch.as_tensor(q, device=DEVICE)][:, torch.as_tensor(pick, device=DEVICE)]
            tg = torch.as_tensor(np.searchsorted(gal[pick], img[q]), device=DEVICE)
            tr = Sg.gather(1, tg[:, None])
            rr = (Sg > tr).sum(1)
            vals.append(float((rr < 5).float().mean()))
        out["top5_concepts"], out["n_way"] = float(np.mean(vals)), int(min(n_way, len(uc)))
    elif gal.max() >= 16540:                                                          # NEW: the 200 official images
        out["top5_concepts"], out["n_way"] = out["top5_img"], int(len(gal))
    return out


def sel(sc):
    return sc["top5_concepts"] + 0.5 * sc["mrp"]


# ---------------------------------------------------------------- 1 · out-of-fold blocks
OOF = {a: [] for a in ARCHS}
rows = []
for f in FOLDS:
    for a in ARCHS:
        rd = f["runs"].get(a)
        if not rd or not (Path(rd) / "metrics.json").exists():
            continue
        Z, keys, v = load_set(rd, "VAL", f["split_dir"])
        uk, M = to_queries(Z, keys)
        sc = score(M, uk)
        OOF[a].append(dict(k=f["k"], uk=uk, M=M))
        m = json.loads((Path(rd) / "metrics.json").read_text())
        rows.append(dict(arch=a, fold=f["k"], variant=v, people=len(np.unique(keys["subject_id"])),
                         concepts=len(np.unique(keys["concept_id"])), **{k_: sc[k_] for k_ in ("mrp", "top5_concepts", "top5_img")},
                         minutes=m.get("train_minutes")))
        log(f"OOF {a:16s} fold {f['k']}: MRP {sc['mrp']:.4f} | top-5/{sc['n_way']} concepts {sc['top5_concepts']:.4f} | "
            f"top-5/{sc['n_gallery']} images {sc['top5_img']:.4f} ({v})")
fold_table = pd.DataFrame(rows)
fold_table.to_csv(OUT / "oof_by_fold.csv", index=False)
ARCHS = [a for a in ARCHS if OOF[a]]
common_folds = sorted(set.intersection(*[{o["k"] for o in OOF[a]} for a in ARCHS]))
assert common_folds, "no fold has every architecture"
log(f"architectures {ARCHS} | folds with every architecture: {common_folds}")
# sanity: our query-level scoring equals the training script's scoring (same MRP) on one block
_chk = json.loads((Path(FOLDS[0]["runs"][ARCHS[0]]) / "metrics.json").read_text())
_mrp_train = _chk["variants"][_chk["recommended"]]["VAL"]["mrp@all"]
_mrp_here = next(r["mrp"] for r in rows if r["arch"] == ARCHS[0] and r["fold"] == FOLDS[0]["k"])
assert abs(_mrp_train - _mrp_here) < 2e-3, f"scoring mismatch: {_mrp_train} vs {_mrp_here}"


def oof_scores(w):
    """Mean over the common folds of the scores of the weighted combination on each fold's block."""
    per = []
    for k in common_folds:
        blocks = {a: next(o for o in OOF[a] if o["k"] == k) for a in ARCHS}
        uk = blocks[ARCHS[0]]["uk"]
        assert all(np.array_equal(blocks[a]["uk"], uk) for a in ARCHS)
        per.append(score(sum(float(wi) * blocks[a]["M"] for wi, a in zip(w, ARCHS)), uk))
    return {k_: float(np.mean([p[k_] for p in per])) for k_ in ("mrp", "top5_concepts", "top5_img")}


# ---------------------------------------------------------------- 2 · architecture weights on the pooled OOF blocks
step = float(CV.get("weight_step", 0.05))
n = int(round(1 / step))
ANCHOR = CV.get("anchor_weights")              # {arch: weight} of an earlier tuned ensemble: only the share of the new architecture(s) is searched
new_a = []
if ANCHOR:
    new_a = [a for a in ARCHS if a not in ANCHOR]
    base = np.array([float(ANCHOR.get(a, 0.0)) for a in ARCHS]); base = base / base.sum()
    newv = np.array([a in new_a for a in ARCHS], float); newv = newv / max(1.0, newv.sum())
    shares = np.round(np.arange(0, float(CV.get("anchor_max_share", 0.5)) + 1e-9, step), 6)
    grid = [(1 - x) * base + x * newv for x in shares]
else:
    grid = [np.array(c, float) / n for c in itertools.product(range(n + 1), repeat=len(ARCHS)) if sum(c) == n]
t0 = time.time()
search = []
for w in grid:
    sc = oof_scores(w)
    search.append({**{f"w_{a}": float(x) for x, a in zip(w, ARCHS)}, **sc, "sel": sel(sc)})
search = pd.DataFrame(search)
search.to_csv(OUT / "oof_weight_search.csv", index=False)
W_EQ = np.full(len(ARCHS), 1.0 / len(ARCHS))
eq = oof_scores(W_EQ)
best = search.iloc[int(search["sel"].idxmax())]
W_BEST = np.array([best[f"w_{a}"] for a in ARCHS])
margin = float(CV.get("min_gain_over_equal", 0.003))
W = W_BEST if best["sel"] > sel(eq) + margin else W_EQ
if ANCHOR:                                       # the anchor weights were tuned already; the new share is chosen on OOF (0 allowed)
    W = W_BEST
if ANCHOR and new_a:
    log("anchored search: " + ", ".join(f"share {r[f'w_{new_a[0]}'] * len(new_a):.2f} → sel {r['sel']:.4f}" for _, r in search.iterrows()))
if CV.get("fixed_weights") is not None:                              # e.g. equal weights, decided beforehand
    W = np.array([float(x) for x in CV["fixed_weights"]], float); W = W / W.sum()
    log("FIXED architecture weights: " + ", ".join(f"{a} {x:.3f}" for x, a in zip(W, ARCHS)))
log(f"weight search on {len(common_folds)} OOF blocks ({len(grid)} combinations, {time.time() - t0:.0f} s): best "
    + ", ".join(f"{a} {x:.2f}" for x, a in zip(W_BEST, ARCHS)) + f" (sel {best['sel']:.4f}) vs equal (sel {sel(eq):.4f}) → "
    + ("TUNED weights used" if W is W_BEST else "EQUAL weights used (the tuned gain is below the margin)"))

# ---------------------------------------------------------------- 3 · local test and NEW, once
TEST_Q, NEW_Q = {}, {}
HAS_TEST = all((Path(f["runs"][a]) / f"pred_test_{json.loads((Path(f['runs'][a]) / 'metrics.json').read_text())['recommended']}.npy").exists()
               for f in FOLDS for a in ARCHS if f["runs"].get(a)) and cfg.get("score_local_test", True)
for a in (ARCHS if HAS_TEST else []):
    for set_name, store in (("TEST", TEST_Q), ("NEW", NEW_Q)):
        Ms, uk0 = [], None
        for f in FOLDS:
            rd = f["runs"].get(a)
            if not rd or not (Path(rd) / "metrics.json").exists():
                continue
            Z, keys, _ = load_set(rd, set_name, f["split_dir"])
            uk, M = to_queries(Z, keys)
            assert uk0 is None or np.array_equal(uk, uk0), f"{set_name} queries differ between folds"
            uk0 = uk; Ms.append(M)
        store[a] = (uk0, Ms)


def combo(store, w, folds_used=None):
    uk = store[ARCHS[0]][0]
    return uk, sum(float(wi) * np.mean(store[a][1] if folds_used is None else [store[a][1][i] for i in folds_used[a]], axis=0)
                   for wi, a in zip(w, ARCHS))


results = []
OOF_ARCH = {a: oof_scores(np.eye(len(ARCHS))[i]) for i, a in enumerate(ARCHS)}
cands = [(f"{a} ({len(FOLDS)}-fold average)" if len(FOLDS) > 1 else f"{a} (single model)", np.eye(len(ARCHS))[i])
         for i, a in enumerate(ARCHS)]
cands += [("ENSEMBLE equal weights", W_EQ), ("ENSEMBLE chosen weights (" + ", ".join(f"{x:.2f}" for x in W) + ")", W)]
seen = set()
for name, w in cands:
    if tuple(np.round(w, 4)) in seen and name.startswith("ENSEMBLE chosen"):
        name += " = equal"
    seen.add(tuple(np.round(w, 4)))
    row = dict(model=name)
    o = OOF_ARCH[ARCHS[int(np.argmax(w))]] if (w == 1).any() else oof_scores(w)
    row.update({"OOF MRP": o["mrp"], "OOF top-5 /150 concepts": o["top5_concepts"], "OOF top-5 /images": o["top5_img"]})
    if HAS_TEST:
        uk_t, M_t = combo(TEST_Q, w)
        uk_n, M_n = combo(NEW_Q, w)
        t, nw = score(M_t, uk_t), score(M_n, uk_n)
        row.update({"TEST MRP": t["mrp"], "TEST top-5 /150 concepts": t["top5_concepts"], "TEST top-5 /1500 img": t["top5_img"],
                    "NEW top-5 /200 official": nw["top5_img"]})
    results.append(row)
    log(f"{name:48s} OOF MRP {row['OOF MRP']:.4f} top-5/c {row['OOF top-5 /150 concepts']:.4f}"
        + (f" | TEST MRP {row['TEST MRP']:.4f} top-5/c {row['TEST top-5 /150 concepts']:.4f} | NEW {row['NEW top-5 /200 official']:.4f}"
           if "TEST MRP" in row else ""))
res_table = pd.DataFrame(results)
res_table.to_csv(OUT / "cv_results.csv", index=False)
for set_name, store in ((("test", TEST_Q), ("new", NEW_Q)) if HAS_TEST else ()):   # the ensemble's predictions, for the team
    uk, M = combo(store, W)
    Mn = M / np.maximum(np.linalg.norm(M, axis=1, keepdims=True), 1e-12)
    np.savez(OUT / f"pred_{set_name}_cv15_queries.npz", subject_id=(uk // 100000).astype(np.int64), image_id=(uk % 100000).astype(np.int64),
             pred=Mn.astype(np.float16))
for a in ARCHS:                                                       # OOF per architecture (all folds), for the team
    uk = np.concatenate([o["uk"] for o in OOF[a]]); M = np.concatenate([o["M"] for o in OOF[a]])
    Mn = M / np.maximum(np.linalg.norm(M, axis=1, keepdims=True), 1e-12)
    np.savez(OUT / f"oof_{a}_queries.npz", subject_id=(uk // 100000).astype(np.int64), image_id=(uk % 100000).astype(np.int64),
             fold=np.concatenate([np.full(len(o["uk"]), o["k"]) for o in OOF[a]]), pred=Mn.astype(np.float16))

# ---------------------------------------------------------------- 4 · the ZIP: every fold model of every architecture
members, names, mweights = [], [], []
keep_reve = CV.get("zip_reve_folds")
for i, a in enumerate(ARCHS):
    rds = [(f["k"], f["runs"][a]) for f in FOLDS if f["runs"].get(a) and (Path(f["runs"][a]) / "metrics.json").exists()]
    if a == "reve" and keep_reve and len(rds) > int(keep_reve):   # optional size cap: the best REVE folds on their own blocks
        by = {r_["fold"]: r_["top5_concepts"] + 0.5 * r_["mrp"] for r_ in rows if r_["arch"] == a}
        rds = sorted(rds, key=lambda x: -by[x[0]])[:int(keep_reve)]
        log(f"ZIP: REVE limited to folds {[k for k, _ in rds]} (zip_reve_folds = {keep_reve})")
    for k, rd in rds:
        m = json.loads((Path(rd) / "metrics.json").read_text())
        mc = json.loads((Path(rd) / "model_cfg.json").read_text())
        st = torch.load(Path(rd) / f"{m['recommended']}.pt", map_location="cpu", weights_only=True)
        dt = getattr(torch, cfg.get("weights_dtype", "float16"))
        st = {kk: (vv.to(dt) if vv.is_floating_point() and vv.numel() >= 4096 else vv) for kk, vv in st.items()}
        members.append((mc, st)); names.append(f"{a}_fold{k}_{m['recommended']}"); mweights.append(float(W[i]) / len(rds))
log(f"ZIP members ({len(members)}): " + ", ".join(f"{nm} ({w_:.3f})" for nm, w_ in zip(names, mweights)))
sub = f"{cfg['zip_prefix']}_{CV.get('zip_tag', 'cv')}{len(members)}_{datetime.datetime.now():%m%d}"
d = WORK / sub
d.mkdir(parents=True)
ens_cfg = dict(members=[mc for mc, _ in members], member_weights=mweights, member_names=names)
state = {f"members.{j}.{kk}": vv for j, (_, st) in enumerate(members) for kk, vv in st.items()}
header = (f'"""Neural Interfaces 2026 · Track 1 (EEG-to-Image) · 5-fold cross-validated ensemble of {len(members)} models '
          f'({", ".join(ARCHS)}; architecture weights {", ".join(f"{x:.2f}" for x in W)}).\n\n'
          "prediction = sum of weight x L2-normalised member prediction. Folds: the team seed-33 manifest's train + validation people\n"
          "and concepts in 5 subject- and concept-disjoint groups (seed 33). The REVE encoder is vendored from braindecode 1.8.1\n"
          "(BSD-3-Clause; REVE: El Ouahidi et al. 2025): neither braindecode nor any download is needed. Trained on Alljoined-1.6M\n"
          f'(NEMAR nm000134). Built {datetime.datetime.now():%Y-%m-%d %H:%M}."""\n')
(d / "submission.py").write_text(header + MODEL_SRC + SOLVER_SRC.replace("__SOLVER_NAME__", f"T1-{sub.replace('_', '-')}"))
torch.save(state, d / "weights.pt")
(d / "config.json").write_text(json.dumps(ens_cfg, indent=1))
UP = Path(cfg["upload_dir"]); UP.mkdir(parents=True, exist_ok=True)
zp = UP / f"{sub}.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_STORED) as z:             # weights do not compress: stored is as small and faster
    for f in ("submission.py", "weights.pt", "config.json"):
        z.write(d / f, arcname=f)
st = dict(zip=str(zp), size_mb=round(zp.stat().st_size / 2**20, 1), members=names, member_weights=mweights,
          arch_weights={a: float(x) for a, x in zip(ARCHS, W)},
          layout=sorted(zipfile.ZipFile(zp).namelist()) == ["config.json", "submission.py", "weights.pt"])
del state, members

# checks run on epochs of fold 0's block (held out for fold 0 only: fine for code checks)
P = TrialPool(CACHE)
trials = load_trials(CACHE)
S0, _, IDS0 = build_split(P, trials, {**cfg, "split_manifest": FOLDS[0]["manifest"]})
g = S0["VAL"][np.random.default_rng(0).permutation(len(S0["VAL"]))[:int(cfg["check_epochs"])]]


def load_dir(dd):
    ns = runpy.run_path(str(dd / "submission.py"))
    mc = json.loads((dd / "config.json").read_text())
    mdl = ns["build_model"](mc)
    s_ = torch.load(dd / "weights.pt", map_location="cpu", weights_only=True)
    mdl.load_state_dict({kk: (vv.float() if vv.is_floating_point() else vv) for kk, vv in s_.items()}, strict=True)
    ok_, n_ = mdl.set_input_channels(list(CH), len(CH))
    return mdl.to(DEVICE).eval(), f"{ok_}/{n_}"


chk = WORK / f"{sub}_unzipped"
chk.mkdir(); zipfile.ZipFile(zp).extractall(chk)
ens, el = load_dir(chk)
st["electrodes"] = el
t0 = time.time()
p_zip = torch.cat([ens.predict(torch.from_numpy(P.get(g[s:s + 64])).to(DEVICE).float()).float().cpu() for s in range(0, len(g), 64)])
if DEVICE.type == "cuda":
    torch.cuda.synchronize()
rate = len(g) / (time.time() - t0)
Xa = torch.from_numpy(P.get(g[:64])).to(DEVICE).float()
a1 = ens.predict(Xa); ens.predict(torch.from_numpy(P.get(g[64:128])).to(DEVICE).float()); a2 = ens.predict(Xa)
del ens
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
# reference: the same ensemble built in this process from the notebook's model code and the same float16 weights
ref_m = build_model(ens_cfg)
s_ = torch.load(d / "weights.pt", map_location="cpu", weights_only=True)
ref_m.load_state_dict({kk: (vv.float() if vv.is_floating_point() else vv) for kk, vv in s_.items()}, strict=True)
del s_
ref_m.set_input_channels(list(CH), len(CH)); ref_m = ref_m.to(DEVICE).eval()
ref = torch.cat([ref_m.predict(torch.from_numpy(P.get(g[s:s + 64])).to(DEVICE).float()).float().cpu() for s in range(0, len(g), 64)])
del ref_m
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
c = F.cosine_similarity(p_zip, ref, dim=-1)
st.update(parity_mean_cos=float(c.mean()), parity_min_cos=float(c.min()), stateless=bool(torch.allclose(a1, a2, atol=1e-5)),
          output_ok=bool(torch.isfinite(p_zip).all() and p_zip.shape[1] == D_OUT))
st["parity"] = st["parity_mean_cos"] > 0.9999 and st["parity_min_cos"] > 0.999 and st["stateless"] and st["output_ok"]
st["epochs_per_s_batch64"] = round(rate)
st["minutes_per_million_epochs_here"] = round(1e6 / max(rate, 1) / 60, 1)
log(f"[{sub}] {st['size_mb']} MB | layout {'ok' if st['layout'] else 'FAIL'} | electrodes {el} | parity {'ok' if st['parity'] else 'FAIL'} "
    f"(cos mean {st['parity_mean_cos']:.6f}, min {st['parity_min_cos']:.6f}, stateless {st['stateless']})")

# offline replay in a clean process
gr = g[:int(cfg["replay_epochs"])]
npz, pred_f = WORK / f"{sub}_replay.npz", WORK / f"{sub}_replay.npy"
EMB = np.load(CACHE / "emb_table.npy").astype(np.float32)
galr = np.unique(P.img[gr])
np.savez(npz, X=P.get(gr), subj=P.subj[gr], tidx=np.searchsorted(galr, P.img[gr]), cand=EMB[galr])
empty = Path(tempfile.mkdtemp(prefix="offline_"))
env = {**os.environ, "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1", "HF_HOME": str(empty), "MNE_DATA": str(empty),
       "REVE_POSITIONS_PATH": str(empty), "PYTHONWARNINGS": "ignore"}
snap = {p.name: (p.stat().st_size, p.stat().st_mtime_ns) for p in chk.iterdir()}
for p in [chk, *chk.iterdir()]:
    p.chmod(0o555 if p.is_dir() else 0o444)
try:
    r = subprocess.run([sys.executable, "-I", "-c", REPLAY, str(TRACK), str(chk), str(npz), DEVICE.type, json.dumps(list(CH)), str(SF),
                        str(pred_f)], cwd=str(empty), capture_output=True, text=True, timeout=5400, env=env)
finally:
    for p in [chk, *chk.iterdir()]:
        p.chmod(0o755 if p.is_dir() else 0o644)
line = [l for l in r.stdout.splitlines() if l.startswith("REPLAY_OK")]
ok = r.returncode == 0 and bool(line) and pred_f.exists()
if ok:
    cr = F.cosine_similarity(torch.from_numpy(np.load(pred_f)).float(), p_zip[:len(gr)], dim=-1)
    t5, _, _, bd = line[-1].split()[1:5]
    wrote = {p.name: (p.stat().st_size, p.stat().st_mtime_ns) for p in chk.iterdir()} != snap
    st.update(replay_top5_official=float(t5), replay_mean_cos=float(cr.mean()), replay_braindecode_loaded=bool(int(bd)),
              replay_wrote_into_folder=wrote)
    ok = float(cr.mean()) > 0.999 and not int(bd) and not wrote
st["real_replay"] = ok
log(f"   offline replay ({DEVICE.type}, {len(gr):,} epochs, official metric): {'ok' if ok else 'FAIL FAILED'}"
    + (f" top-5 {st['replay_top5_official']:.4f} among {len(galr)} | cosine {st['replay_mean_cos']:.5f}" if "replay_mean_cos" in st else ""))
if not ok:
    print((r.stdout + r.stderr)[-3000:])
if cfg["check_harness"]:
    outp = WORK / f"{sub}_harness.parquet"
    dataset = f"Simulated[n_chans=32,n_times={META['n_times']},n_images=8,n_outputs={D_OUT},n_train=12,n_test=8,sfreq={SF:g}]"
    cmd = [sys.executable, "-m", "benchopt", "run", str(TRACK), "-d", dataset, "-o", "Image-decoding[training=False]",
           "-s", str(chk / "submission.py"), "--no-plot", "--no-html", "-r", "1", "--no-cache", "--output", str(outp)]
    try:
        h = subprocess.run(cmd, cwd=cfg["kit"], capture_output=True, text=True, timeout=3600,
                           env={**os.environ, "COMPET_SUBMISSION_DIR": str(chk), "HF_HUB_OFFLINE": "1"})
        txt = h.stdout + h.stderr
        hok = h.returncode == 0 and "Traceback" not in txt
    except Exception as e:
        hok, txt = False, repr(e)
    st["official_harness"] = hok
    log(f"   official benchopt harness: {'ok' if hok else 'FAIL FAILED'}")
    if not hok:
        print(txt[-3000:])
st["timing_ok"] = st["minutes_per_million_epochs_here"] < cfg["eval_limit_min"]
log(f"   timing: {st['epochs_per_s_batch64']:,} epochs/s in batches of 64 → {st['minutes_per_million_epochs_here']} min per million epochs "
    f"here (sealed limit {cfg['eval_limit_min']} min on a faster GPU) {'ok' if st['timing_ok'] else 'WARNING'}")
st["all_checks_passed"] = all(st.get(k_, True) for k_ in ("layout", "parity", "real_replay", "official_harness", "timing_ok"))
if not st["all_checks_passed"]:                                      # (a smoke ZIP is already named DO_NOT_UPLOAD)
    bad = zp.with_name(zp.stem + "__CHECK_FAILED_DO_NOT_UPLOAD.zip")
    zp.rename(bad); st["zip"] = str(bad)
summary = dict(archs=ARCHS, folds=[f["k"] for f in FOLDS], common_folds=common_folds, arch_weights=st["arch_weights"],
               weights_rule=("fixed" if CV.get("fixed_weights") is not None else "tuned" if W is W_BEST else "equal"), tuned_weights={a: float(x) for a, x in zip(ARCHS, W_BEST)},
               oof_equal=eq, oof_tuned={k_: float(best[k_]) for k_ in ("mrp", "top5_concepts", "top5_img")}, zip=st,
               fold_table=fold_table.to_dict("records"), results=res_table.to_dict("records"))
(OUT / "cv_metrics.json").write_text(json.dumps(summary, indent=1, default=str))
npz.unlink(missing_ok=True); pred_f.unlink(missing_ok=True); shutil.rmtree(empty, ignore_errors=True); shutil.rmtree(chk, ignore_errors=True)
log(f"cv ensemble done: {Path(st['zip']).name} {'all checks passed' if st['all_checks_passed'] else 'WARNING: a check failed'}")

## Data
Uses the extracted cache if there is one, otherwise extracts it with the kit's own loader.

In [ ]:
%%writefile {SRC}/t1_extract.py
"""Extraction of Alljoined-1.6M through the competition kit's own loader (the official preprocessing, i.e. what the
sealed test sees), into a float16 cache: X_<split>.npy, img_/subj_/rec_<split>.npy, emb_table.npy (DINOv2-giant
targets), trials_<split>.parquet (event metadata incl. stimulus paths), meta.json.

usage: python t1_extract.py <data_dir> <out_dir> <kit_track_dir>
  data_dir: a writable folder that contains (or links to) Xu2025Alljoined/download/nm000134
"""
import gc
import hashlib
import json
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

DATA_DIR, OUT, TRACK = Path(sys.argv[1]), Path(sys.argv[2]), Path(sys.argv[3])
sys.path.insert(0, str(TRACK))
OUT.mkdir(parents=True, exist_ok=True)
SPLITS, STUDY = ("train", "val", "test"), "xu2025alljoined"
SCALAR = (str, int, float, bool, type(None), np.integer, np.floating)


def log(*a):
    print(time.strftime("%H:%M:%S"), *a, flush=True)


from torch.utils.data import DataLoader, Subset  # noqa: E402
from benchmark_utils.nb_task import load_task  # noqa: E402

t0 = time.time()
log("building the official loaders: preprocessing of the 1,520 recordings + DINOv2-giant targets of the 16,740 images "
    "(the longest part, ~3 h on a T4; cached, so a restarted run resumes)")
ov = {"target.infra.cluster": None, "target.infra.folder": str(DATA_DIR / "cache"), "target.infra.permissions": None}
loaders, meta = load_task("eeg", "image", data_dir=DATA_DIR, dataset=STUDY, device="cpu", batch_size=256, seed=0,
                          num_workers=0, overrides=ov)
log(f"loaders ready in {(time.time() - t0) / 60:.0f} min | n_chans={meta['n_chans']} n_times={meta['n_times']} sfreq={meta['sfreq']}")
keys, rows, counts = {}, [], {}
for split in SPLITS:
    ds = loaders[split].dataset
    idx = np.arange(len(ds))
    N, C, T = len(idx), int(meta["n_chans"]), int(meta["n_times"])
    Xmm = np.lib.format.open_memmap(OUT / f"X_{split}.npy", mode="w+", dtype=np.float16, shape=(N, C, T))
    img = np.empty(N, np.int32); subj = np.empty(N, np.int32); rec = np.empty(N, np.int32)
    pos = 0
    for X, y, info in tqdm(DataLoader(Subset(ds, idx.tolist()), batch_size=256, shuffle=False, num_workers=0),
                           desc=f"extract {split} ({N:,} epochs)", mininterval=30):
        b = len(X)
        Xmm[pos:pos + b] = X.numpy().astype(np.float16)
        yn = np.ascontiguousarray(y.numpy().astype(np.float32))
        for j in range(b):                              # image id = index of its (unique) DINOv2 embedding
            h = hashlib.blake2b(yn[j].tobytes(), digest_size=16).digest()
            k = keys.get(h)
            if k is None:
                k = keys[h] = len(rows)
                rows.append(yn[j].copy())
            img[pos + j] = k
        subj[pos:pos + b] = np.asarray(info["subject_id"]).reshape(-1)
        rec[pos:pos + b] = np.asarray(info["record_id"]).reshape(-1)
        pos += b
    Xmm.flush(); del Xmm
    for name, arr in (("img", img), ("subj", subj), ("rec", rec)):
        np.save(OUT / f"{name}_{split}.npy", arr)
    segs, meta_rows = ds.seg_ds.segments, []
    for i in tqdm(idx, desc=f"metadata {split}", mininterval=30):
        trig = segs[int(ds.index[int(i)])].trigger
        d = trig.to_dict() if trig is not None else {}
        meta_rows.append({k: v for k, v in d.items() if isinstance(v, SCALAR)})
    df = pd.DataFrame(meta_rows).assign(subject_id=subj, img=img)
    for c in df.columns:
        if pd.api.types.is_object_dtype(df[c]):
            df[c] = df[c].astype(str)
    df.to_parquet(OUT / f"trials_{split}.parquet")
    counts[split] = int(N)
    log(f"{split}: {N:,} epochs written")
np.save(OUT / "emb_table.npy", np.stack(rows).astype(np.float32))
info = dict(study=STUDY, query=None, ch_names=[str(c) for c in meta["ch_names"]], sfreq=float(meta["sfreq"]),
            n_times=int(meta["n_times"]), n_chans=int(meta["n_chans"]), d_out=int(len(rows[0])), counts=counts,
            n_images=len(rows), extract_seconds=round(time.time() - t0))
(OUT / "meta.json").write_text(json.dumps(info, indent=1))
(OUT / "DONE").write_text("ok")
log(f"extraction finished in {(time.time() - t0) / 60:.0f} min: {sum(counts.values()):,} epochs, {len(rows):,} images → {OUT}")

In [ ]:
def walk_dirs(root, max_depth):
    """Folders under root up to max_depth, without descending into the raw recordings."""
    root = Path(root)
    if not root.exists():
        return
    for dirpath, dirnames, filenames in os.walk(root, followlinks=True):
        rel = Path(dirpath).relative_to(root).parts
        yield Path(dirpath), set(filenames)
        dirnames[:] = [] if len(rel) >= max_depth else [d for d in dirnames if not d.startswith(("sub-", "stimuli", "."))]


def find_cache():
    cands = ([Path(CACHE_DIR)] if CACHE_DIR else []) + [WORK / "alljoined_cache", SCRATCH / "alljoined_cache"]
    cands += [d for d, f in walk_dirs(INPUT, 5) if {"meta.json", "X_train.npy", "emb_table.npy"} <= f]
    return next((c for c in cands if (c / "meta.json").exists() and (c / "X_train.npy").exists() and (c / "emb_table.npy").exists()), None)


def find_raw():
    if RAW_STUDY:
        return Path(RAW_STUDY)
    return next((d.parent.parent for d, _ in walk_dirs(INPUT, 5) if d.name == "nm000134" and d.parent.name == "download"), None)


CACHE = find_cache()
if CACHE is not None:
    print(f"extracted cache: {CACHE} ok (no extraction needed)")
    if str(CACHE).startswith(str(INPUT)):           # /kaggle/input is network storage: random reads make training ~30x slower
        local = SCRATCH / "alljoined_cache_local"
        if not (local / "COPIED").exists():
            t0 = time.time(); local.mkdir(parents=True, exist_ok=True)
            for f in sorted(CACHE.iterdir()):
                if f.is_file():
                    shutil.copy(f, local / f.name)
            (local / "COPIED").write_text(str(CACHE))
            print(f"copied to local disk {local} in {(time.time() - t0) / 60:.1f} min")
        CACHE = local
else:
    RAW = find_raw()
    if RAW is None:
        raise RuntimeError(f"no data found under {INPUT}: attach the Alljoined dataset (raw …/download/nm000134, or an extracted cache)")
    nm = RAW / "download" / "nm000134"
    n_edf, n_tsv = len(list(nm.glob("sub-*/ses-*/eeg/*_eeg.edf"))), len(list(nm.glob("sub-*/ses-*/eeg/*_events.tsv")))
    n_subj = len(list(nm.glob("sub-*")))
    print(f"raw study: {RAW} | {n_subj} subjects, {n_edf} EDF recordings, {n_tsv} event tables (official count: 20, 1,520) | "
          f"stimuli folder: {'ok' if (nm / 'stimuli').exists() else 'FAIL MISSING'}")
    assert n_edf and n_tsv == n_edf and (nm / "stimuli").exists(), "the raw folder is incomplete (EDF, events.tsv and stimuli are all needed)"
    DATA_DIR = SCRATCH / "neural_compet"
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    link = DATA_DIR / "Xu2025Alljoined"                 # the kit expects <data_dir>/Xu2025Alljoined/download/nm000134
    if not link.exists():
        link.symlink_to(RAW, target_is_directory=True)
    track = KIT / "tracks" / "image_decoding"
    sh([PY, "-c", f"import sys; sys.path.insert(0, {str(track)!r}); from benchmark_utils.nb_task import load_task; import neuralbench; "
                  "print('official loader importable ok')"], env=ENV)
    out = (WORK if KEEP_EXTRACTION else SCRATCH) / "alljoined_cache"
    free = shutil.disk_usage(SCRATCH).free / 2**30
    if free < 30:
        print(f"WARNING: only {free:.0f} GiB free in {SCRATCH}: the extraction caches need ~25 GiB")
    if SMOKE_TEST:
        print("SMOKE TEST: raw data found and the loader imports; the extraction (~4–5 h) runs in the real run. "
              "Training is not smoke-tested without an extracted cache.")
    else:
        sh([PY, SRC / "t1_extract.py", DATA_DIR, out, track], env=ENV, cwd=SCRATCH)
        CACHE = out
CFG = dict(cache=str(CACHE) if CACHE else None, out_dir=str(OUT), scratch=str(SCRATCH), kit=str(KIT), recipes=RECIPES, score_local_test=SCORE_LOCAL_TEST,
           deadline_unix=T_START + KAGGLE_LIMIT_HOURS * 3600 - 15 * 60, finish_minutes=FINISH_MINUTES, gpu_mem_fraction=0.80,
           weights_dtype=WEIGHTS_DTYPE, check_epochs=CHECK_EPOCHS, replay_epochs=REPLAY_EPOCHS, check_harness=CHECK_HARNESS,
           eval_limit_min=EVAL_LIMIT_MIN, precision=PRECISION, zip_prefix="t1" + ("_SMOKE_DO_NOT_UPLOAD" if SMOKE_TEST else ""), gpu_name=GPU_NAME)
CFG_PATH = OUT / "config.json"
CFG_PATH.write_text(json.dumps(CFG, indent=1))
if CACHE:
    meta = json.loads((CACHE / "meta.json").read_text())
    print(f"data: {meta['n_chans']} electrodes × {meta['n_times']} samples")

## Split
Team manifest (seed 33), embedded and checked against its sha256.

In [ ]:
import importlib
import pandas as pd
from IPython.display import display
sys.path.insert(0, str(SRC))
import t1_split as TS
importlib.reload(TS)


def find_input_files(pred, depth=6):
    return sorted(d / f for d, files in walk_dirs(INPUT, depth) for f in files
                  if pred(d / f) and "reve_kaggle" not in str(d))      # never an attached output of this notebook


# 1 · the team manifest: an attached threeway_manifest.json, else the copy embedded in this notebook (verified the same way)
EMBEDDED = json.loads((SRC / "threeway_manifest_seed33.compact.json").read_text())
AUDIT = EMBEDDED.pop("prepared_loader_audit", None)
team = TS.expand_team_manifest(EMBEDDED)
assert TS.team_science_sha(team) == team["science_sha256"] == EXPECTED_SCIENCE_SHA, "the embedded team manifest does not verify"
SOURCE = f"team manifest {team['protocol']} seed {team['seed']} (embedded copy)"
paths = ([Path(TEAM_MANIFEST)] if TEAM_MANIFEST not in (None, "auto") else
         find_input_files(lambda p: p.name.lower().endswith(".json") and "threeway" in p.name.lower() and "manifest" in p.name.lower()
                          and "audit" not in p.name.lower() and "compact" not in p.name.lower()))
if len(paths) > 1:
    raise RuntimeError(f"several 3-way manifests attached, set TEAM_MANIFEST to one of them: {[str(p) for p in paths]}")
if paths:
    attached = json.loads(paths[0].read_text())
    sha = TS.team_science_sha(attached)
    if sha != attached.get("science_sha256"):
        raise RuntimeError(f"{paths[0]}: its science_sha256 does not verify (edited or truncated file)")
    if sha == team["science_sha256"]:
        print(f"attached {paths[0]}: identical to the embedded copy ok (science sha256 {sha[:16]}…)")
    else:
        print(f"WARNING: attached {paths[0]} is a DIFFERENT manifest (science sha256 {sha[:16]}…, expected {EXPECTED_SCIENCE_SHA[:16]}…): it is used")
        AUDIT = None
        for a in find_input_files(lambda p: "audit" in p.name.lower() and p.suffix.lower() == ".json"):
            aj = json.loads(a.read_text())
            if aj.get("science_sha256") == sha:
                AUDIT = aj
    team = attached
    SOURCE = f"team manifest {team['protocol']} seed {team['seed']} ({paths[0].name})"
print(f"split: {SOURCE} | science sha256 {team['science_sha256']}")

# 2 · canonical ids (subject 1-20, image 0-16539, concept = image // 10 + 1), with the team own checks
USED, NAMES, N_CONCEPTS = TS.from_team_manifest(team)
problems = TS.check_manifest(USED, N_CONCEPTS)
if problems:
    raise RuntimeError("the manifest breaks the protocol:\n- " + "\n- ".join(problems))
print(f"team checks ok (science sha256, per-list sha256, disjoint people / concepts / images) | "
      f"THINGS concept names agree with image // 10 + 1 for all {N_CONCEPTS:,} concepts ok")
EXPECTED_EPOCHS = None
if AUDIT:
    c = AUDIT["counts"]
    EXPECTED_EPOCHS = dict(FIT=c["train_total"], VAL=c["validation_total"], official_train=c["original_train"], official_val=c["original_val"])
    print(f"team prepared-loader counts (checked against our cache in training): {EXPECTED_EPOCHS}")

# 3 · the frozen split for training (+ files for the team)
SPLIT_DIR = OUT / "split"
SPLIT_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH = SPLIT_DIR / "split_manifest.json"
mj = TS.manifest_json(USED, dict(protocol=team["protocol"], seed=team["seed"]), SOURCE, None, N_CONCEPTS)
mj.update(team_science_sha256=team["science_sha256"], expected_epochs=EXPECTED_EPOCHS, evaluation_policy=team.get("evaluation_policy"),
          concept_names={str(k): v for k, v in NAMES["concepts"].items()})
MANIFEST_PATH.write_text(json.dumps(mj, indent=1))
tab = TS.manifest_table(USED)
tab["things_name"] = [NAMES["concepts"].get(i, "") if k == "concepts" else "" for k, i in zip(tab["kind"], tab["id"])]
tab.to_csv(SPLIT_DIR / "split_manifest.csv", index=False)
(SPLIT_DIR / "team_threeway_manifest.json").write_text(json.dumps(team, indent=1, ensure_ascii=False))
CFG["split_manifest"] = str(MANIFEST_PATH)
CFG_PATH.write_text(json.dumps(CFG, indent=1))
display(pd.DataFrame({s: {"subjects": ", ".join(map(str, USED[s]["subjects"])), "concepts": len(USED[s]["concepts"]),
                          "images": len(USED[s]["images"]),
                          "first concepts": ", ".join(NAMES["concepts"][c] for c in USED[s]["concepts"][:4]) + ", …"}
                      for s in TS.SPLITS3}).T)

## Folds

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

# the 5 folds: the manifest's train + validation people and concepts, each cut into 5 groups (seed 33); local test untouched
rng = np.random.default_rng(FOLD_SEED)
POOL_S = sorted(set(USED["train"]["subjects"]) | set(USED["val"]["subjects"]))
POOL_C = sorted(set(USED["train"]["concepts"]) | set(USED["val"]["concepts"]))
S_GROUPS = [sorted(int(x) for x in g) for g in np.array_split(rng.permutation(POOL_S), N_FOLDS)]
C_GROUPS = [sorted(int(x) for x in g) for g in np.array_split(rng.permutation(POOL_C), N_FOLDS)]
CV_DIR = OUT / "split" / "cv"
CV_DIR.mkdir(parents=True, exist_ok=True)
FOLD_MANIFESTS, rows = {}, []
for k in range(N_FOLDS):
    m = {"train": dict(subjects=sorted(set(POOL_S) - set(S_GROUPS[k])), concepts=sorted(set(POOL_C) - set(C_GROUPS[k]))),
         "val": dict(subjects=S_GROUPS[k], concepts=C_GROUPS[k]),
         "test": dict(subjects=USED["test"]["subjects"], concepts=USED["test"]["concepts"])}
    for s in m.values():
        s["images"] = TS.images_of(s["concepts"])
    problems = TS.check_manifest(m, N_CONCEPTS)
    assert not problems, f"fold {k}: {problems}"
    mj = TS.manifest_json(m, dict(fold=k, n_folds=N_FOLDS, fold_seed=FOLD_SEED), f"5-fold CV (seed {FOLD_SEED}) of {SOURCE}: fold {k}",
                          None, N_CONCEPTS)
    mj.update(team_science_sha256=team["science_sha256"], concept_names={str(c): v for c, v in NAMES["concepts"].items()},
              expected_epochs=None)
    p = CV_DIR / f"fold{k}_manifest.json"
    p.write_text(json.dumps(mj, indent=1))
    FOLD_MANIFESTS[k] = p
    rows.append(dict(fold=k, val_people=", ".join(map(str, S_GROUPS[k])), val_concepts=len(C_GROUPS[k]),
                     train_people=len(m["train"]["subjects"]), train_concepts=len(m["train"]["concepts"]),
                     manifest_sha256=mj["checksums"]["manifest"][:16]))
assert set().union(*map(set, S_GROUPS)) == set(POOL_S) and set().union(*map(set, C_GROUPS)) == set(POOL_C)
assert not set(POOL_S) & set(USED["test"]["subjects"]) and not set(POOL_C) & set(USED["test"]["concepts"])
(CV_DIR / "cv_folds.json").write_text(json.dumps(dict(seed=FOLD_SEED, n_folds=N_FOLDS, people=S_GROUPS, concepts=C_GROUPS,
                                                      local_test=USED["test"]), indent=1))
FOLD_TABLE = pd.DataFrame(rows)
display(FOLD_TABLE)
print(f"every fold: trains on the other people x the other concepts, validated on its own people x its own concepts; "
      f"local test (people {USED['test']['subjects']}, {len(USED['test']['concepts'])} concepts) never trained on")

In [ ]:
# runs finished in an earlier version of this notebook (its output attached as input) are reused
RUNS_DIR = WORK / "cv_runs"
RUNS_DIR.mkdir(parents=True, exist_ok=True)
reused = 0
for prev in [d for d, _ in walk_dirs(INPUT, 7) if d.name == "cv_runs" and d != RUNS_DIR]:
    for m in prev.glob("fold*/*/metrics.json"):
        fold_dir, run = m.parent.parent, m.parent
        dst = RUNS_DIR / fold_dir.name
        if (dst / run.name / "metrics.json").exists():
            continue
        if (fold_dir / "split").exists() and not (dst / "split").exists():
            shutil.copytree(fold_dir / "split", dst / "split")
        shutil.copytree(run, dst / run.name, dirs_exist_ok=True)
        reused += 1
    old = prev.parent / "reve_kaggle" / "split" / "cv" / "cv_folds.json"
    if old.exists():
        o = json.loads(old.read_text())
        assert o["people"] == S_GROUPS and o["concepts"] == C_GROUPS, f"{old}: different folds, cannot reuse these runs"
print(f"reused runs: {reused}")

## Training
5 architectures × 5 folds, fold by fold. A run starts only if it fits in the time left.

In [ ]:
DEADLINE = T_START + TIME_BUDGET_HOURS * 3600
MEASURED = dict(EST_MINUTES)
DONE = {k: {} for k in range(N_FOLDS)}
LOG = []


def run_fold(arch, k):
    out = RUNS_DIR / f"fold{k}"
    R = {**RECIPES[arch], "tag": f"{RECIPES[arch]['tag']}_f{k}"}
    seed = 1000 + k
    run_dir = out / f"{arch}_{R['tag']}_s{seed}"
    if (run_dir / "metrics.json").exists():
        return run_dir
    c = {**CFG, "out_dir": str(out), "split_manifest": str(FOLD_MANIFESTS[k]), "recipes": {arch: R}, "eval_sets": list(globals().get("EVAL_SETS", ["VAL", "TEST", "NEW"])),
         "save_weights": True, "save_preds": True, "score_local_test": False, "monitor_max_epochs": MONITOR_MAX_EPOCHS,
         "deadline_unix": DEADLINE - RESERVE_MIN * 60}
    (RUNS_DIR / "cfgs").mkdir(parents=True, exist_ok=True)
    cp = RUNS_DIR / "cfgs" / f"{arch}_f{k}.json"
    cp.write_text(json.dumps(c, indent=1))
    script = [SRC / "t1_train.py", cp, seed, arch]
    cmd = [PY, *script] if NPROC == 1 else [PY, "-m", "torch.distributed.run", "--standalone", f"--nproc_per_node={NPROC}", *script]
    try:
        sh(cmd, env=ENV, cwd=SRC)
    except RuntimeError as e:
        gpu_problem = any(x in str(e) for x in ("NCCL error", "DistBackendError", "Watchdog caught", "CUDA error", "OutOfMemoryError"))
        if NPROC == 1 or not gpu_problem:
            print(f"WARNING: {arch} fold {k} failed: {str(e).splitlines()[0][:200]}")
            return None
        print("WARNING: multi-GPU run failed: retrying on one GPU")
        sh([PY, *script], env={**ENV, "CUDA_VISIBLE_DEVICES": "0"}, cwd=SRC, check=False)
    return run_dir if (run_dir / "metrics.json").exists() else None


for k in range(N_FOLDS):                                 # fold by fold: if time runs short, the last folds are the ones missing
    for arch in ARCHS:
        left = (DEADLINE - time.time()) / 60
        if left - MEASURED[arch] * 1.1 < RESERVE_MIN:
            print(f"{arch} fold {k}: skipped (time: {left:.0f} min left, ~{MEASURED[arch]:.0f} needed + {RESERVE_MIN} for the ensemble)")
            LOG.append(dict(fold=k, arch=arch, status="skipped (time)"))
            continue
        t0 = time.time()
        rd = run_fold(arch, k)
        mins = (time.time() - t0) / 60
        if rd is not None:
            DONE[k][arch] = rd
            MEASURED[arch] = mins if mins > 1 else MEASURED[arch]        # measured time of this architecture's last run
            m = json.loads((rd / "metrics.json").read_text())
            v = m["recommended"]
            LOG.append(dict(fold=k, arch=arch, status="done", minutes=round(mins, 1), variant=v,
                            val_mrp=m["variants"][v]["VAL"]["mrp@all"]))
        else:
            LOG.append(dict(fold=k, arch=arch, status="failed", minutes=round(mins, 1)))
        print(f"→ fold {k} {arch}: {LOG[-1]['status']} in {mins:.0f} min | {(DEADLINE - time.time()) / 3600:.1f} h left")
pd.DataFrame(LOG).to_csv(OUT / "cv_training_log.csv", index=False)
display(pd.DataFrame(LOG))

## Ensemble and submission ZIP

In [ ]:
CFG_CV = {**CFG, "cv": dict(archs=ARCHS, n_way=150, draws=20, weight_step=WEIGHT_STEP, min_gain_over_equal=MIN_GAIN_OVER_EQUAL,
                            zip_reve_folds=ZIP_REVE_FOLDS,
                            folds=[dict(k=k, manifest=str(FOLD_MANIFESTS[k]), split_dir=str(RUNS_DIR / f"fold{k}" / "split"),
                                        runs={a: str(rd) for a, rd in DONE[k].items()}) for k in range(N_FOLDS) if DONE[k]]),
          "upload_dir": str(WORK / "UPLOAD_TO_CODABENCH"), "score_local_test": SCORE_LOCAL_TEST, "smoke": SMOKE_TEST}
CFG_CV_PATH = OUT / "config_cv.json"
CFG_CV_PATH.write_text(json.dumps(CFG_CV, indent=1))
sh([PY, SRC / "t1_cv_ensemble.py", CFG_CV_PATH], env={**ENV, "CUDA_VISIBLE_DEVICES": "0"}, cwd=SCRATCH)

## Results

In [ ]:
import zipfile
from IPython.display import Markdown

cvm = json.loads((OUT / "cv_ensemble" / "cv_metrics.json").read_text())
ft = pd.DataFrame(cvm["fold_table"])
print("Out-of-fold scores of every model on its own block (people_k x concepts_k; top-5 among 150 concepts, one image each):")
display(ft.pivot_table(index="arch", columns="fold", values=["mrp", "top5_concepts"]).style.format("{:.4f}"))
res = pd.DataFrame(cvm["results"])
REF = pd.DataFrame([
    {"model": "chance", "OOF MRP": 0.5, "OOF top-5 /150 concepts": 5 / 150, "TEST MRP": 0.5, "TEST top-5 /150 concepts": 5 / 150,
     "TEST top-5 /1500 img": 5 / 1500, "NEW top-5 /200 official": 5 / 200}])
table = pd.concat([res, REF], ignore_index=True)
display(table.style.format({c: "{:.4f}" for c in table.columns if c != "model"}, na_rep="—"))
st = cvm["zip"]
team_dir = WORK / "FOR_TEAM"; team_dir.mkdir(exist_ok=True)
for f in (OUT / "cv_ensemble").glob("*.npz"):
    shutil.copy(f, team_dir / f.name)
for f in ("oof_by_fold.csv", "cv_results.csv", "oof_weight_search.csv"):
    shutil.copy(OUT / "cv_ensemble" / f, team_dir / f)
shutil.copy(OUT / "split" / "cv" / "cv_folds.json", team_dir / "cv_folds.json")
with zipfile.ZipFile(WORK / "FOR_TEAM.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(team_dir.iterdir()):
        zf.write(f, arcname=f"FOR_TEAM/{f.name}")
missing = [f"{r['arch']} fold {r['fold']}" for r in LOG if r["status"] != "done"]
recap = f"""**{'SMOKE TEST — do not upload' if SMOKE_TEST else '5-fold run'}** finished in {(time.time() - T_START) / 3600:.1f} h · {N_GPU} × {GPU_NAME}

**Models**: {sum(len(v) for v in DONE.values())} of {len(ARCHS) * N_FOLDS} trained{(' (missing: ' + ', '.join(missing) + ')') if missing else ''}

**Architecture weights** ({cvm['weights_rule']}, chosen on the out-of-fold blocks): {json.dumps({a: round(w, 3) for a, w in cvm['arch_weights'].items()})}
· out-of-fold selection score: equal {cvm['oof_equal']['top5_concepts'] + 0.5 * cvm['oof_equal']['mrp']:.4f}, tuned {cvm['oof_tuned']['top5_concepts'] + 0.5 * cvm['oof_tuned']['mrp']:.4f}

**ZIP**: `{Path(st['zip']).relative_to(WORK) if str(st['zip']).startswith(str(WORK)) else st['zip']}` · {st['size_mb']} MB · {len(st['members'])} models
· layout {'ok' if st['layout'] else 'FAIL'} · parity {'ok' if st['parity'] else 'FAIL'} · offline replay {'ok' if st.get('real_replay') else 'FAIL'}
· harness {'ok' if st.get('official_harness', True) else 'FAIL'} · {st['minutes_per_million_epochs_here']} min per million epochs on this T4 {'ok' if st['timing_ok'] else 'WARNING'}
· {'upload it' if st['all_checks_passed'] and not SMOKE_TEST else ('smoke test: not for upload' if SMOKE_TEST else 'WARNING: a check failed: do not upload')}

**For the team**: `FOR_TEAM.zip` (out-of-fold predictions per architecture, the ensemble's local-test and NEW predictions, the tables, the folds)."""
(OUT / "cv_recap.md").write_text(recap + "\n\n" + table.to_string(index=False))
display(Markdown(recap))